# validation — 正しさの検証（計算資源の比較とは独立）

`D4_noisy_simulation.ipynb`・`noisy_cost_comparison.ipynb` が使う構成要素が正しいことを、比較の計算とは**独立に**確認する。
各検査は `record(名前, 合否, 詳細)` で記録し（数値を記録するだけで合否の無い項目は `INFO`）、最後に `results/validation.json` 1 個にまとめて保存する。
第 1 部・第 2 部は v4 から引き継いだ雑音なしの検証、第 3 部は v5 で追加した雑音付きシミュレーションの検証。所要時間は約 2 時間（`QUICK = False`；4 コアの計算機で他の計算と並行して実測。最も長いのは N1 の $L=9$ と N5 の窓の全列挙）。`QUICK = True` で大幅に短縮する。

| ID | 内容 | 方法 |
|---|---|---|
| **第 1 部** | **D4 符号・code switching（[KZH]）** | |
| V1 | 安定化子の定義が [KZH] と一致 | Fig. 4 から読み取った特徴との照合 |
| V2 | 符号空間の整合性 | $C_Z$ 保存・可換性・$S^2=1$・$k=1$（$\mathbb F_2$ 厳密）＋負の対照 |
| V3 | 独立実装による照合 | $L=1$ を密ベクトルで独立に実装 |
| V4 | 論理演算子と距離 | MILP の距離を全探索で照合 |
| V5, V5b | Knill–Laflamme 条件 | $L=2$ の全 weight≤2 Pauli；D4 相の距離 $=L$（$L\le5$） |
| V6 | transversal $U=WV$ | 局所厳密列挙、論理位相、負の対照 |
| V7, V7b, V8 | プロトコル全体 | $L=1,2$ 厳密シミュレーション；$L=3,4$ 回路と位相公式の照合；$L\le8$ 局所厳密証明 |
| V9, V10, V11 | 測定ガジェット・スケジュール・計数 | 密行列、可換性、幾何からの直接公式 |
| **第 2 部** | **15-to-1・GoSC の配置（[Lit]）** | |
| M1–M5 | 15-to-1 の行列・回路・誤り検出、Fig. 17a/b の回転 | 状態ベクトル |
| M6–M9 | tile 数・時間、物理レベルの格子手術（Fig. 17c の 11 個の多パッチ測定） | $\mathbb F_2$、MILP、stim |
| **第 3 部** | **雑音付きシミュレーション（v5）** | |
| N1 | Takada らの方法の十分条件（Definition 2, Sec. VIII）の、D4 code switching の生成元・論理演算子についての代数的な確認 | $\mathbb F_2$ の厳密判定（$L=2..7$ と 9。有限サイズの確認であり一般の $L$ の証明ではない）、演算子の比較は大域位相だけを許す厳密比較＋負の対照 |
| N2 | Pauli proxy の「ドレッシング」$C_e=D^\dagger X^eDX^e$ が Clifford で、実装したゲート列と一致 | 全ての非 Clifford ゲート・全パターンの行列 |
| N3 | **Pauli proxy 回路（Clifford）と真の非 Clifford 回路の一致** | 同じ誤り・同じ補正で並走させ、全測定の確率と最終 $\langle\bar H\rangle$ を比較：(a) $L=1,2$、ランダムな $X$ 補正；(b) $L=2$、**本番と同じ適応処理**（flux 復号の補正、switch-out の復号済み読み出しと物理 CZ 層とその雑音、Pauli frame） |
| N4 | 雑音なしで出力誤り 0（復号器込み） | $d=3,5,7$ |
| N5 | 1 個の故障に対する結果（INFO：故障の網羅範囲と失敗の記録） | switch-out の動的な CZ・アイドルを含む全雑音位置×全 Pauli、各 2 つの測定乱数の種；$d=3$ 全列挙（A, B）、$d=5$ 抽出。確率で重み付けした一次の係数 $c_1$（$\varepsilon\approx c_1p$）を窓の内外に分けて記録 |
| N10 | FSC の並列スケジュール（感度用） | 全検査を同時に測る 6 CNOT 層の順序が、stim のメモリ実験で雑音なしに決定的、回路レベルの距離 $\ge L$（X, Z 両基底） |
| N9 | switch-out の物理スケジュール | 補正 CZ の層が正しい（同じ量子ビットを同時に使わない）、適用される CZ が $\prod_{v\in\kappa}S^g_v$ の CZ 部分（偶数回は相殺）と一致、時間の計数 |
| N6 | Litinski（2019）の誤り予算の再実装が公開値を再現 | 公開ノートブックに保存された出力 8 件 |
| N7 | 較正用の表面符号回路 | 雑音なしで検出器 0、回路レベルの距離 $=d$、1 ラウンド 6 時間ステップ |
| N8 | GoSC 15-to-1 の誤り予算モデル | 雑音なしで誤り 0、入力 $Z$ 誤りだけなら $35p^3$ |

In [ ]:
# ==== 環境設定（Google Colab / ローカル共通）====
# このノートブックは単独で実行できる（共通コードは下に埋め込まれており、他のファイルを読み込まない）。
# 出力（results/, figures/）はこのランタイムの作業ディレクトリに保存される。Colab ではランタイム終了で
# 消えるので、残したい場合は Google Drive を使う：
#   from google.colab import drive; drive.mount('/content/drive')
#   os.makedirs('/content/drive/MyDrive/D4_vs_MSD', exist_ok=True); os.chdir('/content/drive/MyDrive/D4_vs_MSD')
import os, sys, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
RESULTS, FIGURES = 'results', 'figures'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
SEED = 20261024
print('cwd =', os.getcwd(), '| Colab =', IN_COLAB)

In [ ]:
# ==== 外部ライブラリ（Colab に無いものだけをインストール）====
import importlib.util
for _mod, _pkg in [('stim', 'stim'), ('pymatching', 'pymatching'), ('mpmath', 'mpmath'), ('scipy', 'scipy'),
                   ('sympy', 'sympy'), ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
import numpy as np, pandas as pd, scipy, stim, pymatching, mpmath
print('numpy', np.__version__, '| scipy', scipy.__version__, '| stim', stim.__version__,
      '| pymatching', pymatching.__version__, '| mpmath', mpmath.__version__)

## 共通ライブラリ（埋め込み・関数定義のみ）
Google Colab ではノートブックごとに別のランタイムで実行されるため、他のファイルを読み込まずに済むよう、共通コードをこのノートブックに埋め込んでいる（各ノートの共通コードは同一のソースから自動生成されており、内容は一致する）。
**このセクションは上から順に実行するだけでよく、編集する必要はない**（Colab では見出しの左の三角で折りたためる）。

### 共通ライブラリ（F2 線形代数・ゲート／ガジェット・スケジューラ・Clifford+T 分解・計数ルール R1–R3）

In [ ]:
"""common_lib: F2 linear algebra, gate/gadget model, scheduler, decompositions, cost model rules."""
import itertools, math
import numpy as np
from collections import Counter, defaultdict

In [ ]:
# ---------------------------------------------------------------- F2 algebra
def f2_rref(M):
    """Row-reduce a uint8 matrix over F2. Returns (R, pivot_cols)."""
    M = (np.array(M, dtype=np.uint8) & 1).copy()
    rows, cols = M.shape
    piv = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        nz = np.nonzero(M[r:, c])[0]
        if len(nz) == 0:
            continue
        p = r + nz[0]
        if p != r:
            M[[r, p]] = M[[p, r]]
        others = np.nonzero(M[:, c])[0]
        others = others[others != r]
        if len(others):
            M[others] ^= M[r]
        piv.append(c)
        r += 1
    return M[:r], piv


def f2_rank(M):
    if len(M) == 0:
        return 0
    return len(f2_rref(M)[1])


def f2_nullspace(M, n=None):
    """Basis (rows) of {x : M x = 0} over F2."""
    M = np.array(M, dtype=np.uint8)
    if M.size == 0:
        return np.eye(n, dtype=np.uint8)
    n = M.shape[1]
    Rm, piv = f2_rref(M)
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        v = np.zeros(n, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if Rm[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis, dtype=np.uint8).reshape(len(basis), n)


def f2_in_rowspace(Rref, piv, v):
    """Is v in the row space of the RREF matrix Rref (pivots piv)?"""
    v = np.array(v, dtype=np.uint8) & 1
    w = v.copy()
    for i, p in enumerate(piv):
        if w[p]:
            w ^= Rref[i]
    return not w.any()


def f2_solve(A, b):
    """Solve x A = b (x row vector, A rows = generators). Returns x or None."""
    A = np.array(A, dtype=np.uint8)
    m, n = A.shape
    aug = np.concatenate([A, np.eye(m, dtype=np.uint8)], axis=1)
    Rm, piv = f2_rref(aug)
    w = np.concatenate([np.array(b, dtype=np.uint8) & 1, np.zeros(m, dtype=np.uint8)])
    for i, p in enumerate(piv):
        if p >= n:
            break
        if w[p]:
            w ^= Rm[i]
    if w[:n].any():
        return None
    return w[n:]




import itertools
from collections import Counter, defaultdict

ARITY = {'INIT_X': 1, 'INIT_Z': 1, 'MEAS_X': 1, 'MEAS_Z': 1, 'H': 1, 'X': 1, 'Z': 1, 'S': 1, 'SDG': 1,
         'T': 1, 'TDG': 1, 'CNOT': 2, 'CZ': 2, 'CS': 2, 'CSDG': 2, 'CCZ': 3}
NONCLIFFORD = {'T', 'TDG', 'CS', 'CSDG', 'CCZ'}

In [ ]:
# ------------------------------------------------------------------ minimum-weight logical (MILP)
def min_weight_odd(Hc, lvec, cols):
    """min |x| over x in F2^cols with Hc x = 0 and lvec . x = 1 (exact MILP, HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    H = np.array(Hc, dtype=float)[:, cols]
    H = H[np.any(H, axis=1)]
    l = np.array(lvec, dtype=float)[cols]
    m, n = H.shape
    # variables: x (n, binary), y (m, integer), z (1, integer)
    nv = n + m + 1
    c = np.concatenate([np.ones(n), np.zeros(m + 1)])
    A1 = np.concatenate([H, -2 * np.eye(m), np.zeros((m, 1))], axis=1)
    A2 = np.concatenate([l[None, :], np.zeros((1, m)), -2 * np.ones((1, 1))], axis=1)
    cons = [LinearConstraint(A1, 0, 0), LinearConstraint(A2, 1, 1)]
    integrality = np.ones(nv)
    ub = np.concatenate([np.ones(n), np.full(m + 1, np.inf)])
    res = milp(c, constraints=cons, integrality=integrality, bounds=Bounds(np.zeros(nv), ub))
    assert res.status == 0, res.message
    x = np.round(res.x[:n]).astype(np.uint8)
    out = np.zeros(len(lvec), dtype=np.uint8)
    out[cols] = x
    return int(x.sum()), out

In [ ]:
# ------------------------------------------------------------------ gadgets
def pauli_check_gadget(anc, xq=(), zq=()):
    """Measure X(xq) Z(zq) with ancilla anc (standard CNOT circuit)."""
    g = [('INIT_X', anc)]
    g += [('CNOT', anc, q) for q in sorted(xq)]           # ancilla controls -> X on data
    g += [('CZ', anc, q) for q in sorted(zq)]             # controlled Z on data
    g += [('MEAS_X', anc)]
    return g


def z_check_gadget(anc, zq):
    g = [('INIT_Z', anc)] + [('CNOT', q, anc) for q in sorted(zq)] + [('MEAS_Z', anc)]
    return g


def clifford_x_gadget(anc, parts):
    """Measure a product of X-type Clifford operators S = S_1 S_2 ... with
    S_k = X(s_k) prod CZ(pairs_k): ancilla |+>, controlled-S (= CNOTs from anc and
    CCZ(anc,i,j) for each CZ pair; rightmost factor first), measure X.
    parts: list of (xqubits, czpairs) in operator order S_1 S_2 ..."""
    g = [('INIT_X', anc)]
    for xq, pairs in reversed(parts):
        g += [('CCZ', anc, i, j) for (i, j) in pairs]      # CZ part acts first within S_k
        g += [('CNOT', anc, q) for q in sorted(xq)]
    g += [('MEAS_X', anc)]
    return g

In [ ]:
# ------------------------------------------------------------------ scheduling
def asap_layers(gates):
    """Greedy ASAP layering of a gate list respecting qubit order (no reordering of gates that
    share a qubit).  Returns list of layers (lists of gates)."""
    ready = defaultdict(int)
    layers = []
    for gt in gates:
        qs = gt[1:]
        t = max(ready[q] for q in qs)
        while len(layers) <= t:
            layers.append([])
        layers[t].append(gt)
        for q in qs:
            ready[q] = t + 1
    return layers


def gadget_support(gad):
    s = set()
    for gt in gad:
        s |= set(gt[1:])
    return s


def disjoint_groups(gadgets):
    """Greedy colouring: partition gadgets so that gadgets in a group touch disjoint qubits.
    Measurements within a group are then exactly simultaneous and the sequential-group
    schedule is valid for commuting (on the relevant subspace) generators."""
    groups = []          # list of (used_qubits, [gadgets])
    order = sorted(range(len(gadgets)), key=lambda i: -len(gadget_support(gadgets[i])))
    for i in order:
        sup = gadget_support(gadgets[i])
        for used, members in groups:
            if not (used & sup):
                used |= sup
                members.append(gadgets[i])
                break
        else:
            groups.append((set(sup), [gadgets[i]]))
    return [m for _, m in groups]


def layer_duration(layer, dur):
    return max(dur[g[0]] for g in layer)


def circuit_time(layers, dur):
    return sum(layer_duration(l, dur) for l in layers)


def round_schedule(gadgets, dur):
    """Upper-bound schedule of one round: disjoint groups executed sequentially, each group
    ASAP-layered.  Returns dict with depth (layers), time (weighted by dur), groups."""
    groups = disjoint_groups(gadgets)
    depth = 0
    time = 0
    for grp in groups:
        gl = asap_layers([gt for gad in grp for gt in gad])
        depth += len(gl)
        time += circuit_time(gl, dur)
    return {'groups': len(groups), 'depth_ub': depth, 'time_ub': time}


def round_lower_bound(gadgets, dur):
    """Lower bound on the duration of one round: every qubit executes its gates sequentially,
    so time >= max_q sum of durations of gates acting on q."""
    busy = defaultdict(float)
    cnt = defaultdict(int)
    for gad in gadgets:
        for gt in gad:
            for q in gt[1:]:
                busy[q] += dur[gt[0]]
                cnt[q] += 1
    return {'depth_lb': max(cnt.values()), 'time_lb': max(busy.values())}


def gate_counts(gates):
    c = Counter(g[0] for g in gates)
    return dict(c)

In [ ]:
# ------------------------------------------------------------------ decompositions
# Explicit Clifford+T decompositions (verified numerically in the validation notebook).
def ccz_to_clifford_t(a, b, c):
    """CCZ with 7 T/T^dag and 6 CNOT (standard Toffoli decomposition, Nielsen & Chuang Fig. 4.9,
    with the target Hadamards removed)."""
    return [('CNOT', b, c), ('TDG', c), ('CNOT', a, c), ('T', c), ('CNOT', b, c), ('TDG', c),
            ('CNOT', a, c), ('T', b), ('T', c), ('CNOT', a, b), ('T', a), ('TDG', b), ('CNOT', a, b)]


def cs_to_clifford_t(a, b):
    """CS = diag(1,1,1,i) = T_a T_b CNOT(a,b) T^dag_b CNOT(a,b)."""
    return [('T', a), ('T', b), ('CNOT', a, b), ('TDG', b), ('CNOT', a, b)]


def csdg_to_clifford_t(a, b):
    return [('TDG', a), ('TDG', b), ('CNOT', a, b), ('T', b), ('CNOT', a, b)]


def decompose(gates):
    out = []
    for g in gates:
        if g[0] == 'CCZ':
            out += ccz_to_clifford_t(*g[1:])
        elif g[0] == 'CS':
            out += cs_to_clifford_t(*g[1:])
        elif g[0] == 'CSDG':
            out += csdg_to_clifford_t(*g[1:])
        else:
            out.append(g)
    return out


DUR_UNIT = {k: 1 for k in ARITY}            # every primitive = 1 time unit (ASSUMPTION, parametrised)

In [ ]:
# ------------------------------------------------- commuting-class scheduler
DIAG_GATES = {'CZ', 'CCZ', 'CS', 'CSDG', 'T', 'TDG', 'S', 'SDG', 'Z'}


def gate_roles(gt):
    """Per-qubit role of an entangling/1q unitary: 'Z' (diagonal in Z basis), 'X' (X-type)."""
    name = gt[0]
    if name in DIAG_GATES:
        return {q: 'Z' for q in gt[1:]}
    if name == 'CNOT':
        return {gt[1]: 'Z', gt[2]: 'X'}
    if name == 'X':
        return {gt[1]: 'X'}
    return {q: '?' for q in gt[1:]}      # H, INIT, MEAS: never shared between gadgets


def gadget_roles(gad):
    roles = {}
    for gt in gad:
        for q, r in gate_roles(gt).items():
            if roles.get(q, r) != r:
                roles[q] = '?'
            else:
                roles[q] = r
    return roles


def commuting_classes(gadgets, n_orders=6, seed=0):
    """Best (fewest classes) of several greedy colourings (different gadget orders)."""
    import random
    best = None
    orders = [list(range(len(gadgets)))]
    orders.append(sorted(orders[0], key=lambda i: tuple(sorted(set(gadget_roles(gadgets[i]).values())))))
    rnd = random.Random(seed)
    for _ in range(n_orders):
        o = orders[0][:]
        rnd.shuffle(o)
        orders.append(o)
    for o in orders:
        cl = _commuting_classes_greedy([gadgets[i] for i in o])
        if best is None or len(cl) < len(best):
            best = cl
    return best


def _commuting_classes_greedy(gadgets):
    """Greedy colouring of the gadget conflict graph.  Two gadgets are compatible if on every
    shared qubit both act with the same role ('Z' or 'X'); then all their gates commute and the
    joint circuit is equivalent to measuring them one after another (sufficient condition)."""
    roles = [gadget_roles(g) for g in gadgets]
    classes = []     # list of (role_map, [idx])
    for i, rm in enumerate(roles):
        placed = False
        for cmap, members in classes:
            ok = all(cmap.get(q, r) == r and r != '?' for q, r in rm.items() if q in cmap)
            if ok:
                for q, r in rm.items():
                    cmap[q] = r
                members.append(i)
                placed = True
                break
        if not placed:
            classes.append((dict(rm), [i]))
    return [[gadgets[i] for i in m] for _, m in classes]


def class_layers(gadgets):
    """Layering of one commuting class: init layer, greedy layering of the (mutually commuting)
    entangling gates, measurement layer."""
    inits = [g for gad in gadgets for g in gad if g[0].startswith('INIT')]
    meas = [g for gad in gadgets for g in gad if g[0].startswith('MEAS')]
    body = [g for gad in gadgets for g in gad if not (g[0].startswith('INIT') or g[0].startswith('MEAS'))]
    # greedy: most-constrained gates first
    body = sorted(body, key=lambda g: -len(g))
    layers = []
    busy = []
    for g in body:
        qs = set(g[1:])
        for li, b in enumerate(busy):
            if not (b & qs) and layers[li][0][0] == g[0]:     # keep one gate type per layer
                b |= qs
                layers[li].append(g)
                break
        else:
            busy.append(set(qs))
            layers.append([g])
    return ([inits] if inits else []) + layers + ([meas] if meas else [])


def round_schedule_classes(gadgets, dur):
    classes = commuting_classes(gadgets)
    depth = 0
    time = 0
    for c in classes:
        ls = class_layers(c)
        depth += len(ls)
        time += circuit_time(ls, dur)
    return {'classes': len(classes), 'depth_ub': depth, 'time_ub': time}


import math
from collections import Counter

SOURCE_LIT = 'literature'
SOURCE_COUNT = 'counted (explicit circuit)'
SOURCE_ASSUME = 'assumption (parameter)'
SOURCE_UNRES = 'unresolved (excluded / illustrative)'

DEFAULT_PARAMS = dict(
    kappa0=1.0,      # FSC |+_L> preparation rounds = ceil(kappa0*d)   [ASSUMPTION; Kobayashi: not specified]
    kappa_in=1.0,    # switch-in rounds            = ceil(kappa_in*d)  [ASSUMPTION; Kobayashi: O(d)]
    kappa_out=1.0,   # FSC EC rounds after return  = ceil(kappa_out*d) [ASSUMPTION; Kobayashi: O(d)]
    tau_rounds=1.0,  # Litinski: 1 time step = tau_rounds*d code cycles [LITERATURE: d cycles]
    t_inj_FT=6,      # Litinski App. A (Li's injection): 3 + 3 code cycles, d-independent [LITERATURE]
    t_inj_NL=1,      # noiseless-minimal injection: 1 round [ASSUMPTION]
)


def rounds_for(model, kappa, d):
    if model == 'NL_min':
        return 1
    return int(math.ceil(kappa * d))

In [ ]:
# ----------------------------------------------------------------- summaries
def summarize(rows):
    tot = Counter()
    for r in rows:
        for k, v in r['gates'].items():
            tot[k] += v
    peak = max(r['qubits'] for r in rows)
    rounds = sum(r['rounds'] for r in rows)
    tl = sum(r['time_lb'] for r in rows if r['time_lb'] is not None)
    tu = sum(r['time_ub'] for r in rows if r['time_ub'] is not None)
    v_round = sum(r['qubits'] * r['rounds'] for r in rows)
    v_tl = sum(r['qubits'] * r['time_lb'] for r in rows if r['time_lb'] is not None)
    v_tu = sum(r['qubits'] * r['time_ub'] for r in rows if r['time_ub'] is not None)
    nonclif = {k: v for k, v in tot.items() if k in NONCLIFFORD}
    tcount = 7 * tot.get('CCZ', 0) + 3 * (tot.get('CS', 0) + tot.get('CSDG', 0)) + tot.get('T', 0) + tot.get('TDG', 0)
    return dict(peak_qubits=peak, rounds=rounds, time_lb=tl, time_ub=tu, V_qubit_round=v_round,
                V_qubit_time_lb=v_tl, V_qubit_time_ub=v_tu, V_peak_x_rounds_UB=peak * rounds,
                gates=dict(tot), nonclifford=nonclif, T_count_if_decomposed=tcount)


def uncounted_costs(protocol, comparison):
    """Costs that are known to exist but are not included in the totals (listed, never summed as 0)."""
    out = [('classical decoding / control latency', 'both comparisons, both protocols')]
    if protocol == 'Kobayashi_D4':
        out.append(('JIT-decoder classical processing', 'not modelled'))
        if comparison == 'B':
            out.append(('FSC -> standard rotated surface-code patch conversion', 'UNRESOLVED: no protocol constructed'))
    else:
        out.append(('twist-based Z_m (x) Y_0 seam checks between |m> and |0>', 'O(d) measurement qubits, not constructed'))
    return out

In [ ]:
# ------------------------------------------------------------------ table helpers
def jsonify_columns(df, cols=('gates', 'nonclifford')):
    """Store dict-valued columns as JSON strings (same format as the saved CSV files)."""
    import json
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda g: g if isinstance(g, str) else json.dumps({k: float(v) for k, v in g.items()}))
    return df


def identify_polynomial(values, npts=4):
    """Interpolate a polynomial in d through the first npts points (degree <= npts-1) and report
    whether it reproduces ALL given points exactly.  values: dict d -> integer.  Returns (expr, ok)."""
    import sympy
    x = sympy.Symbol('d')
    ds = sorted(values)
    if len(ds) <= npts:
        return None, False
    p = sympy.interpolate([(dd, int(round(values[dd]))) for dd in ds[:npts]], x)
    ok = all(sympy.nsimplify(p.subs(x, dd) - int(round(values[dd]))) == 0 for dd in ds)
    return sympy.expand(p), ok

### D4 ライブラリ（境界付き D4 符号・transversal $U$・code switching・疎状態シミュレータ・資源モデル）

In [ ]:
"""d4_lib: D4 Clifford stabilizer code (Kobayashi-Zhu-Hsin) on the bounded square lattice, transversal U, code switching, resource model."""
import itertools, cmath
import numpy as np
from collections import Counter

R, G, B = 0, 1, 2
CNAME = "rgb"

In [ ]:
# ------------------------------------------------- phase-shift operators
class PSOp:
    """Operator  O |x> = (-1)^{f(x)} |x + s>  on n qubits (computational basis x in F2^n).

    f(x) = const + sum_{i in lin} x_i + sum_{(i,j) in quad} x_i x_j   (mod 2), i<j.
    X-type Clifford stabilizers of the D4 code, Pauli X/Z strings and products
    thereof all have this form.  Composition rule (A o B)|x> = A B |x>:
        s = s_A + s_B,   f(x) = f_B(x) + f_A(x + s_B).
    """
    __slots__ = ("s", "lin", "quad", "const")

    def __init__(self, s=frozenset(), lin=frozenset(), quad=frozenset(), const=0):
        self.s = frozenset(s)
        self.lin = frozenset(lin)
        self.quad = frozenset(tuple(sorted(p)) for p in quad)
        self.const = const & 1

    def shifted_phase(self, t):
        """Return (lin, quad, const) of x -> f(x + t)."""
        lin = set(self.lin)
        const = self.const
        for i in self.lin:
            if i in t:
                const ^= 1
        for (i, j) in self.quad:
            ti, tj = i in t, j in t
            if ti and tj:
                const ^= 1
            if tj:
                lin ^= {i}
            if ti:
                lin ^= {j}
        return frozenset(lin), self.quad, const

    def __matmul__(self, other):
        lA, qA, cA = self.shifted_phase(other.s)
        return PSOp(self.s ^ other.s, lA ^ other.lin, qA ^ other.quad, cA ^ other.const)

    def phase_at(self, x):
        """f(x) for x given as a set/frozenset of 1-positions."""
        v = self.const
        for i in self.lin:
            if i in x:
                v ^= 1
        for (i, j) in self.quad:
            if i in x and j in x:
                v ^= 1
        return v

    def apply(self, x):
        """O|x> = sign |x'> ; returns (x', sign_bit)."""
        return frozenset(x) ^ self.s, self.phase_at(x)

    def support(self):
        sup = set(self.s) | set(self.lin)
        for p in self.quad:
            sup |= set(p)
        return sup


def commutator_affine(A, B):
    """A B = (-1)^{g(x)} B A on |x>.  Returns g as (lin, const) -- quadratic terms cancel."""
    AB = A @ B
    BA = B @ A
    assert AB.s == BA.s
    q = AB.quad ^ BA.quad
    assert not q, "commutator has quadratic part (should not happen)"
    return AB.lin ^ BA.lin, AB.const ^ BA.const

In [ ]:
# ------------------------------------------------------------- the lattice
class D4Lattice:
    def __init__(self, L):
        assert L >= 1
        self.L = L
        self.edges = []          # (kind, x, y), kind 'h' or 'v'
        self.eidx = {}
        for y in range(L + 1):
            for x in range(L):
                self._add(('h', x, y))
        for x in range(L + 1):
            for y in range(L):
                self._add(('v', x, y))
        self.E = len(self.edges)
        self.n = 3 * self.E
        self.vertices = [(x, y) for y in range(L + 1) for x in range(L + 1)]
        self.plaquettes = [(x, y) for y in range(L) for x in range(L)]

    def _add(self, e):
        self.eidx[e] = len(self.edges)
        self.edges.append(e)

    def q(self, e, c):
        if not isinstance(e, int):
            e = self.eidx[e]
        return 3 * e + c

    # --- geometry --------------------------------------------------------
    def endpoints(self, e):
        k, x, y = self.edges[e] if isinstance(e, int) else e
        return ((x, y), (x + 1, y)) if k == 'h' else ((x, y), (x, y + 1))

    def incident(self, v):
        x, y = v
        out = []
        for e in [('h', x, y), ('h', x - 1, y), ('v', x, y), ('v', x, y - 1)]:
            if e in self.eidx:
                out.append(self.eidx[e])
        return out

    def plaq_edges(self, p):
        """(e01, e13, e02, e23) of plaquette p=(x,y)."""
        x, y = p
        ei = self.eidx
        return ei[('h', x, y)], ei[('v', x + 1, y)], ei[('v', x, y)], ei[('h', x, y + 1)]

    def plaq_boundary(self, p):
        return list(self.plaq_edges(p))

    def has_plaq(self, x, y):
        return 0 <= x < self.L and 0 <= y < self.L

    def boundary_of_edge(self, e):
        k, x, y = self.edges[e]
        L = self.L
        if k == 'v' and x == 0:
            return 'Lr'
        if k == 'h' and y == L:
            return 'Lb'
        if k == 'h' and y == 0:
            return 'Lrb'
        if k == 'v' and x == L:
            return 'Lrb'
        return None

    def boundary_edges(self, name):
        return [e for e in range(self.E) if self.boundary_of_edge(e) == name]

    # --- Z-type checks (rows of H_Z) --------------------------------------
    def z_checks(self, colours=(R, G, B)):
        """List of (label, set_of_qubits) Z-type stabilizer generators.
        Plaquettes for every colour + single-edge boundary conditions (Fig. 4 / eq. (7))."""
        rows = []
        for p in self.plaquettes:
            for c in colours:
                rows.append((('plaq', p, CNAME[c]), {self.q(e, c) for e in self.plaq_boundary(p)}))
        for e in range(self.E):
            bd = self.boundary_of_edge(e)
            if bd == 'Lr' and R in colours:
                rows.append((('bdZ', e, 'r'), {self.q(e, R)}))
            elif bd == 'Lb' and B in colours:
                rows.append((('bdZ', e, 'b'), {self.q(e, B)}))
            elif bd == 'Lrb':
                if R in colours and B in colours:
                    rows.append((('bdZ', e, 'rb'), {self.q(e, R), self.q(e, B)}))
                if G in colours:
                    rows.append((('bdZ', e, 'g'), {self.q(e, G)}))
        return rows

    # --- X-type Clifford stabilizers ----------------------------------------
    def cz_pairs(self, v, c):
        """CZ pairs (qubit, qubit) of the dressing of S^c_{X;v} (SM eq. for S_X, cup products
        evaluated with the hypercube formula; plaquettes outside the lattice are dropped)."""
        x, y = v
        pairs = []
        if c == R:      # int v u e' u e'' : v = vertex 0 of p(x,y);   CZ^{g,b}
            if self.has_plaq(x, y):
                e01, e13, e02, e23 = self.plaq_edges((x, y))
                pairs += [(self.q(e01, G), self.q(e13, B)), (self.q(e02, G), self.q(e23, B))]
        elif c == G:    # int e'' u v u e' : v = vertex 1 of p(x-1,y), vertex 2 of p(x,y-1); CZ^{r,b}
            if self.has_plaq(x - 1, y):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y))
                pairs.append((self.q(e01, R), self.q(e13, B)))
            if self.has_plaq(x, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x, y - 1))
                pairs.append((self.q(e02, R), self.q(e23, B)))
        else:           # int e' u e'' u v : v = vertex 3 of p(x-1,y-1); CZ^{r,g}
            if self.has_plaq(x - 1, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y - 1))
                pairs += [(self.q(e01, R), self.q(e13, G)), (self.q(e02, R), self.q(e23, G))]
        return pairs

    def SX(self, v, c, dressed=True):
        s = {self.q(e, c) for e in self.incident(v)}
        quad = self.cz_pairs(v, c) if dressed else []
        # CZ applied first, then X (S = (prod X)(prod CZ)); X and CZ act on disjoint colours.
        return PSOp(s=s, quad=quad)

    def vertex_x_types(self, v, colours=(R, G, B)):
        """Allowed colour combinations at vertex v: the F2 span of colour sets whose
        X-part commutes with the boundary Z conditions.  Returns a list of colour tuples
        (a basis), derived automatically from the boundary Z checks."""
        bdrows = [rows for (lab, rows) in self.z_checks(colours) if lab[0] == 'bdZ']
        inc = self.incident(v)
        cand = []
        for k in range(1, len(colours) + 1):
            for combo in itertools.combinations(colours, k):
                s = {self.q(e, c) for e in inc for c in combo}
                if all(len(s & rw) % 2 == 0 for rw in bdrows):
                    cand.append(combo)
        # reduce to an F2 basis over colour space, preferring single colours
        basis, vecs = [], []
        for combo in sorted(cand, key=len):
            vec = np.zeros(3, dtype=np.uint8)
            vec[list(combo)] = 1
            if f2_rank(np.array(vecs + [vec])) > len(vecs):
                vecs.append(vec)
                basis.append(combo)
        return basis

    def x_generators(self, colours=(R, G, B), dressed=True):
        """List of (label, PSOp).  At a vertex where only a product of colours is allowed
        (L_rb), the generator is the ordered product S^r S^b (Fig. 4 shows S^r S^b there)."""
        gens = []
        for v in self.vertices:
            for combo in self.vertex_x_types(v, colours):
                op = PSOp()
                for c in combo:
                    op = op @ self.SX(v, c, dressed)
                gens.append((('X', v, ''.join(CNAME[c] for c in combo)), op))
        return gens

    def vec(self, qubits):
        v = np.zeros(self.n, dtype=np.uint8)
        v[list(qubits)] = 1
        return v

In [ ]:
# ---------------------------------------------------------- code data
class CodeData:
    """Linear-algebra data of a (D4 or folded-surface) code on a lattice."""

    def __init__(self, lat, colours=(R, G, B), dressed=True):
        self.lat = lat
        self.colours = tuple(colours)
        self.zlabels, zr = zip(*lat.z_checks(colours))
        self.HZ = np.array([lat.vec(r) for r in zr], dtype=np.uint8)
        self.xgens = lat.x_generators(colours, dressed)
        self.S = np.array([lat.vec(op.s) for _, op in self.xgens], dtype=np.uint8)
        self.cols = sorted({lat.q(e, c) for e in range(lat.E) for c in colours})
        self.HZr, self.HZpiv = f2_rref(self.HZ)
        self.Sr, self.Spiv = f2_rref(self.S)
        # logical representatives
        CZ = f2_nullspace(self.HZ[:, self.cols])
        self.t_log = None
        for row in CZ:
            t = np.zeros(lat.n, dtype=np.uint8)
            t[self.cols] = row
            if not f2_in_rowspace(self.Sr, self.Spiv, t):
                self.t_log = t
                break
        KS = f2_nullspace(self.S[:, self.cols])
        self.z_log = None
        for row in KS:
            u = np.zeros(lat.n, dtype=np.uint8)
            u[self.cols] = row
            if not f2_in_rowspace(self.HZr, self.HZpiv, u):
                self.z_log = u
                break
        self.k = len(self.cols) - f2_rank(self.HZ[:, self.cols]) - f2_rank(self.S[:, self.cols])


def distances(cd):
    """(d_Z, d_X_shift) : min weight of a logical Z-type Pauli, and min weight of an
    X-shift lying in C_Z but outside the stabilizer shift span."""
    dz, uz = min_weight_odd(cd.S, cd.t_log, cd.cols)
    dx, tx = min_weight_odd(cd.HZ, cd.z_log, cd.cols)
    return dz, dx, uz, tx

In [ ]:
# ------------------------------------------- transversal automorphism U = W V
class DiagPhase:
    """Diagonal unitary |x> -> omega^{w(x)} |x>, omega = exp(i pi/4), w integer mod 8.
    w(x) = sum_k c_k * prod_{i in mono_k} x_i  (monomials of degree <= 2)."""

    def __init__(self, terms=()):
        self.terms = list(terms)   # (coef mod 8, tuple_of_qubits)

    def value(self, x):
        v = 0
        for c, mono in self.terms:
            if all(i in x for i in mono):
                v += c
        return v % 8

    def support(self):
        s = set()
        for _, m in self.terms:
            s |= set(m)
        return s

    def terms_touching(self, qs):
        return DiagPhase([(c, m) for c, m in self.terms if set(m) & qs])


def W_operator(lat, cs_signs=(+1, -1), boundary_T=-1):
    """W of eq. (8):  prod_p CS^{r,b}_{e01,e13} (CS^{r,b}_{e02,e23})^dag  x  prod_{e in bdry_rb} (T^r_e)^dag.
    CS = diag(1,1,1,i) -> omega^{2 x y};  T^dag -> omega^{-x}."""
    terms = []
    for p in lat.plaquettes:
        e01, e13, e02, e23 = lat.plaq_edges(p)
        terms.append(((2 * cs_signs[0]) % 8, (lat.q(e01, R), lat.q(e13, B))))
        terms.append(((2 * cs_signs[1]) % 8, (lat.q(e02, R), lat.q(e23, B))))
    if boundary_T:
        for e in lat.boundary_edges('Lrb'):
            terms.append((boundary_T % 8, (lat.q(e, R),)))
    return DiagPhase(terms)


def V_map(lat, x):
    """V = prod_e CNOT^{(r,g)}_e CNOT^{(b,g)}_e :  x_g <- x_g + x_r + x_b  (set representation)."""
    x = set(x)
    out = set(x)
    for e in range(lat.E):
        qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
        if (qr in x) ^ (qb in x):
            out ^= {qg}
    return frozenset(out)


def V_support_map(lat, qs):
    """Qubits whose values determine (V x) restricted to qs."""
    out = set()
    for q in qs:
        e, c = divmod(q, 3)
        out.add(q)
        if c == G:
            out |= {3 * e + R, 3 * e + B}
    return out


def CZ_basis(cd):
    if not hasattr(cd, '_CZbasis'):
        nb = f2_nullspace(cd.HZ[:, cd.cols])
        full = np.zeros((nb.shape[0], cd.lat.n), dtype=np.uint8)
        full[:, cd.cols] = nb
        cd._CZbasis = full
    return cd._CZbasis


def project_CZ(cd, Q, functionals=()):
    """Exact image of C_Z under x -> (x|_Q, phi_1(x), ..., phi_k(x)) for F2-linear functionals
    phi_j (callables on a set of qubits, evaluated on basis vectors).  Returns a list of
    configurations (frozenset of Q-qubits) or, if functionals are given, (config, bits)."""
    Q = sorted(Q)
    Bs = CZ_basis(cd)
    P = Bs[:, Q]
    if functionals:
        extra = np.array([[phi(frozenset(np.nonzero(b)[0])) for phi in functionals] for b in Bs],
                         dtype=np.uint8).reshape(len(Bs), len(functionals))
        P = np.concatenate([P, extra], axis=1)
    Pr, piv = f2_rref(P)
    k = Pr.shape[0]
    assert k <= 24, f"local projection too large ({k})"
    out = []
    nq = len(Q)
    for bits in itertools.product((0, 1), repeat=k):
        v = np.zeros(P.shape[1], dtype=np.uint8)
        for b, row in zip(bits, Pr):
            if b:
                v ^= row
        cfg = frozenset(Q[i] for i in np.nonzero(v[:nq])[0])
        out.append((cfg, tuple(int(z) for z in v[nq:])) if functionals else cfg)
    return out


def check_conjugation(cd, Wop, gen_op, target_op):
    """Exhaustively check  U S U^dag == S'  on C_Z, where U = W V.
    Returns (ok, n_local_configs)."""
    lat = cd.lat
    As = V_map(lat, gen_op.s)
    if As != target_op.s:
        return False, 0
    # qubits on which the phase difference can depend
    Wt = Wop.terms_touching(set(As))
    Q = set(Wt.support()) | V_support_map(lat, gen_op.support()) | target_op.support() | set(As)
    configs = project_CZ(cd, Q)
    for x in configs:
        # omega^{ w(x+As) - w(x) + 4 f(A x) - 4 f'(x) }   (only W-terms touching As contribute)
        d = Wt.value(x ^ As) - Wt.value(x) + 4 * gen_op.phase_at(V_map(lat, x)) - 4 * target_op.phase_at(x)
        if d % 8:
            return False, len(configs)
    return True, len(configs)


def stabilizer_path_phase(cd, x, target):
    """Given x and target = x + (sum of generator shifts), apply generators to |x> and
    return the accumulated sign bit, i.e. psi(target) = (-1)^bit psi(x) for code states."""
    shifts = cd.S
    delta = cd.lat.vec(set(x) ^ set(target))
    coeff = f2_solve(shifts, delta)
    if coeff is None:
        return None
    bit = 0
    cur = frozenset(x)
    for i in np.nonzero(coeff)[0]:
        op = cd.xgens[i][1]
        cur, b = op.apply(cur)
        bit ^= b
    assert cur == frozenset(target)
    return bit


def logical_phases(cd, Wop):
    """Eigenphases (in units of pi/4) of U on |0_L>, |1_L>, assuming U preserves the code space
    (checked separately).  Uses x0 = 0 and x1 = t_log."""
    lat = cd.lat
    res = []
    for x in [frozenset(), frozenset(np.nonzero(cd.t_log)[0])]:
        Ax = V_map(lat, x)
        bit = stabilizer_path_phase(cd, x, Ax)
        assert bit is not None, "V does not map the coset to itself"
        res.append((Wop.value(Ax) + 4 * bit) % 8)
    return res

In [ ]:
# ------------------------------------------- per-configuration protocol phase
def lrb_vertices(lat):
    vs = set()
    for e in lat.boundary_edges('Lrb'):
        vs |= set(lat.endpoints(e))
    return vs


def cprime(lat, a):
    """0-cochain c' with  delta c' = a_r + a_b  and c' = 0 on L_rb vertices (BFS).
    a: set of qubits (configuration of r,b qubits).  Returns dict vertex -> bit.
    Raises if a_r + a_b is not exact relative to L_rb."""
    from collections import deque
    val = {v: 0 for v in lrb_vertices(lat)}
    dq = deque(val)
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            bit = val[u] ^ (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
            if w in val:
                if val[w] != bit:
                    raise ValueError("a_r + a_b not exact")
            else:
                val[w] = bit
                dq.append(w)
    return val


def g_phase_poly(lat, v):
    """CZ^{r,b} pairs of S^g_{X;v}  (f^g_v)."""
    return lat.cz_pairs(v, G)


def protocol_phase(lat, Wop, a, gverts):
    """Phase exponent (mod 8, units of pi/4) acquired by FSC configuration a through
    switch-in (S^g measurement) -> U -> Z^g measurement -> S^g correction -> Pauli frame,
    i.e.  F(a) = w(a) + 4 * sum_v c'_v(a) f^g_v(a)."""
    cp = cprime(lat, a)
    phi = 0
    for v in gverts:
        if cp[v]:
            for (i, j) in g_phase_poly(lat, v):
                if i in a and j in a:
                    phi ^= 1
    return (Wop.value(a) + 4 * phi) % 8


def check_protocol_invariance(lat, fsc, Wop, gverts, gen_op):
    """Exhaustive local proof that F(a + t) = F(a) (mod 8) for all a in C_FSC, t = shift of gen_op."""
    t = frozenset(gen_op.s)
    # c'(t) (linear in a)
    ct = cprime(lat, t)
    N2 = {v for v in gverts if ct[v]}
    N1 = {v for v in gverts if set(q for pr in g_phase_poly(lat, v) for q in pr) & t}
    Wt = Wop.terms_touching(set(t))
    Q = set(Wt.support()) | set(t)
    for v in N1 | N2:
        Q |= {q for pr in g_phase_poly(lat, v) for q in pr}
    # paths from a reference vertex w0 to each w in N1 (BFS on the lattice graph)
    w0 = min(N1 | N2 | {lat.endpoints(next(iter(t)) // 3)[0]})
    paths = {}
    from collections import deque
    prev = {w0: None}
    dq = deque([w0])
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            if w not in prev:
                prev[w] = (u, e)
                dq.append(w)
    for w in N1:
        es = []
        cur = w
        while prev[cur] is not None:
            u, e = prev[cur]
            es.append(e)
            cur = u
        paths[w] = es
        Q |= {lat.q(e, c) for e in es for c in (R, B)}
    beta_fn = lambda b: cprime(lat, b)[w0]          # c'_{w0}(a) is F2-linear in a
    configs = project_CZ(fsc, Q, functionals=[beta_fn])
    for a, (beta,) in configs:
        if True:
            cpa = {}
            for w in N1:
                bit = beta
                for e in paths[w]:
                    bit ^= (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
                cpa[w] = bit
            at = a ^ t
            d = Wt.value(at) - Wt.value(a)
            phi = 0
            for w in N1:
                if cpa[w]:
                    for (i, j) in g_phase_poly(lat, w):
                        phi ^= ((i in at) and (j in at)) ^ ((i in a) and (j in a))
            for w in N2:
                for (i, j) in g_phase_poly(lat, w):
                    phi ^= ((i in at) and (j in at))
            if (d + 4 * phi) % 8:
                return False, len(configs)
    return True, len(configs)


OMEGA = cmath.exp(1j * cmath.pi / 4)


class SparseState:
    def __init__(self, amps):
        self.a = {k: v for k, v in amps.items() if abs(v) > 1e-14}

    def norm2(self):
        return sum(abs(v) ** 2 for v in self.a.values())

    def normalize(self):
        n = self.norm2() ** 0.5
        self.a = {k: v / n for k, v in self.a.items()}
        return self

    def apply_psop(self, op):
        out = {}
        for x, v in self.a.items():
            y, b = op.apply(x)
            out[y] = out.get(y, 0) + (-v if b else v)
        return SparseState(out)

    def apply_diag(self, D):
        return SparseState({x: v * OMEGA ** D.value(x) for x, v in self.a.items()})

    def apply_perm(self, f):
        out = {}
        for x, v in self.a.items():
            y = f(x)
            out[y] = out.get(y, 0) + v
        return SparseState(out)

    def project(self, op, sign):
        """(1 + sign*O)/2 |psi>  (unnormalised)."""
        Opsi = self.apply_psop(op).a
        out = {}
        for k in set(self.a) | set(Opsi):
            out[k] = 0.5 * (self.a.get(k, 0) + sign * Opsi.get(k, 0))
        return SparseState(out)

    def expect(self, op):
        Opsi = self.apply_psop(op).a
        return sum(np.conj(self.a.get(k, 0)) * v for k, v in Opsi.items())

    def measure_op(self, op, rng, forced=None):
        """Projective measurement of a Hermitian involution O; returns (outcome bit, state)."""
        p0 = self.project(op, +1)
        n0 = p0.norm2()
        if forced is None:
            bit = 0 if rng.random() < n0 else 1
        else:
            bit = forced
        st = p0 if bit == 0 else self.project(op, -1)
        prob = n0 if bit == 0 else 1 - n0
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def measure_z(self, q, rng, forced=None):
        p1 = sum(abs(v) ** 2 for x, v in self.a.items() if q in x)
        bit = (1 if rng.random() < p1 else 0) if forced is None else forced
        st = SparseState({x: v for x, v in self.a.items() if (q in x) == bool(bit)})
        prob = p1 if bit else 1 - p1
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def overlap(self, other):
        return sum(np.conj(v) * other.a.get(k, 0) for k, v in self.a.items())


def coset_state(cd, rep, phases=True):
    """Code state supported on rep + span(shifts):  psi(rep)=1, psi(x+s_v)=(-1)^{f_v(x)} psi(x)."""
    amps = {frozenset(rep): 1.0}
    # breadth-first application of generators
    frontier = [frozenset(rep)]
    while frontier:
        new = []
        for x in frontier:
            for _, op in cd.xgens:
                y, b = op.apply(x)
                if y not in amps:
                    amps[y] = amps[x] * (-1 if b else 1)
                    new.append(y)
                else:
                    if phases:
                        assert abs(amps[y] - amps[x] * (-1 if b else 1)) < 1e-9, "inconsistent code state"
        frontier = new
    return SparseState(amps).normalize()




def frozen_qubits(lat):
    """Qubits fixed to |0> by single-qubit boundary Z stabilizers (L_r: r, L_b: b, L_rb: g)."""
    fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1:
            fz |= rows
    return fz


class D4Protocol:
    def __init__(self, L, prune_frozen=True):
        self.L = L
        self.lat = lat = D4Lattice(L)
        self.frozen = frozen_qubits(lat) if prune_frozen else set()
        self.prune = prune_frozen
        self.d4 = CodeData(lat)
        self.fsc = CodeData(lat, (R, B), dressed=False)
        # ancilla allocation: one ancilla per stabilizer generator label (D4 superset)
        self.anc = {}
        nxt = lat.n
        labels = [lab for lab, _ in self.d4.xgens] + list(self.d4.zlabels)
        labels += [lab for lab, _ in self.fsc.xgens] + list(self.fsc.zlabels)
        for lab in labels:
            key = self._anc_key(lab)
            if key not in self.anc and not self._is_frozen_check(lab):
                self.anc[key] = nxt
                nxt += 1
        self.gverts = [lab[1] for lab, op in self.d4.xgens if lab[2] == 'g']

    @staticmethod
    def _anc_key(lab):
        return lab

    def _is_frozen_check(self, lab):
        return lab[0] == 'bdZ' and lab[2] in ('r', 'b', 'g') and self.prune

    def data_qubits(self, colours):
        return {self.lat.q(e, c) for e in range(self.lat.E) for c in colours} - self.frozen

    # ---------------------------------------------------------- gadgets
    def _keep_pair(self, i, j):
        return i not in self.frozen and j not in self.frozen

    def x_gadget(self, lab, dressed):
        _, v, cols = lab
        parts = []
        for ch in cols:
            c = CNAME.index(ch)
            xq = {self.lat.q(e, c) for e in self.lat.incident(v)} - self.frozen
            pairs = [p for p in self.lat.cz_pairs(v, c) if self._keep_pair(*p)] if dressed else []
            parts.append((xq, pairs))
        return clifford_x_gadget(self.anc[lab], parts)

    def z_gadget(self, lab, rows):
        return z_check_gadget(self.anc[lab], set(rows) - self.frozen)

    def round_gadgets(self, code, which='all'):
        """Gadgets of one stabilizer round.  code: 'D4' or 'FSC'.
        which: 'all' | 'X' | 'Z' | 'g' (D4 S^g only)."""
        cd = self.d4 if code == 'D4' else self.fsc
        dressed = code == 'D4'
        gads = []
        if which in ('all', 'X', 'g'):
            for lab, op in cd.xgens:
                if which == 'g' and lab[2] != 'g':
                    continue
                gads.append(self.x_gadget(lab, dressed))
        if which in ('all', 'Z'):
            for lab, (l2, rows) in zip(cd.zlabels, self.lat.z_checks(cd.colours)):
                if self._is_frozen_check(lab):
                    continue
                gads.append(self.z_gadget(lab, rows))
        return gads

    # ---------------------------------------------------------- transversal U
    def U_gates(self):
        lat, fz = self.lat, self.frozen
        gates = []
        for e in range(lat.E):
            qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
            for ctrl in (qr, qb):
                if ctrl in fz:
                    continue
                if qg in fz:
                    continue   # L_rb: CNOT(r,g) CNOT(b,g) = identity on Z^r Z^b = +1 (g removed)
                gates.append(('CNOT', ctrl, qg))
        for p in lat.plaquettes:
            e01, e13, e02, e23 = lat.plaq_edges(p)
            for (a, b, name) in [(lat.q(e01, R), lat.q(e13, B), 'CS'), (lat.q(e02, R), lat.q(e23, B), 'CSDG')]:
                if a in fz or b in fz:
                    continue
                gates.append((name, a, b))
        for e in lat.boundary_edges('Lrb'):
            gates.append(('TDG', lat.q(e, R)))
        return gates

    # ---------------------------------------------------------- switch-out
    def switch_out_gates(self, worst_case=True):
        lat = self.lat
        gq = sorted(self.data_qubits((G,)))
        gates = [('MEAS_Z', q) for q in gq]
        cz = []
        for v in self.gverts:
            for p in lat.cz_pairs(v, G):
                if self._keep_pair(*p):
                    cz.append(('CZ',) + tuple(p))
        return gates, cz      # cz: worst-case correction (all vertices in kappa)

    def qubit_sets(self):
        fsc_data = self.data_qubits((R, B))
        d4_data = self.data_qubits((R, G, B))
        fsc_anc = {self.anc[l] for l, _ in self.fsc.xgens} | {self.anc[l] for l in self.fsc.zlabels
                                                              if not self._is_frozen_check(l)}
        d4_anc = {self.anc[l] for l, _ in self.d4.xgens} | {self.anc[l] for l in self.d4.zlabels
                                                            if not self._is_frozen_check(l)}
        return dict(fsc_data=fsc_data, d4_data=d4_data, fsc_anc=fsc_anc, d4_anc=d4_anc)

In [ ]:
# ----------------------------------------------------------------- Kobayashi (D4)
class D4CostData:
    """All d-dependent counts of the D4 protocol, computed once per L from explicit circuits."""

    def __init__(self, L, dur=None, decomposed=False):
        dur = dur or DUR_UNIT
        P = D4Protocol(L)
        self.L = L
        qs = P.qubit_sets()
        self.n = {k: len(v) for k, v in qs.items()}
        self.n_total = len(qs['d4_data'] | qs['d4_anc'])
        dec = decompose if decomposed else (lambda g: g)
        gad = {
            'D4': [dec(g) for g in P.round_gadgets('D4')],
            'D4g': [dec(g) for g in P.round_gadgets('D4', 'g')],
            'FSC': P.round_gadgets('FSC'),
        }
        self.round = {}
        for k, g in gad.items():
            ub = round_schedule_classes(g, dur)
            lb = round_lower_bound(g, dur)
            self.round[k] = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'], depth_lb=lb['depth_lb'],
                                 depth_ub=ub['depth_ub'], classes=ub['classes'],
                                 gates=Counter(gate_counts([x for gg in g for x in gg])),
                                 n_anc=len(g))
        self.n_anc_g = len(gad['D4g'])
        U = dec(P.U_gates())
        Ul = class_layers([U])              # all U gates mutually commute (validated)
        self.U = dict(gates=Counter(gate_counts(U)), time=circuit_time(Ul, dur), depth=len(Ul))
        meas, cz = P.switch_out_gates()
        czl = class_layers([cz]) if cz else []
        ml = [meas]
        self.out = dict(gates_worst=Counter(gate_counts(meas + cz)),
                        gates_expected=Counter({**gate_counts(meas), 'CZ': len(cz) / 2}),
                        time=circuit_time(ml + czl, dur), depth=len(ml) + len(czl))
        self.x_corr = dict(time=dur['X'], gates_worst=Counter({'X': self.n['fsc_data']}))
        self.init = Counter({'INIT_X': self.n['fsc_data'], 'INIT_Z': len(qs['d4_data']) - self.n['fsc_data']})


def d4_stages(cd, model, params=DEFAULT_PARAMS, comparison='A', data_only=False):
    d = cd.L
    n = cd.n
    fsc_q = n['fsc_data'] + (0 if data_only else n['fsc_anc'])
    d4_q = n['d4_data'] + (0 if data_only else n['d4_anc'])
    rows = []

    def add(stage, qubits, rounds, rkey, extra_time, gates, source, note=''):
        rt = cd.round[rkey] if rkey else dict(time_lb=0, time_ub=0)
        rows.append(dict(protocol='Kobayashi_D4', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    r0 = rounds_for(model, params['kappa0'], d)
    g0 = Counter(cd.init)
    for k, v in cd.round['FSC']['gates'].items():
        g0[k] += v * r0
    g0['X'] += cd.x_corr['gates_worst']['X']
    add('S0 FSC |+_L> preparation', fsc_q, r0, 'FSC', cd.x_corr['time'], g0,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'kappa0*d rounds (not specified in Kobayashi); X-correction worst case counted')
    if model == 'NL_min':
        r1, key = 1, 'D4g'
        q1 = n['d4_data'] + (0 if data_only else cd.n_anc_g)
        note1 = 'noiseless minimum: one round of S^g_X measurements only'
    else:
        r1, key = rounds_for(model, params['kappa_in'], d), 'D4'
        q1 = d4_q
        note1 = 'O(d) rounds of all D4 stabilizers (Kobayashi step 2); kappa_in*d assumed'
    g1 = Counter()
    for k, v in cd.round[key]['gates'].items():
        g1[k] += v * r1
    add('S1 switch-in (gauging) to D4', q1, r1, key, 0, g1,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT, note1)
    add('S2 transversal U = W V (logical T^dag)', n['d4_data'], 0, None, cd.U['time'], cd.U['gates'],
        SOURCE_COUNT, 'constant depth; gates from eq. (8)')
    add('S3 switch-out: Z^g meas + S^g correction', n['d4_data'], 0, None, cd.out['time'],
        cd.out['gates_worst'], SOURCE_COUNT, 'CZ correction counted worst case (expected = half)')
    r4 = 0 if model == 'NL_min' else rounds_for(model, params['kappa_out'], d)
    g4 = Counter()
    for k, v in cd.round['FSC']['gates'].items():
        g4[k] += v * r4
    add('S4 FSC error-correction rounds', fsc_q, r4, 'FSC', 0, g4,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'O(d) rounds (Kobayashi step 4); kappa_out*d assumed; 0 in the noiseless minimum')
    add('S5 logical X frame: T^dag|+> -> T|+>', 0, 0, None, 0, Counter(), SOURCE_COUNT,
        'Pauli frame update, no physical cost')
    # NOTE (comparison B): the FSC -> standard rotated-patch conversion is UNRESOLVED.  It is deliberately
    # NOT added as a zero-cost stage; it is reported separately by uncounted_costs().
    return rows



def d4_conversion_illustrative(cd, model, params=DEFAULT_PARAMS, data_only=False):
    """ILLUSTRATIVE ONLY: conversion modelled as one lattice-surgery time step (tau) during which the
    FSC plus 2 tiles (target patch + ancilla strip) are active.  Not taken from literature."""
    d = cd.L
    r = rounds_for(model, params['tau_rounds'], d)
    q = cd.n['fsc_data'] + (0 if data_only else cd.n['fsc_anc']) + 2 * (d * d if data_only else 2 * d * d)
    return dict(protocol='Kobayashi_D4', comparison='B_illustrative', model=model, d=d,
                stage='S6 conversion (ILLUSTRATIVE: 1 tau lattice surgery)', qubits=q, rounds=r,
                time_lb=None, time_ub=None, gates={}, source=SOURCE_UNRES,
                note='illustrative assumption, not a validated protocol')

In [ ]:
# ---------------------------------------------------------- consistency checks of a code
def code_consistency(cd):
    """Exact F2 checks of a (possibly non-commuting) Clifford stabilizer code on its Z-subspace C_Z:
      shift_in_CZ : every X-type generator preserves C_Z (its X part commutes with all Z checks)
      noncommuting_pairs : generator pairs that do NOT commute on C_Z
      bad_squares : generators with S^2 != 1 on C_Z
      k : number of logical qubits = dim C_Z - rank(shifts)   (generators have independent shifts)"""
    lat = cd.lat
    HZ, S = cd.HZ, cd.S
    res = {}
    res['shift_in_CZ'] = not ((HZ.astype(int) @ S.T.astype(int)) % 2).any()
    supp = [op.support() for _, op in cd.xgens]
    bad = []
    for i in range(len(cd.xgens)):
        for j in range(i + 1, len(cd.xgens)):
            if not (supp[i] & supp[j]):
                continue
            lin, const = commutator_affine(cd.xgens[i][1], cd.xgens[j][1])
            if const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(lin)):
                bad.append((cd.xgens[i][0], cd.xgens[j][0]))
    res['noncommuting_pairs'] = bad
    sq = []
    for lab, op in cd.xgens:
        s2 = op @ op
        if s2.s or s2.quad or s2.const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(s2.lin)):
            sq.append(lab)
    res['bad_squares'] = sq
    res['n_qubits'] = len(cd.cols)
    res['dim_CZ'] = len(cd.cols) - f2_rank(HZ[:, cd.cols])
    res['rank_shifts'] = f2_rank(S[:, cd.cols])
    res['n_xgens'] = len(cd.xgens)
    res['k'] = res['dim_CZ'] - res['rank_shifts']
    res['shifts_independent'] = res['rank_shifts'] == res['n_xgens']
    return res


def U_target(lat, lab):
    """Image of generator `lab` under U predicted by SM eq. (permutingstabilizers):
    S^r -> S^r S^g, S^g -> S^g, S^b -> S^g S^b, and (boundary L_rb) S^r S^b -> S^r S^b."""
    _, v, cols = lab
    if cols == 'rb':
        return lat.SX(v, R) @ lat.SX(v, B)
    op = PSOp()
    for ch in cols:
        c = CNAME.index(ch)
        for cc in ([c] if c == G else [c, G]):
            op = op @ lat.SX(v, cc)
    return op

In [ ]:
# ---------------------------------------------------------- Knill-Laflamme distance lower bound
def _milp_binary(c_obj, A_eq_mod2, rhs_mod2, extra_ub=None, fixed_zero=()):
    """min c.x over binary x subject to A x = rhs (mod 2) (rows of A_eq_mod2), and optional
    linear constraints extra_ub = [(row, ub), ...] (row.x <= ub).
    Returns (value, x) if an optimum was found, (None, None) only if the problem is proven
    infeasible (scipy status 2).  Any other status (time/iteration limit, unbounded, numerical
    failure) raises RuntimeError, so that a solver failure is never mistaken for "no solution"."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    A = np.array(A_eq_mod2, dtype=float)
    m, n = A.shape
    nv = n + m
    c = np.concatenate([np.asarray(c_obj, float), np.zeros(m)])
    cons = [LinearConstraint(np.concatenate([A, -2 * np.eye(m)], axis=1), rhs_mod2, rhs_mod2)]
    for row, ub in (extra_ub or []):
        cons.append(LinearConstraint(np.concatenate([np.asarray(row, float), np.zeros(m)])[None, :], -np.inf, ub))
    ub_ = np.concatenate([np.ones(n), np.full(m, np.inf)])
    for i in fixed_zero:
        ub_[i] = 0
    res = milp(c, constraints=cons, integrality=np.ones(nv), bounds=Bounds(np.zeros(nv), ub_))
    if res.status == 2:
        return None, None
    if res.status != 0:
        raise RuntimeError(f'MILP solver did not finish (status {res.status}: {res.message})')
    return round(res.fun), np.round(res.x[:n]).astype(np.uint8)


def low_weight_stabilizer_shifts(cd, wmax, frozen=()):
    """Exact enumeration (MILP with no-good cuts) of all t in span(S) with 1 <= |t| <= wmax,
    where span(S) = C_Z  intersect  z_log^perp  (k = 1).  Frozen qubits are excluded
    (an X on a frozen qubit violates its single-qubit Z check, so t would leave C_Z).
    Returns (found, stop) where stop records why the enumeration ended:
    'infeasible' (no further t with 1 <= |t| <= wmax) or 'min weight > wmax'."""
    cols = [c for c in cd.cols if c not in set(frozen)]
    H = np.vstack([cd.HZ[:, cols], cd.z_log[cols][None, :]])
    H = H[np.any(H, axis=1)]
    rhs = np.zeros(len(H))
    n = len(cols)
    cuts = [(np.ones(n), wmax), (-np.ones(n), -1)]
    found = []
    while True:
        val, x = _milp_binary(np.ones(n), H, rhs, extra_ub=cuts)
        if val is None:
            stop = 'infeasible'
            break
        if val > wmax:
            stop = 'min weight > wmax'
            break
        sup = np.nonzero(x)[0]
        t = np.zeros(cd.lat.n, dtype=np.uint8)
        t[[cols[i] for i in sup]] = 1
        found.append(t)
        row = -np.ones(n)
        row[sup] = 1
        cuts.append((row, len(sup) - 1))                  # exclude exactly this vector
    return found, stop


def kl_distance_lower_bound(cd, wmax, frozen=()):
    r"""Rigorous lower-bound argument for the Knill-Laflamme distance of the (non-Pauli) code cd.

    For a Pauli E = X^t Z^u (up to phase):
      * t not in C_Z           -> E maps the code space to an orthogonal space: detectable;
      * t in the logical coset -> |t| >= X-shift distance (checked separately);
      * t in span(S)           -> X^t = S_c F_c with S_c a stabilizer and F_c = (-1)^{f_c} diagonal,
        so P E P = P F_c Z^u P.  Expanding (-1)^{f_c} in Z-type Paulis Z^v with v supported on
        Q_f = supp(f_c), a violation of KL requires some v + u to be a logical Z operator l.
        Hence |E| >= |t| + |u \ t| >= |t| + min_l |l \ (Q_f u t)|  =: LB_t.
    Returns (min over enumerated t of LB_t, ts, details, stop) where stop is the termination
    reason of the enumeration (see low_weight_stabilizer_shifts).  t = 0 is covered by d_Z."""
    lat = cd.lat
    ts, stop = low_weight_stabilizer_shifts(cd, wmax, frozen)
    cols = [c for c in cd.cols if c not in set(frozen)]
    Srows = cd.S[:, cols]
    Srows = Srows[np.any(Srows, axis=1)]
    best, details = None, []
    for t in ts:
        coef = f2_solve(cd.S, t)
        op = PSOp()
        for i in np.nonzero(coef)[0]:
            op = op @ cd.xgens[i][1]
        assert op.s == frozenset(np.nonzero(t)[0])
        Qf = {q for pr in op.quad for q in pr} | set(op.lin) | set(np.nonzero(t)[0])
        cost = np.array([0.0 if c in Qf else 1.0 for c in cols])
        A = np.vstack([Srows, cd.t_log[cols][None, :]])
        rhs = np.concatenate([np.zeros(len(Srows)), [1.0]])
        val, _ = _milp_binary(cost, A, rhs)
        assert val is not None                            # a logical Z always exists
        lb = int(t.sum()) + val
        details.append(dict(weight_t=int(t.sum()), Qf_size=len(Qf), LB=lb))
        best = lb if best is None else min(best, lb)
    return best, ts, details, stop

In [ ]:
# ---------------------------------------------------------- full resource tables (used by main + comparison)
def d4_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Kobayashi protocol for every d in D_LIST, gate set
    ('primitive' | 'clifford_t'), counting variant (data_only), time model and comparison A/B.
    Returns (stage_df, sum_df, blocks)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, blocks = [], [], {}
    for d in D_LIST:
        t0 = time.time()
        for gs in ['primitive', 'clifford_t']:
            cd = D4CostData(d, dur, decomposed=(gs == 'clifford_t'))
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = d4_stages(cd, model, params, comp, data_only=data_only)
                        for r in st:
                            r.update(gateset=gs, data_only=data_only)
                        records += st
                        s = summarize(st)
                        s.update(protocol='Kobayashi_D4', d=d, model=model, comparison=comp, gateset=gs,
                                 data_only=data_only)
                        summaries.append(s)
            blocks[f'{d}|{gs}'] = dict(d=d, gateset=gs, n=cd.n, n_total=cd.n_total, n_anc_g=cd.n_anc_g,
                                       round={k: {kk: (dict(vv) if isinstance(vv, Counter) else vv) for kk, vv in v.items()}
                                              for k, v in cd.round.items()},
                                       U_time=cd.U['time'], U_gates=dict(cd.U['gates']), out_time=cd.out['time'],
                                       out_gates=dict(cd.out['gates_worst']), xcorr_time=cd.x_corr['time'])
        if verbose:
            print(f'Kobayashi d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), blocks

### 15-to-1 ライブラリ（行列・状態ベクトル回路・Fig. 17c 配置・多パッチ格子手術の明示的構成・資源モデル）

In [ ]:
"""Litinski 'A Game of Surface Codes' Sec. 3: 15-to-1 distillation.
Logical-level circuits (statevector) + tile-level layout model + physical (patch-level) model."""
import itertools
import numpy as np

In [ ]:
# ----------------------------------------------------------------- matrices (transcribed)
# Eq. (1) of Litinski: rows 1-4 = X stabilizers, row 5 = logical X, columns = 15 qubits
M15 = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],
    [0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1],
    [1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1]], dtype=np.uint8)
# Eq. (2): triorthogonal matrix G ; Eq. (4): its row-echelon form G~
G_TRI = np.array([[1] * 16,
                  [0] * 8 + [1] * 8,
                  [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
                  [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
                  [0, 1] * 8], dtype=np.uint8)
G_TILDE = np.array([
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1],
    [0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1]], dtype=np.uint8)


def triorthogonal(G):
    G = G.astype(int)
    a = all(G[i].sum() % 8 == 0 for i in range(len(G)))
    b = all((G[i] * G[j]).sum() % 4 == 0 for i, j in itertools.combinations(range(len(G)), 2))
    c = all((G[i] * G[j] * G[k]).sum() % 2 == 0 for i, j, k in itertools.combinations(range(len(G)), 3))
    return a, b, c

In [ ]:
# ----------------------------------------------------------------- statevector helpers
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
PAULI = {'I': I2, 'X': X, 'Y': Y, 'Z': Z}
KET0 = np.array([1, 0], dtype=complex)
KETP = np.array([1, 1], dtype=complex) / np.sqrt(2)
T_GATE = np.diag([1, np.exp(1j * np.pi / 4)])
KET_M = T_GATE @ KETP                 # |m> = |0> + e^{i pi/4}|1>  (normalised)


def kron_all(ops):
    out = np.array([[1]], dtype=complex)
    for o in ops:
        out = np.kron(out, o)
    return out


def pauli_on(n, paulistr_by_qubit):
    """Dense Pauli operator; paulistr_by_qubit: dict qubit -> 'X'|'Y'|'Z'.  Qubit 0 = most significant."""
    return kron_all([PAULI[paulistr_by_qubit.get(i, 'I')] for i in range(n)])


def apply_1q(psi, n, q, U):
    psi = psi.reshape([2] * n)
    psi = np.moveaxis(np.tensordot(U, psi, axes=([1], [q])), 0, q)
    return psi.reshape(-1)


def z_rotation_diag(n, qubits, phi):
    """Diagonal of exp(-i phi Z_{q1} Z_{q2} ...)."""
    idx = np.arange(2 ** n)
    par = np.zeros(2 ** n, dtype=int)
    for q in qubits:
        par ^= (idx >> (n - 1 - q)) & 1
    return np.exp(-1j * phi * (1 - 2 * par))


def measure_pauli(psi, P, rng=None, forced=None):
    """Projective measurement of Hermitian Pauli matrix P.  Returns (outcome +-1, post-state, prob)."""
    Ppsi = P @ psi
    p_plus = np.real(np.vdot(psi, psi + Ppsi)) / 2
    if forced is None:
        out = 1 if rng.random() < p_plus else -1
    else:
        out = forced
    post = (psi + out * Ppsi) / 2
    prob = p_plus if out == 1 else 1 - p_plus
    if prob > 1e-12:
        post = post / np.sqrt(prob)
    return out, post, prob


def equal_up_to_phase(a, b, tol=1e-9):
    ov = np.vdot(a, b)
    return abs(abs(ov) - np.linalg.norm(a) * np.linalg.norm(b)) < tol

In [ ]:
# ----------------------------------------------------------------- 15-to-1, 5-qubit circuit (Fig. 15)
def fig15_rotations():
    """The 11 Z-type pi/8 rotations of Fig. 15 = columns 5..15 of M15 (rows = qubits 1..5).
    Columns 1..4 (weight-1 on qubits 4,3,2,1) are absorbed into the |m> initial states."""
    rots = []
    for j in range(4, 15):
        rots.append(tuple(int(i) for i in np.nonzero(M15[:, j])[0]))
    return rots


def run_15to1_5qubit(inputs=None, phi=np.pi / 8, rng=None):
    """Simulate Fig. 15.  inputs: list of 4 single-qubit states for qubits 1-4 (default ideal |m>).
    Returns dict outcome-tuple -> (probability, output state of qubit 5)."""
    n = 5
    if inputs is None:
        inputs = [KET_M] * 4
    psi = kron_all([s.reshape(2, 1) for s in inputs] + [KETP.reshape(2, 1)]).reshape(-1)
    for qs in fig15_rotations():
        psi = z_rotation_diag(n, qs, phi) * psi
    # measure qubits 1-4 in X basis: enumerate all outcomes
    res = {}
    for outs in itertools.product((1, -1), repeat=4):
        st = psi.copy()
        prob = 1.0
        for q, o in enumerate(outs):
            _, st, p = measure_pauli(st, pauli_on(n, {q: 'X'}), forced=o)
            prob *= p
            if p < 1e-14:
                break
        if prob < 1e-14:
            res[outs] = (0.0, None)
            continue
        # reduced state of qubit 5 (qubits 1-4 are in X eigenstates -> product)
        st = st.reshape(16, 2)
        # project onto the X-eigenstates of qubits 1..4
        xs = [np.array([1, o], dtype=complex) / np.sqrt(2) for o in outs]
        bra = kron_all([x.reshape(1, 2) for x in xs]).reshape(-1)
        out5 = bra.conj() @ st
        out5 = out5 / np.linalg.norm(out5)
        res[outs] = (prob, out5)
    return res

In [ ]:
# ----------------------------------------------------------------- 15-qubit code check (Fig. 14)
def rm15_logical_check(gate='T'):
    """Prepare |+_L> of the 15-qubit code (X stabilisers = rows 1-4 of M15, logical X = row 5,
    Z stabilisers = F2-orthogonal complement), apply transversal T (or T^dag) and return the
    logical expectation values <X_L>, <Y_L-like> to identify the logical gate."""
    n = 15
    from_rows = M15.astype(int)
    # Z stabilisers: nullspace of M15 over F2
    Hz = f2_nullspace_simple(M15)
    dim = 2 ** n
    psi = np.zeros(dim, dtype=complex)
    psi[0] = 1.0
    idx = np.arange(dim)
    # |+_L> = prod (1+X_s)/2 (1+X_L)/2 |0..0>  (|0..0> is +1 eigenstate of all Z stabilisers)
    for row in from_rows:
        mask = int(''.join(str(b) for b in row), 2)
        psi = (psi + psi[idx ^ mask]) / 2
    psi /= np.linalg.norm(psi)
    # transversal gate
    w = np.array([bin(i).count('1') for i in idx])
    ph = np.exp(1j * np.pi / 4 * w) if gate == 'T' else np.exp(-1j * np.pi / 4 * w)
    psi = ph * psi
    # logical operators: X_L = X(row5), Z_L = Z on a vector z with z.row5 = 1, z.rows1-4 = 0
    xl = int(''.join(str(b) for b in from_rows[4]), 2)
    zvec = find_logical_z(M15)
    zl = int(''.join(str(b) for b in zvec), 2)
    par = np.array([bin(i & zl).count('1') % 2 for i in idx])
    Zl_psi = (1 - 2 * par) * psi
    Xl_psi = psi[idx ^ xl]
    # Y_L = i X_L Z_L
    Yl_psi = 1j * Zl_psi[idx ^ xl]
    ex = np.real(np.vdot(psi, Xl_psi))
    ey = np.real(np.vdot(psi, Yl_psi))
    ez = np.real(np.vdot(psi, Zl_psi))
    # stabiliser expectation values (X stabilisers should be lost only up to the Clifford
    # correction; Z stabilisers preserved)
    zst = []
    for hz in Hz:
        m = int(''.join(str(b) for b in hz), 2)
        par = np.array([bin(i & m).count('1') % 2 for i in idx])
        zst.append(np.real(np.vdot(psi, (1 - 2 * par) * psi)))
    return dict(X_L=ex, Y_L=ey, Z_L=ez, min_Zstab=min(zst), n_Zstab=len(Hz))


def f2_nullspace_simple(M):
    M = M.copy() % 2
    rows, cols = M.shape
    A = M.copy()
    piv = []
    r = 0
    for c in range(cols):
        p = next((i for i in range(r, rows) if A[i, c]), None)
        if p is None:
            continue
        A[[r, p]] = A[[p, r]]
        for i in range(rows):
            if i != r and A[i, c]:
                A[i] ^= A[r]
        piv.append(c)
        r += 1
    free = [c for c in range(cols) if c not in piv]
    basis = []
    for f in free:
        v = np.zeros(cols, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if A[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis)


def find_logical_z(M):
    """z with M[0:4] z = 0, M[4] z = 1 (brute force over weight)."""
    for w in range(1, 16):
        for sup in itertools.combinations(range(15), w):
            z = np.zeros(15, dtype=np.uint8)
            z[list(sup)] = 1
            s = (M.astype(int) @ z) % 2
            if not s[:4].any() and s[4] == 1:
                return z
    raise RuntimeError

In [ ]:
# ----------------------------------------------------------------- auto-corrected pi/8 (Fig. 17b)
def auto_corrected_pi8(P_data, psi_data, rng=None):
    """Fig. 17b on k data qubits + |m> + |0>.  P_data: dict qubit->Pauli on the data.
    Enumerates all measurement branches; for each, returns the Pauli correction needed and
    whether the post-correction data state equals exp(-i pi/8 P)|psi>.
    Measurement sequence (Litinski Fig. 17b):
      1. measure P (x) Z_m  and  Z_m (x) Y_0   (commuting, simultaneous)
      2. if P(x)Z_m = +1: measure |0>-qubit in Z ; else in X     (selective pi/4, Fig. 17a)
      3. measure |m> in X
      4. Pauli correction P^{c} with c determined by the outcomes (P_{pi/2} = P up to phase)."""
    k = int(np.log2(len(psi_data)))
    n = k + 2
    m, a = k, k + 1
    psi = np.kron(np.kron(psi_data, KET_M), KET0)
    P1 = pauli_on(n, {**P_data, m: 'Z'})
    P2 = pauli_on(n, {m: 'Z', a: 'Y'})
    Pd = pauli_on(k, P_data)
    target = np.cos(np.pi / 8) * psi_data - 1j * np.sin(np.pi / 8) * (Pd @ psi_data)
    branches = []
    for o1 in (1, -1):
        for o2 in (1, -1):
            s1, st, p1 = measure_pauli(psi, P1, forced=o1)
            if p1 < 1e-14:
                continue
            s2, st, p2 = measure_pauli(st, P2, forced=o2)
            if p2 < 1e-14:
                continue
            basis = 'Z' if o1 == 1 else 'X'
            for o3 in (1, -1):
                s3, st3, p3 = measure_pauli(st, pauli_on(n, {a: basis}), forced=o3)
                if p3 < 1e-14:
                    continue
                for o4 in (1, -1):
                    s4, st4, p4 = measure_pauli(st3, pauli_on(n, {m: 'X'}), forced=o4)
                    if p4 < 1e-14:
                        continue
                    # extract data state: ancillas are in known product eigenstates
                    anc_m = np.array([1, o4], dtype=complex) / np.sqrt(2)
                    anc_a = (np.array([1, 0], dtype=complex) if o3 == 1 else np.array([0, 1], dtype=complex)) \
                        if basis == 'Z' else np.array([1, o3], dtype=complex) / np.sqrt(2)
                    bra = np.kron(anc_m, anc_a)
                    dat = st4.reshape(2 ** k, 4) @ bra.conj()
                    dat /= np.linalg.norm(dat)
                    ok0 = equal_up_to_phase(dat, target)
                    ok1 = equal_up_to_phase(Pd @ dat, target)
                    branches.append(dict(PZ=o1, ZY=o2, anc_basis=basis, anc=o3, Xm=o4,
                                         prob=p1 * p2 * p3 * p4,
                                         correction='I' if ok0 else ('P' if ok1 else 'FAIL')))
    return branches

In [ ]:
# ----------------------------------------------------------------- tile layout (Fig. 17c)
# Grid of 3 rows x 5 columns (row 0 = top).  Each entry: label of the patch occupying the tile.
FIG17C_STEP1 = [
    [None, None, 'q1', 'q3', None],
    [None, None, None, None, 'q5'],
    [None, None, 'q2', 'q4', None]]
FIG17C_ROTATION = [
    ['zero', 'm', 'q1', 'q3', None],
    ['zero', 'm', 'anc', 'anc', 'q5'],
    [None, None, 'q2', 'q4', None]]
FIG17C_FINAL = [
    ['zero', 'm', 'q1', 'q3', None],
    ['zero', 'm', None, None, 'q5'],
    [None, None, 'q2', 'q4', None]]
# NOTE: in Step 1 the 2x2 block on the left is shaded as part of the block (reserved) and the
# |m> states for the rotations are prepared there; Litinski counts 11 tiles for the block.


def tile_schedule(n_rot=11):
    """List of (label, tau, layout) for the Fig. 17c protocol.  tau = time steps (1 = d cycles)."""
    sched = [('step1_init', 0, FIG17C_STEP1)]
    for r in range(n_rot):
        sched.append((f'rotation_{r + 1}', 1, FIG17C_ROTATION))
        sched.append((f'readout_{r + 1}', 0, FIG17C_FINAL))
    sched.append(('final_X_measurement', 0, FIG17C_FINAL))
    return sched


def tiles_used(sched):
    used = set()
    for _, _, lay in sched:
        for i, row in enumerate(lay):
            for j, lab in enumerate(row):
                if lab is not None:
                    used.add((i, j))
    return used


import numpy as np


def rotated_checks(rows, cols, origin=(0, 0)):
    """Stabilizers of a rotated planar code on data qubits (r,c), 0<=r<rows, 0<=c<cols (shifted by
    origin).  Check type from global parity (r+c)%2 of the plaquette corner -> consistent tiling.
    Weight-2 Z checks on top/bottom, weight-2 X checks on left/right, so that the logical Z
    string runs vertically, along the left/right boundaries (the boundaries used for Z(x)Z surgery)."""
    r0, c0 = origin
    checks = []
    for r in range(rows + 1):
        for c in range(cols + 1):
            qs = [(r0 + rr, c0 + cc) for rr in (r - 1, r) for cc in (c - 1, c)
                  if 0 <= rr < rows and 0 <= cc < cols]
            typ = 'X' if (r0 + r + c0 + c) % 2 == 0 else 'Z'
            if len(qs) == 4:
                checks.append((typ, tuple(qs)))
            elif len(qs) == 2:
                horizontal_edge = (r == 0 or r == rows)
                if (horizontal_edge and typ == 'Z') or ((c == 0 or c == cols) and typ == 'X'):
                    checks.append((typ, tuple(qs)))
    return checks


def patch_gadgets(checks, qindex, anc_start):
    gads = []
    a = anc_start
    for typ, qs in checks:
        idx = [qindex[q] for q in qs]
        if typ == 'X':
            gads.append(pauli_check_gadget(a, xq=idx))
        else:
            gads.append(z_check_gadget(a, idx))
        a += 1
    return gads


def patch_qubit_count(d):
    ch = rotated_checks(d, d)
    return d * d, len(ch)


def lattice_surgery_zz_check(d, gap=None, seed=0):
    """Two d x d rotated patches separated by `gap` columns (default d, i.e. one tile).
    Prepare both in |+_L>, perform one noiseless merge round on the full rectangle, and verify
    with stim that  Zbar1 Zbar2  becomes deterministic and equals the product of the merge-round
    outcomes of an F2 combination of merged Z checks.  Returns a dict of results."""
    import stim
    if gap is None:
        gap = d
    W = 2 * d + gap
    data = [(r, c) for r in range(d) for c in range(W)]
    qi = {q: i for i, q in enumerate(data)}
    p1 = rotated_checks(d, d, (0, 0))
    p2 = rotated_checks(d, d, (0, d + gap))
    merged = rotated_checks(d, W, (0, 0))
    nq = len(data)

    def vec(qs):
        v = np.zeros(nq, dtype=np.uint8)
        for q in qs:
            v[qi[q]] ^= 1
        return v

    def logical_z(checks, cols):
        # Z string along a column that commutes with the patch's X checks
        X = [vec(qs) for t, qs in checks if t == 'X']
        for c in cols:
            z = vec([(r, c) for r in range(d)])
            if all((x.astype(int) @ z) % 2 == 0 for x in X):
                return z
        raise RuntimeError('no column logical')

    z1 = logical_z(p1, range(0, d))
    z2 = logical_z(p2, range(d + gap, W))
    O = z1 ^ z2
    sim = stim.TableauSimulator()
    sim.set_num_qubits(nq)
    rng = np.random.default_rng(seed)
    # |+> on all data of both patches and on the gap (Litinski Fig. 41: ancilla region in |+>)
    for i in range(nq):
        sim.h(i)

    def meas(typ, qs):
        ps = stim.PauliString(nq)
        for q in qs:
            ps[qi[q]] = typ
        return sim.measure_observable(ps)

    pre = {}
    for t, qs in p1 + p2:
        pre[(t, qs)] = meas(t, qs)
    ps = stim.PauliString(nq)
    for i in np.nonzero(O)[0]:
        ps[int(i)] = 'Z'
    before = sim.peek_observable_expectation(ps)
    out = {}
    for t, qs in merged:
        out[(t, qs)] = meas(t, qs)
    after = sim.peek_observable_expectation(ps)
    Zm = [(k, vec(k[1])) for k in merged if k[0] == 'Z']
    A = np.array([v for _, v in Zm])
    coef = f2_solve(A, O)
    parity = 1
    for c, (k, _) in zip(coef, Zm):
        if c:
            parity *= -1 if out[k] else 1
    n_checks_merged = len(merged)
    return dict(d=d, gap=gap, data_qubits=nq, merged_checks=n_checks_merged,
                total_qubits=nq + n_checks_merged,
                exp_before=before, exp_after=after, product_of_outcomes=parity,
                consistent=(after == parity), O_in_span=coef is not None)

In [ ]:
# ----------------------------------------------------------------- Litinski (15-to-1)

In [ ]:
# ---------------------------------------------------------- Fig. 17c multi-patch lattice surgery (physical level)
def _tile_rect(r, c, d, h=1, w=1):
    return (r * d, (r + h) * d, c * d, (c + w) * d)


def fig17_layout(d):
    """Patches of the rotation step of [Lit] Fig. 17c on a grid of d x d-data-qubit tiles
    (3 tile rows x 5 tile columns, row 0 = top).  Each entry: (rect, rule) with rect =
    (r0, r1, c0, c1) data-qubit ranges and rule(side, r, c) -> 'X'|'Z' = type of the weight-2
    boundary checks on that side.  For every data patch the boundary facing the ancilla strip is of
    'X-check' type, so that its logical Z string runs along it ([Lit] 'Z edge')."""
    tb = lambda side, r, c: 'X' if side in ('top', 'bottom') else 'Z'       # q1..q4 (merge via top/bottom)
    lr = lambda side, r, c: 'X' if side in ('left', 'right') else 'Z'       # q5 (merge via its left edge)
    mr = _tile_rect(0, 1, d, h=2)

    def m_rule(side, r, c, r0=mr[0]):
        # |m> occupies two tiles; its corners are moved ([Lit] Fig. 2b) so that only the lower half of
        # its right edge (facing the ancilla strip) is a Z edge: segments Z(top+upper right), X(lower
        # right), Z(bottom), X(left).
        if side == 'right':
            return 'X' if r >= r0 + d else 'Z'
        return 'X' if side == 'left' else 'Z'
    return {'q1': (_tile_rect(0, 2, d), tb), 'q3': (_tile_rect(0, 3, d), tb),
            'q2': (_tile_rect(2, 2, d), tb), 'q4': (_tile_rect(2, 3, d), tb),
            'q5': (_tile_rect(1, 4, d), lr), 'm': (mr, m_rule)}


def _owner_map(patches, anc_rect=None):
    own = {}
    for name, ((r0, r1, c0, c1), _) in patches.items():
        for r in range(r0, r1):
            for c in range(c0, c1):
                own[(r, c)] = name
    if anc_rect:
        r0, r1, c0, c1 = anc_rect
        for r in range(r0, r1):
            for c in range(c0, c1):
                own[(r, c)] = 'anc'
    return own


def _side_of(rect, r, c):
    r0, r1, c0, c1 = rect
    if r == r0: return 'top'
    if r == r1: return 'bottom'
    if c == c0: return 'left'
    if c == c1: return 'right'
    return None


def patch_checks(name, rect, rule):
    """Checks (type, qubits) of one standalone rotated patch with the given boundary rule."""
    r0, r1, c0, c1 = rect
    out = []
    for r in range(r0, r1 + 1):
        for c in range(c0, c1 + 1):
            typ = 'X' if (r + c) % 2 == 0 else 'Z'
            qs = [(rr, cc) for rr in (r - 1, r) for cc in (c - 1, c) if r0 <= rr < r1 and c0 <= cc < c1]
            if len(qs) == 4:
                out.append((typ, tuple(qs)))
            elif len(qs) == 2 and typ == rule(_side_of(rect, r, c), r, c):
                out.append((typ, tuple(qs)))
    return out


def surgery_new_checks(d, merged, anc_free='Z', w3=('Z',), seam_end=('Z',), corner1=('Z',)):
    """Checks newly measured when the ancilla strip (tiles (1,2),(1,3), all qubits prepared in |+>)
    is merged with the patches in `merged` ([Lit] App. A, Fig. 41): every plaquette that touches an
    ancilla qubit, restricted to the merged region.  Truncated plaquettes: weight 3 kept if its type is
    in w3; weight 2 on the free ancilla boundary kept if its type == anc_free; weight 2 straddling
    ancilla/patch kept if its type is in seam_end; an ancilla corner qubit left without a weight-2 check
    (both neighbouring edges free) is measured in Z if the plaquette type is in corner1."""
    pats = fig17_layout(d)
    anc_rect = _tile_rect(1, 2, d, w=2)
    own = _owner_map(pats, anc_rect)
    region = {q for q, o in own.items() if o == 'anc' or o in set(merged)}
    r0, r1, c0, c1 = anc_rect
    N = []
    for r in range(r0, r1 + 1):
        for c in range(c0, c1 + 1):
            typ = 'X' if (r + c) % 2 == 0 else 'Z'
            qs = [(rr, cc) for rr in (r - 1, r) for cc in (c - 1, c) if (rr, cc) in region]
            if not any(own[q] == 'anc' for q in qs):
                continue
            if len(qs) == 4:
                N.append((typ, tuple(qs)))
            elif len(qs) == 1 and typ in corner1:
                N.append((typ, tuple(qs)))           # isolated ancilla corner qubit: measured in Z
            elif len(qs) == 3 and typ in w3:
                N.append((typ, tuple(qs)))
            elif len(qs) == 2:
                if all(own[q] == 'anc' for q in qs):
                    af = anc_free(_side_of(anc_rect, r, c), r, c) if callable(anc_free) else anc_free
                    if typ == af:
                        N.append((typ, tuple(qs)))
                elif typ in seam_end:
                    N.append((typ, tuple(qs)))
    return N, pats, anc_rect, own


def _vec(qs, idx, n):
    v = np.zeros(n, dtype=np.uint8)
    for q in qs:
        v[idx[q]] ^= 1
    return v


def _mats(checks, idx, n):
    X = np.array([_vec(qs, idx, n) for t, qs in checks if t == 'X'], dtype=np.uint8).reshape(-1, n)
    Z = np.array([_vec(qs, idx, n) for t, qs in checks if t == 'Z'], dtype=np.uint8).reshape(-1, n)
    return X, Z


def _rank(M):
    return f2_rank(M) if len(M) else 0


def _span_dim_intersection(A, B):
    """dim(span A  intersect  span B)."""
    if len(A) == 0 or len(B) == 0:
        return 0
    return _rank(A) + _rank(B) - _rank(np.vstack([A, B]))


def commutant(G, Nopp):
    """Basis of {g in span(G) : g . v = 0 for all rows v of Nopp}, plus the indices of the rows of G
    that individually commute (kept as measured checks)."""
    if len(G) == 0:
        return np.zeros((0, Nopp.shape[1]), np.uint8), []
    Bm = (G.astype(int) @ Nopp.T.astype(int)) % 2 if len(Nopp) else np.zeros((len(G), 0), int)
    keep = [i for i in range(len(G)) if not Bm[i].any()]
    if Bm.shape[1] == 0:
        return G.copy(), keep
    C = f2_nullspace(Bm.T.astype(np.uint8), len(G))
    return (C.astype(int) @ G.astype(int) % 2).astype(np.uint8), keep


def css_logical_bases(SX, SZ, n):
    """Independent logical representatives (modulo stabilizers) of a CSS code."""
    def indep(base, cands):
        out = []
        cur = base if len(base) else np.zeros((0, n), np.uint8)
        r = _rank(cur)
        for v in cands:
            nxt = np.vstack([cur, v[None, :]])
            rn = _rank(nxt)
            if rn > r:
                out.append(v); cur = nxt; r = rn
        return out
    Lz = indep(SZ, list(f2_nullspace(SX, n)) if len(SX) else list(np.eye(n, dtype=np.uint8)))
    Lx = indep(SX, list(f2_nullspace(SZ, n)) if len(SZ) else list(np.eye(n, dtype=np.uint8)))
    return Lx, Lz


def verify_multipatch(d, merged, distance=False, **opts):
    """Exact (F2) verification that measuring the new checks N implements the logical measurement of
    O = prod_{p in merged} Zbar_p on the Fig. 17c layout and nothing else.
      V1 N mutually commute;  V2 every standalone patch has k = 1 (distance d if distance=True);
      V3 O in span(N_Z) + span(G0_Z) and exactly one independent Z-logical is measured;
      V4 no X-logical is measured;
      V5 every logical commuting with O (all Zbar_p, Xbar_p for p not merged, Xbar_p Xbar_q for merged
         pairs) has a representative (mod G0) commuting with N;
      V6 (distance=True) merged code G1 = <N, commutant of N in G0> has k = 5 and distance >= d.
    Also returns the physically measured generating set during the merged rounds."""
    N, pats, anc_rect, own = surgery_new_checks(d, merged, **opts)
    qs = sorted(own); idx = {q: i for i, q in enumerate(qs)}; n = len(qs)
    G0 = []
    for name, (rect, rule) in pats.items():
        G0 += patch_checks(name, rect, rule)
    G0X, G0Z = _mats(G0, idx, n)
    anc_q = [idx[q] for q in qs if own[q] == 'anc']
    G0Xa = np.vstack([G0X, np.eye(n, dtype=np.uint8)[anc_q]])          # ancilla |+> : single-qubit X
    NX, NZ = _mats(N, idx, n)
    res = {'n_data': n, 'n_new_checks': len(N)}
    res['V1_new_checks_commute'] = not ((NX.astype(int) @ NZ.T.astype(int)) % 2).any()
    # standalone logicals
    Zb, Xb, ok2, dists = {}, {}, True, {}
    for name, (rect, rule) in pats.items():
        ch = patch_checks(name, rect, rule)
        sub = sorted({q for _, c in ch for q in c} | {(r, c) for r in range(rect[0], rect[1]) for c in range(rect[2], rect[3])})
        sidx = {q: i for i, q in enumerate(sub)}
        SX, SZ = _mats(ch, sidx, len(sub))
        Lx, Lz = css_logical_bases(SX, SZ, len(sub))
        ok2 &= len(Lz) == 1 and len(Lx) == 1
        emb = lambda v: _vec([sub[i] for i in np.nonzero(v)[0]], idx, n)
        Zb[name], Xb[name] = emb(Lz[0]), emb(Lx[0])
        if distance:
            dz = min_weight_odd(SX, Lx[0], list(range(len(sub))))[0]
            dx = min_weight_odd(SZ, Lz[0], list(range(len(sub))))[0]
            dists[name] = (dx, dz)
    res['V2_patches_k1'] = ok2
    if distance:
        res['V2_patch_distances'] = dists
        res['V2_patch_distance_ok'] = all(min(v) >= d for v in dists.values())
    names = list(pats)
    O = np.zeros(n, np.uint8)
    for p in merged:
        O ^= Zb[p]
    AZ = np.vstack([G0Z, NZ])
    res['V3_O_determined'] = _rank(np.vstack([AZ, O[None, :]])) == _rank(AZ)
    ZL = np.array([Zb[p] for p in names])
    res['V3_num_Z_logicals_measured'] = _span_dim_intersection(AZ, np.vstack([G0Z, ZL])) - _rank(G0Z)
    AX = np.vstack([G0Xa, NX])
    XL = np.array([Xb[p] for p in names])
    res['V4_num_X_logicals_measured'] = _span_dim_intersection(AX, np.vstack([G0Xa, XL])) - _rank(G0Xa)

    def has_rep(L, G, Nopp):
        if len(Nopp) == 0:
            return True
        lhs = (Nopp.astype(int) @ L.astype(int)) % 2
        if not lhs.any():
            return True
        Bm = (Nopp.astype(int) @ G.T.astype(int)) % 2
        return f2_solve(Bm.T.astype(np.uint8), lhs.astype(np.uint8)) is not None
    ok5 = all(has_rep(Zb[p], G0Z, NX) for p in names)
    ml = sorted(merged)
    xs = [Xb[p] for p in names if p not in merged] + [Xb[a] ^ Xb[b] for a, b in zip(ml, ml[1:])]
    ok5 &= all(has_rep(v, G0Xa, NZ) for v in xs)
    res['V5_commuting_logicals_preserved'] = ok5
    # merged-phase measured generating set: N + old checks that commute with N + products for the rest
    CZ_, keepZ = commutant(G0Z, NX)
    CX_, keepX = commutant(G0X, NZ)
    G1X = np.vstack([NX, CX_]); G1Z = np.vstack([NZ, CZ_])
    extraZ = _rank(np.vstack([NZ, G0Z[keepZ]] if keepZ else [NZ, np.zeros((0, n), np.uint8)]))
    res['n_old_checks_kept'] = len(keepZ) + len(keepX)
    res['n_old_checks_combined'] = (_rank(G1Z) - _rank(np.vstack([NZ] + ([G0Z[keepZ]] if keepZ else []))) +
                                    _rank(G1X) - _rank(np.vstack([NX] + ([G0X[keepX]] if keepX else []))))
    k1 = n - _rank(G1X) - _rank(G1Z)
    res['merged_k'] = k1
    res['valid'] = (res['V1_new_checks_commute'] and ok2 and res['V3_O_determined'] and
                    res['V3_num_Z_logicals_measured'] == 1 and res['V4_num_X_logicals_measured'] == 0 and ok5
                    and k1 == len(names) - 1)
    if distance and res['valid']:
        Lx, Lz = css_logical_bases(G1X, G1Z, n)
        dz = min(min_weight_odd(G1X, lx, list(range(n)))[0] for lx in Lx)
        dx = min(min_weight_odd(G1Z, lz, list(range(n)))[0] for lz in Lz)
        res['V6_merged_distance'] = (dx, dz)
        res['valid'] &= min(dx, dz) >= d
    res['_data'] = dict(N=N, G0=G0, own=own, idx=idx, pats=pats, anc_rect=anc_rect, G1X=G1X, G1Z=G1Z,
                        keepX=keepX, keepZ=keepZ, CX=CX_, CZ=CZ_, G0X=G0X, G0Z=G0Z, NX=NX, NZ=NZ, O=O)
    return res


def rotation_measured_checks(d, merged, res=None):
    """Explicit list of checks measured in every round of the merged phase of one rotation:
    the new checks N, the old patch checks that commute with N, and (if needed) products of old checks
    completing the commutant (local, weight <= 8).  Returns (checks, res)."""
    res = res or verify_multipatch(d, merged)
    D = res['_data']
    qs = sorted(D['own']); n = len(qs)
    out = list(D['N'])
    G0 = D['G0']
    G0Xc = [c for c in G0 if c[0] == 'X']; G0Zc = [c for c in G0 if c[0] == 'Z']
    out += [G0Xc[i] for i in D['keepX']] + [G0Zc[i] for i in D['keepZ']]
    for typ, C, base in [('X', D['CX'], np.vstack([D['NX']] + ([D['G0X'][D['keepX']]] if D['keepX'] else []))),
                         ('Z', D['CZ'], np.vstack([D['NZ']] + ([D['G0Z'][D['keepZ']]] if D['keepZ'] else [])))]:
        cur = base
        for v in sorted(C, key=lambda v: int(v.sum())):
            nxt = np.vstack([cur, v[None, :]])
            if _rank(nxt) > _rank(cur):
                supp = tuple(qs[i] for i in np.nonzero(v)[0])
                assert len(supp) <= 8, 'non-local combined generator'
                out.append((typ, supp)); cur = nxt
    return out, res


def stim_multipatch_check(d, merged, seed=0):
    """Noiseless stim run: patches in |+_L> (data |+>, one round of their checks), ancilla strip in |+>,
    then one round of the merged-phase checks.  Checks that O = prod Zbar is random before and
    deterministic after, with value equal to the product of the merged-round outcomes of an F2
    combination of measured Z checks (times recorded pre-merge Z-check outcomes)."""
    import stim
    checks, res = rotation_measured_checks(d, merged)
    D = res['_data']
    qs = sorted(D['own']); idx = {q: i for i, q in enumerate(qs)}; n = len(qs)
    sim = stim.TableauSimulator(seed=seed)
    sim.set_num_qubits(n)
    for i in range(n):
        sim.h(i)
    def meas(t, cq):
        ps = stim.PauliString(n)
        for q in cq: ps[idx[q]] = t
        return sim.measure_observable(ps)
    pre = [(t, cq, meas(t, cq)) for t, cq in D['G0']]
    Ops = stim.PauliString(n)
    for i in np.nonzero(D['O'])[0]: Ops[int(i)] = 'Z'
    before = sim.peek_observable_expectation(Ops)
    post = [(t, cq, meas(t, cq)) for t, cq in checks]
    after = sim.peek_observable_expectation(Ops)
    rows = [(_vec(cq, idx, n), o) for t, cq, o in post + pre if t == 'Z']
    A = np.array([r[0] for r in rows]); coef = f2_solve(A, D['O'])
    pred = 1
    for c_, (_, o) in zip(coef, rows):
        if c_: pred *= -1 if o else 1
    return dict(before=before, after=after, predicted=pred, consistent=(after == pred) and before == 0)


def rotation_measured_checks_fast(d, merged):
    """Same measured set as rotation_measured_checks, computed without the global verification:
    old checks that commute with all new checks are kept; among the (few, seam-adjacent) old checks
    that anticommute, the commuting combinations are found by a small nullspace computation.
    (Equality with the fully verified construction is checked in the validation notebook.)"""
    N, pats, anc_rect, own = surgery_new_checks(d, merged)
    G0 = []
    for name, (rect, rule) in pats.items():
        G0 += patch_checks(name, rect, rule)
    out = list(N)
    for typ, opp in [('X', 'Z'), ('Z', 'X')]:
        olds = [c for c in G0 if c[0] == typ]
        news = [set(c[1]) for c in N if c[0] == opp]
        anti = []
        for c in olds:
            s = set(c[1])
            if any(len(s & nn) % 2 for nn in news):
                anti.append(c)
            else:
                out.append(c)
        if anti:
            qset = sorted({q for c in anti for q in c[1]} | {q for nn in news for q in nn})
            li = {q: i for i, q in enumerate(qset)}
            Aold = np.array([_vec(c[1], li, len(qset)) for c in anti])
            Nn = np.array([_vec(nn, li, len(qset)) for nn in news if nn & set(li)])
            C, _ = commutant(Aold, Nn)
            cur = np.zeros((0, len(qset)), np.uint8)
            for v in sorted(C, key=lambda v: int(v.sum())):
                nxt = np.vstack([cur, v[None, :]])
                if _rank(nxt) > _rank(cur):
                    out.append((typ, tuple(qset[i] for i in np.nonzero(v)[0]))); cur = nxt
    return out, pats, own


class MSDCostData:
    """d-dependent counts of the Litinski 15-to-1 block (Fig. 17c).
    * tile model: Litinski's accounting (computed in msd_stages).
    * physical model: explicit construction of every rotation's merged phase (surgery_new_checks /
      rotation_measured_checks_fast): data qubits of q1..q5, |m> (2 tiles, moved corners), the ancilla
      strip (2 tiles) and the |0> patch (2 tiles), one measurement qubit per measured check, CNOTs per
      round and round time from the shared scheduler.  NOT constructed: the twist-based Z_m (x) Y_0
      surgery between |m> and |0> (its extra seam checks are not counted)."""

    def __init__(self, d, dur=None, explicit=True):
        dur = dur or DUR_UNIT
        self.d = d
        self.dur = dur
        ch = rotated_checks(d, d)
        qi = {(r, c): r * d + c for r in range(d) for c in range(d)}
        g = patch_gadgets(ch, qi, d * d)
        ub = round_schedule_classes(g, dur)
        lb = round_lower_bound(g, dur)
        self.round = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'])       # one isolated patch
        self.sq_patch = d * d + len(ch)                                       # 2d^2 - 1
        ch2 = rotated_checks(d, 2 * d)
        self.tall_patch = 2 * d * d + len(ch2)                                # 4d^2 - 1
        self.cnot_sq = sum(len(qs) for _, qs in ch)
        self.cnot_tall = sum(len(qs) for _, qs in ch2)
        ch3 = rotated_checks(d, 3 * d)
        self.gap_tile = 3 * d * d + len(ch3) - 2 * self.sq_patch               # 2-patch Z(x)Z merge increment
        if not explicit:
            return
        pats = fig17_layout(d)
        idle_checks = []
        for name, (rect, rule) in pats.items():
            idle_checks += patch_checks(name, rect, rule)
        zero_rect = (0, 2 * d, -d, 0)                                         # |0> patch: tiles (0,0),(1,0)
        zero_checks = patch_checks('zero', zero_rect, lambda side, r, c: 'X' if side in ('left', 'right') else 'Z')
        def cost(checks, n_data):
            coords = sorted({q for _, qq in checks for q in qq})
            qi_ = {q: i for i, q in enumerate(coords)}
            gads = []
            for j, (t, qq) in enumerate(checks):
                idx_ = [qi_[q] for q in qq]
                gads.append(pauli_check_gadget(10 ** 6 + j, xq=idx_) if t == 'X' else z_check_gadget(10 ** 6 + j, idx_))
            return dict(data=n_data, checks=len(checks), qubits=n_data + len(checks),
                        cnot=sum(len(qq) for _, qq in checks),
                        time_lb=round_lower_bound(gads, dur)['time_lb'],
                        time_ub=round_schedule_classes(gads, dur)['time_ub'])
        n_patch_data = 7 * d * d                     # q1..q5 + |m> (2 tiles)
        self.idle = cost(idle_checks + zero_checks, n_patch_data + 2 * d * d)
        self.rot = []
        for r_ in fig15_rotations():
            P = tuple(sorted({'q%d' % (i + 1) for i in r_})) + ('m',)
            chk, _, _ = rotation_measured_checks_fast(d, P)
            self.rot.append(dict(P=P, **cost(chk + zero_checks, n_patch_data + 2 * d * d + 2 * d * d)))


def msd_stages(md, model, params=DEFAULT_PARAMS, comparison='A', level='physical', data_only=False):
    """level='tile': Litinski's accounting (tiles x d^2 data qubits, or x 2d^2 incl. measurement
    qubits; time = rounds x isolated-patch round time).  level='physical': explicit per-rotation
    construction (see MSDCostData)."""
    d = md.d
    tau = rounds_for(model, params['tau_rounds'], d)
    rows = []
    if level == 'tile':
        per_tile = d * d if data_only else 2 * d * d
        rot = [dict(qubits=11 * per_tile, cnot=None, time_lb=md.round['time_lb'], time_ub=md.round['time_ub'])] * 11
        idle = dict(qubits=9 * per_tile, time_lb=md.round['time_lb'], time_ub=md.round['time_ub'])
        src_rot = SOURCE_LIT
    else:
        key = 'data' if data_only else 'qubits'
        rot = [dict(qubits=r[key], cnot=r['cnot'], time_lb=r['time_lb'], time_ub=r['time_ub']) for r in md.rot]
        idle = dict(qubits=md.idle[key], time_lb=md.idle['time_lb'], time_ub=md.idle['time_ub'])
        src_rot = SOURCE_COUNT
    meas_layer = md.dur['MEAS_X']

    def add(stage, qubits, rounds, rt, extra_time, gates, source, note=''):
        rows.append(dict(protocol='Litinski_15to1', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    inj = 0
    if comparison == 'B':
        inj = params['t_inj_FT'] if model == 'FT_ref' else params['t_inj_NL']
        add('L0 inject 4 |m> (qubits 1-4, parallel)', idle['qubits'], inj, idle, 0, Counter({'T': 4}), SOURCE_LIT,
            "Litinski App. A / Li: 3+3 cycles, d-independent; physical T gate per |m>")
    add('L1 init qubit 5 |+> (step 1)', idle['qubits'], 0, idle, 0, Counter(), SOURCE_LIT, '0 tau (Litinski)')
    for k in range(11):
        if comparison == 'B':
            add(f'L2.{k + 1}a inject |m> for rotation {k + 1}', idle['qubits'], inj, idle, 0, Counter({'T': 1}),
                SOURCE_LIT, 'serial in the 11-tile layout (Litinski "Caveat")')
        g = Counter({'CNOT': rot[k]['cnot'] * tau}) if rot[k]['cnot'] else Counter()
        add(f'L2.{k + 1} auto-corrected pi/8 rotation {k + 1}', rot[k]['qubits'], tau, rot[k], 0, g, src_rot,
            '1 tau = d cycles: P(x)Z_m surgery (explicit) and Z_m(x)Y_0 (twist not constructed)')
        add(f'L2.{k + 1}b readout |0>,|m> (single-qubit meas.)', idle['qubits'], 0, idle, meas_layer, Counter(),
            SOURCE_LIT, '0 tau; classical processing time not counted')
    add('L3 X measurement of qubits 1-4', idle['qubits'], 0, idle, meas_layer, Counter(), SOURCE_LIT, '0 tau')
    add('L4 logical X frame: T^dag|+> -> T|+>', 0, 0, idle, 0, Counter(), SOURCE_COUNT, 'Pauli frame')
    return rows

In [ ]:
# ----------------------------------------------------------------- full resource tables (used by main + comparison)
def msd_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Litinski 15-to-1 block for every d in D_LIST, level
    ('tile' | 'physical'), counting variant, time model and comparison A/B.  Litinski uses only
    Clifford + T, so the 'primitive' and 'clifford_t' gate-set rows are identical.
    Returns (stage_df, sum_df, per_d_df)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, per_d = [], [], []
    for d in D_LIST:
        t0 = time.time()
        md_ = MSDCostData(d, dur)
        for level in ['tile', 'physical']:
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = msd_stages(md_, model, params, comp, level=level, data_only=data_only)
                        for gs in ['primitive', 'clifford_t']:
                            records += [dict(r, level=level, gateset=gs, data_only=data_only) for r in st]
                            s = summarize(st)
                            s.update(protocol='Litinski_15to1', d=d, model=model, comparison=comp, level=level,
                                     gateset=gs, data_only=data_only)
                            summaries.append(s)
        per_d.append(dict(d=d, rotation_qubits_min=min(r['qubits'] for r in md_.rot),
                          rotation_qubits_max=max(r['qubits'] for r in md_.rot), idle_qubits=md_.idle['qubits'],
                          rotation_round_lb=max(r['time_lb'] for r in md_.rot),
                          rotation_round_ub=max(r['time_ub'] for r in md_.rot),
                          CNOT_per_round_max=max(r['cnot'] for r in md_.rot), tile_model_rotation_qubits=22 * d * d,
                          isolated_patch_round_lb=md_.round['time_lb'], isolated_patch_round_ub=md_.round['time_ub']))
        if verbose:
            print(f'Litinski d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), pd.DataFrame(per_d)

### 雑音付き D4 code switching ライブラリ（回路・雑音モデル・Takada らの Pauli proxy 法による非 Clifford シミュレーション・復号器）

In [ ]:
"""noisy_d4_lib: circuit-level noisy simulation of the Kobayashi-Zhu-Hsin D4 code switching protocol.

Non-Clifford simulation method: Clifford-stabilizer simulation with Pauli proxies
(Y. Takada, S. D. Bartlett, D. J. Williamson, arXiv:2609.16929, Algorithm 2 (Sec. VII D) and Sec. VIII).
For every Monte Carlo shot the noisy non-Clifford circuit is mapped to a Clifford circuit (the
"Pauli-proxy circuit") with the same measurement statistics:
  * every diagonal third-level gate D (CCZ of the D4 gadgets, CS / CS^dag / T^dag of U) is removed;
  * the X component e of the sampled Pauli errors and of applied X corrections, propagated forward
    through the Clifford gates of the true circuit, is tracked ("continuing errors"); where e meets
    D the diagonal Clifford gate C_e = D^dag X^e D X^e (a product of S, Z, CZ) is inserted
    (= the controlled Pauli dressing Lambda(D_U(E)) of Takada Fig. 14 / Eq. (165));
  * Z components commute with every diagonal gate and need no propagation.
All Clifford gates, sampled Pauli errors and inserted dressings are executed by stim's TableauSimulator.

Requires (as globals, embedded before this code): numpy as np, stim, common_lib, d4_lib.
"""
import itertools
import zlib
import numpy as np

In [ ]:
# ------------------------------------------------------------------ gate algebra
DIAG_THETA = {                     # phase polynomials (units of pi/4) of the diagonal 3rd-level gates
    'CCZ': lambda x: 4 * x[0] * x[1] * x[2],
    'NCCZ': lambda x: 4 * x[0] * x[1] * x[2],      # virtual noiseless CCZ (deferred switch-out correction)
    'CS': lambda x: 2 * x[0] * x[1],
    'CSDG': lambda x: 6 * x[0] * x[1],
    'T': lambda x: x[0],
    'TDG': lambda x: 7 * x[0],
}
NONCLIFF = set(DIAG_THETA)
_DRESS = {}


def diag_clifford_gates(delta, k):
    """delta: dict bits -> phase exponent mod 8 (even).  Returns gates ('S'|'Z'|'SDG', i), ('CZ', i, j)
    (local indices) equal to diag(omega^delta) up to a global phase; asserts the result is exact."""
    c0 = delta[(0,) * k]
    a, gates = [], []
    for i in range(k):
        e = tuple(int(j == i) for j in range(k))
        ai = (delta[e] - c0) % 8
        assert ai % 2 == 0, 'dressing is not Clifford'
        a.append(ai // 2)
        if ai // 2:
            gates.append(({1: 'S', 2: 'Z', 3: 'SDG'}[ai // 2], i))
    b = {}
    for i, j in itertools.combinations(range(k), 2):
        e = tuple(int(m in (i, j)) for m in range(k))
        v = (delta[e] - c0 - 2 * a[i] - 2 * a[j]) % 8
        assert v in (0, 4), 'dressing has a non-Clifford quadratic term'
        b[(i, j)] = v // 4
        if v:
            gates.append(('CZ', i, j))
    for x in itertools.product((0, 1), repeat=k):
        pred = c0 + sum(2 * a[i] * x[i] for i in range(k)) + sum(4 * b[ij] * x[ij[0]] * x[ij[1]] for ij in b)
        assert (pred - delta[x]) % 8 == 0, 'dressing has a cubic term'
    return gates


def dressing(name, e):
    """C_e = D^dag X^e D X^e for diagonal gate `name` and X pattern e (tuple of bits), local indices."""
    key = (name, e)
    if key not in _DRESS:
        th = DIAG_THETA[name]
        k = len(e)
        delta = {x: (th(tuple(xi ^ ei for xi, ei in zip(x, e))) - th(x)) % 8
                 for x in itertools.product((0, 1), repeat=k)}
        _DRESS[key] = diag_clifford_gates(delta, k)
    return _DRESS[key]

In [ ]:
# ------------------------------------------------------------------ noise model
class NoiseModel:
    """Standard circuit-level Pauli noise (Takada et al. App. A 2) plus optional idling noise:
    after every k-qubit unitary a k-qubit depolarizing channel (probability p; p_ccz for CCZ, p_nc for
    the gates of U), preparation followed by a flip (p_reset), measurement preceded by a flip (p_meas),
    every idle qubit of a time step depolarized with p_idle."""

    def __init__(self, p, p_meas=None, p_reset=None, p_idle=None, p_ccz=None, p_nc=None):
        self.p = p
        self.p_meas = p if p_meas is None else p_meas
        self.p_reset = p if p_reset is None else p_reset
        self.p_idle = p if p_idle is None else p_idle
        self.p_ccz = p if p_ccz is None else p_ccz
        self.p_nc = p if p_nc is None else p_nc

    def locations(self, ops, idle):
        """Noise locations of one time step: list of (when, kind, qubits, prob), when in {'pre','post'}."""
        loc = []
        for op in ops:
            nm, qs = op[0], op[1:]
            if nm in ('NCCZ', 'MZ0'):
                continue
            if nm == 'FLIPX':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MZ':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MX':
                loc.append(('pre', 'Z', qs, self.p_meas))
            elif nm == 'RZ':
                loc.append(('post', 'X', qs, self.p_reset))
            elif nm == 'RX':
                loc.append(('post', 'Z', qs, self.p_reset))
            elif nm == 'CCZ':
                loc.append(('post', 'D3', qs, self.p_ccz))
            elif nm in ('CS', 'CSDG', 'T', 'TDG'):
                loc.append(('post', 'D%d' % len(qs), qs, self.p_nc))
            else:
                loc.append(('post', 'D%d' % len(qs), qs, self.p))
        for q in idle:
            loc.append(('post', 'D1', (q,), self.p_idle))
        return [l for l in loc if l[3] > 0]


PAULI_CHOICES = {k: [''.join(p) for p in itertools.product('IXYZ', repeat=k) if set(p) != {'I'}]
                 for k in (1, 2, 3)}

In [ ]:
# ------------------------------------------------------------------ parallel FSC schedule
_FSC_SCHED = {}


def _fsc_checks(P):
    """[(type 'X'|'Z', ancilla, label, [data qubits])] of one FSC round (frozen qubits removed)."""
    lat, cd = P.lat, P.fsc
    out = []
    for lab, op in cd.xgens:
        _, v, cols = lab
        qs = []
        for ch in cols:
            cc = CNAME.index(ch)
            qs += sorted({lat.q(e, cc) for e in lat.incident(v)} - P.frozen)
        out.append(('X', P.anc[lab], lab, qs))
    for lab, (l2, rows) in zip(cd.zlabels, lat.z_checks(cd.colours)):
        if P._is_frozen_check(lab):
            continue
        out.append(('Z', P.anc[lab], lab, sorted(set(rows) - P.frozen)))
    return out


def _fsc_anneal(P, checks, T, seed, iters):
    """Assign a time slot 0..T-1 to every (check, data qubit) CNOT such that (i) no ancilla and no data qubit
    is used twice in a slot and (ii) for every X check A and Z check B the number of shared qubits on
    which B's CNOT precedes A's is even (so all checks can be measured at the same time).  Start: the
    direction-based order of the bulk (X stars E,N,S,W = 0,1,2,3; Z plaquettes E,S,N,W = 0,1,2,3, one of
    the 16 compatible bulk orders); simulated annealing fixes the boundary (weight-6 X^r X^b and
    weight-2 Z^r Z^b checks of L_rb)."""
    lat = P.lat
    XO, ZO = {'E': 0, 'W': 3, 'N': 1, 'S': 2}, {'S': 1, 'N': 2, 'W': 3, 'E': 0}
    rng = np.random.default_rng(seed)
    var, slot = [], []
    for ci, (typ, anc, lab, qs) in enumerate(checks):
        for q in qs:
            k, x, y = lat.edges[q // 3]
            if typ == 'X':
                d = ('E' if x == lab[1][0] else 'W') if k == 'h' else ('N' if y == lab[1][1] else 'S')
                s0 = XO[d]
            elif lab[0] == 'plaq':
                d = ('S' if y == lab[1][1] else 'N') if k == 'h' else ('W' if x == lab[1][0] else 'E')
                s0 = ZO[d]
            else:
                s0 = 0
            var.append((ci, q)); slot.append(s0)
    slot = np.array(slot)
    by_check, by_qubit = {}, {}
    for i, (ci, q) in enumerate(var):
        by_check.setdefault(ci, []).append(i); by_qubit.setdefault(q, []).append(i)
    pairs = {}
    for q, idx in by_qubit.items():
        xs = [i for i in idx if checks[var[i][0]][0] == 'X']
        zs = [i for i in idx if checks[var[i][0]][0] == 'Z']
        for a_ in xs:
            for b_ in zs:
                pairs.setdefault((var[a_][0], var[b_][0]), []).append((a_, b_))
    pairs = list(pairs.values())
    pv = {}
    for k, pl in enumerate(pairs):
        for a_, b_ in pl:
            pv.setdefault(a_, []).append(k); pv.setdefault(b_, []).append(k)
    bad = lambda k: sum(slot[b_] < slot[a_] for a_, b_ in pairs[k]) % 2
    conf = lambda i: (sum(slot[j] == slot[i] for j in by_check[var[i][0]] if j != i)
                      + sum(slot[j] == slot[i] for j in by_qubit[var[i][1]] if j != i))
    local = lambda i: conf(i) + sum(bad(k) for k in pv.get(i, []))
    total = lambda: sum(conf(i) for i in range(len(var))) // 2 + sum(bad(k) for k in range(len(pairs)))
    cost, temp = total(), 1.0
    for it in range(iters):
        if cost == 0:
            break
        i = int(rng.integers(len(var)))
        if local(i) == 0 and rng.random() < 0.9:
            continue
        old, before = slot[i], local(i)
        slot[i] = rng.integers(T)
        dlt = local(i) - before
        if dlt <= 0 or rng.random() < np.exp(-dlt / temp):
            cost += dlt
        else:
            slot[i] = old
        if it % 20000 == 0:
            cost, temp = total(), max(0.05, temp * 0.85)
    return var, slot, total()


def fsc_memory_circuit(P, sched, basis, rounds, p):
    """stim memory experiment of the FSC with the parallel schedule (validation): data in |+> (X) or |0>
    (Z), `rounds` rounds, final data measurement; observable = logical X (t_log) or a logical Z.  Noise:
    DEPOLARIZE2(p) after every CNOT, flips p on ancilla reset / before measurement / final data."""
    checks, T, layers = sched['checks'], sched['T'], sched['layers']
    data = sorted({q for typ, anc, lab, qs in checks for q in qs})
    idx = {q: i for i, q in enumerate(data)}
    t_log = [int(q) for q in np.nonzero(P.fsc.t_log)[0]]
    Hx = np.array([[1 if q in qs else 0 for q in data] for typ, anc, lab, qs in checks if typ == 'X'], dtype=np.uint8)
    tv = np.array([1 if q in t_log else 0 for q in data], dtype=np.uint8)
    zl = next([data[i] for i in np.nonzero(v)[0]] for v in f2_nullspace(Hx) if int(v.astype(int) @ tv.astype(int)) % 2)
    xa = [anc for typ, anc, lab, qs in checks if typ == 'X']
    za = [anc for typ, anc, lab, qs in checks if typ == 'Z']
    nX, nZ = len(xa), len(za)
    cir = stim.Circuit()
    cir.append('RX' if basis == 'X' else 'R', data)
    for r in range(rounds):
        cir.append('RX', xa); cir.append('R', za)
        if p:
            cir.append('Z_ERROR', xa, p); cir.append('X_ERROR', za, p)
        cir.append('TICK')
        for t in range(T):
            tg = []
            for typ, anc, q in layers[t]:
                tg += [anc, q] if typ == 'X' else [q, anc]
            if tg:
                cir.append('CX', tg)
                if p:
                    cir.append('DEPOLARIZE2', tg, p)
            cir.append('TICK')
        if p:
            cir.append('Z_ERROR', xa, p); cir.append('X_ERROR', za, p)
        cir.append('MX', xa); cir.append('M', za)
        nm = nX + nZ
        for k in range(nX):
            if r:
                cir.append('DETECTOR', [stim.target_rec(-nm + k), stim.target_rec(-2 * nm + k)])
            elif basis == 'X':
                cir.append('DETECTOR', [stim.target_rec(-nm + k)])
        for k in range(nZ):
            if r:
                cir.append('DETECTOR', [stim.target_rec(-nZ + k), stim.target_rec(-nZ - nm + k)])
            elif basis == 'Z':
                cir.append('DETECTOR', [stim.target_rec(-nZ + k)])
    if p:
        cir.append('Z_ERROR' if basis == 'X' else 'X_ERROR', data, p)
    cir.append('MX' if basis == 'X' else 'M', data)
    nd = len(data)
    off = 0 if basis == 'X' else nX
    for k, (anc, qs) in enumerate([(anc, qs) for typ, anc, lab, qs in checks if typ == basis]):
        cir.append('DETECTOR', [stim.target_rec(-nd + idx[q]) for q in qs] + [stim.target_rec(-nd - (nX + nZ) + off + k)])
    cir.append('OBSERVABLE_INCLUDE', [stim.target_rec(-nd + idx[q]) for q in (t_log if basis == 'X' else zl)], 0)
    return cir


def fsc_parallel_schedule(P, T=6, max_seeds=30, iters=3000000):
    """Measure all FSC checks at the same time in T CNOT layers (round = T + 2 time steps; T = 6 is the
    minimum because the X^r X^b checks of L_rb have weight 6; the bulk alone would need 4).  The slot
    assignment is searched (deterministic seeds) and ACCEPTED ONLY IF the stim memory experiments in both
    bases are deterministic without noise and have circuit-level distance >= L (no harmful hook errors).
    Cached per L.  Returns dict(T, checks, layers[t] = [(type, ancilla, data qubit)], seed, distance)."""
    L = P.L
    if L in _FSC_SCHED:
        return _FSC_SCHED[L]
    checks = _fsc_checks(P)
    for seed in range(max_seeds):
        var, slot, cost = _fsc_anneal(P, checks, T, seed, iters)
        if cost:
            continue
        layers = [[] for _ in range(T)]
        for i, (ci, q) in enumerate(var):
            layers[int(slot[i])].append((checks[ci][0], checks[ci][1], q))
        sch = dict(T=T, checks=checks, layers=layers, seed=seed)
        ok, dist = True, {}
        for basis in 'XZ':
            s0 = fsc_memory_circuit(P, sch, basis, 3, 0.0).compile_detector_sampler(seed=1).sample(20, append_observables=True)
            c1 = fsc_memory_circuit(P, sch, basis, max(L, 2), 1e-3)
            dist[basis] = len(c1.detector_error_model(decompose_errors=True).shortest_graphlike_error())
            ok &= (not s0.any()) and dist[basis] >= L
        if ok:
            sch['distance'] = dist
            _FSC_SCHED[L] = sch
            return sch
    raise RuntimeError(f'no valid parallel FSC schedule found for L={L}')

In [ ]:
# ------------------------------------------------------------------ the noisy protocol circuit
class D4SwitchingCircuit:
    """Time-step level description of the noisy KZH protocol on the bounded square lattice.

    Blocks (in order):
      ('S0', r)  r = 1..R0   FSC |+_L> preparation: data |+>, rounds of all FSC checks
      ('S1', r)  r = 1..Rin  switch-in: g |0>, rounds of all D4 stabilizers ([KZH] step 2)
      ('U',)                 transversal U = W V ([KZH] eq. (8), step 3)
      ('SO',)                switch-out: Z^g readout ([KZH] step 4; deferred-correction form)
      ('S4', r)  r = 1..Rout FSC rounds after switching back ([KZH] step 4)
      ('FIN',)               one noiseless FSC round (fictitious error correction, Takada Sec. VI D)
    Qubits are re-indexed compactly: data (non-frozen r,g,b) first, then ancillas.
    sequential=True measures the gadgets of a commuting class one after another (validation only)."""

    def __init__(self, L, R0, Rin, Rout, sequential=False, n_pre_u=2, fsc_schedule='classes'):
        self.n_pre_u = n_pre_u
        P = D4Protocol(L)
        lat = P.lat
        self.P, self.lat, self.L = P, lat, L
        self.R0, self.Rin, self.Rout = R0, Rin, Rout
        data = sorted(P.data_qubits((R, G, B)))
        gadgets = {'D4': P.round_gadgets('D4'), 'FSC': P.round_gadgets('FSC')}
        ancs = sorted({g[0][1] for k in gadgets for g in gadgets[k]})
        c = {q: i for i, q in enumerate(data + ancs)}
        self.c = c
        self.nq, self.nd = len(data) + len(ancs), len(data)
        self.data_rb = sorted(c[q] for q in P.data_qubits((R, B)))
        self.data_g = sorted(c[q] for q in P.data_qubits((G,)))
        self.data_all = sorted(c[q] for q in data)
        conv = {'INIT_X': 'RX', 'INIT_Z': 'RZ', 'MEAS_X': 'MX', 'MEAS_Z': 'MZ', 'CNOT': 'CX'}
        cv = lambda g: (conv.get(g[0], g[0]),) + tuple(c[q] for q in g[1:])
        # label of every ancilla, per code
        self.lab_of = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            labs = [l for l, _ in cd.xgens] + [l for l in cd.zlabels if not P._is_frozen_check(l)]
            for l in labs:
                self.lab_of[(code, c[P.anc[l]])] = l

        def round_moments(code, which='all'):
            gl = gadgets[code]
            if which != 'all':
                gl = [g for g in gl if g[0][0] == ('INIT_Z' if which == 'Z' else 'INIT_X')]
            classes = commuting_classes(gl)
            if sequential:
                classes = [[g] for cls in classes for g in cls]
            moms = []
            for cls in classes:
                for layer in class_layers(cls):
                    moms.append([cv(g) for g in layer])
            return moms
        # D4 rounds: Pauli Z checks first, then (after the real-time flux correction) the X-type
        # Clifford stabilizers, so that fluxes detected by the Z checks are removed before they meet CCZs
        if fsc_schedule == 'parallel' and not sequential:
            # all FSC checks at once: ancilla init, 6 CNOT layers (verified order, fsc_parallel_schedule), measure
            fs = fsc_parallel_schedule(P)
            moms = [[('RX' if typ == 'X' else 'RZ', c[anc]) for typ, anc, lab, qs in fs['checks']]]
            for t in range(fs['T']):
                moms.append([('CX', c[anc], c[q]) if typ == 'X' else ('CX', c[q], c[anc])
                             for (typ, anc, q) in fs['layers'][t]])
            moms.append([('MX' if typ == 'X' else 'MZ', c[anc]) for typ, anc, lab, qs in fs['checks']])
            fsc_moms = moms
        else:
            # default: X class then Z class (14 steps).  The verified 8-step parallel schedule gives a larger
            # output error with OUR flux decoder (its phenomenological error model does not represent the
            # hook errors that the interleaving splits across rounds), so it is used only as a sensitivity
            fsc_moms = round_moments('FSC')
        self.round_moms = {'FSC': fsc_moms, 'D4z': round_moments('D4', 'Z'), 'D4x': round_moments('D4', 'X')}
        self.anc = {code: sorted({c[g[0][1]] for g in gadgets[code]}) for code in gadgets}
        self.U_moms = [[cv(g) for g in layer] for layer in class_layers([P.U_gates()])]
        # ---- switch-out data: S^g generators, Gamma (g configuration -> kappa), CZ^{r,b} pairs
        self.gens_g = [(lab, op) for lab, op in P.d4.xgens if lab[2] == 'g']
        gpos = {q: i for i, q in enumerate(self.data_g)}
        Sg = np.zeros((len(self.gens_g), len(self.data_g)), dtype=np.uint8)
        for i, (lab, op) in enumerate(self.gens_g):
            for q in op.s:
                if q in c and c[q] in gpos:
                    Sg[i, gpos[c[q]]] = 1
        self.Sg = Sg
        self.Gamma = left_inverse_T(Sg)
        self.gpairs = [[(c[i], c[j]) for (i, j) in op.quad if i in c and j in c] for lab, op in self.gens_g]
        # physical switch-out correction CZ(Gamma(a_dec)) ([KZH] step 4): the CZ pairs of the selected S^g
        # (pairs occurring an even number of times cancel) are applied in a FIXED schedule: a greedy edge
        # colouring of the union of all pairs; colour l = time step l.  The number of steps does not depend
        # on the outcome (worst case, used for the time count); empty steps are still idle steps.
        allpairs = sorted({tuple(sorted(pr)) for prs in self.gpairs for pr in prs})
        colour, used = {}, {}
        for pr in allpairs:
            k = 0
            while any(k in used.get(q, ()) for q in pr):
                k += 1
            colour[pr] = k
            for q in pr:
                used.setdefault(q, set()).add(k)
        self.so_colour = colour
        self.so_steps = 1 + (max(colour.values()) + 1 if colour else 0)    # g readout + CZ layers
        self.network = [('NCCZ', self.data_g[e], i, j)
                        for v in range(len(self.gens_g)) for e in np.nonzero(self.Gamma[v])[0]
                        for (i, j) in self.gpairs[v]]
        # Pauli-frame strings Z^r Z^b for the S^g eigenvalues (protocol step 5 of the noiseless check V7)
        self.frame_strings = []
        for lab, op in self.gens_g:
            x0, y0 = lab[1]
            qs = [lat.q(lat.eidx[('v', x0, y)], cc) for y in range(y0) for cc in (R, B)]
            self.frame_strings.append(sorted(c[q] for q in qs if q in c))
        # ---- Z checks of the FSC (rb data, frozen checks removed) -> used for the S1 X frame
        self.zchecks = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            rows = []
            for lab, (l2, qs) in zip(cd.zlabels, lat.z_checks(cd.colours)):
                if P._is_frozen_check(lab):
                    continue
                rows.append((lab, sorted(c[q] for q in qs if q in c)))
            self.zchecks[code] = rows
        self.xgens = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            self.xgens[code] = [(lab, sorted(c[q] for q in op.s if q in c),
                                 [(c[i], c[j]) for (i, j) in op.quad if i in c and j in c]) for lab, op in cd.xgens]
        self._logical()
        self.blocks = self._blocks()

    # ------------------------------------------------------------ logical Clifford stabilizer
    def _logical(self):
        """Ideal output = D_F |+_L>_FSC with F(a) = w(a) + 4 sum_v c'_v(a) f^g_v(a) (protocol phase,
        validated in V7/V7b/V8).  Its logical Clifford stabilizer is H = D_F X^t D_F^dag = X^t D_Delta,
        Delta(x) = F(x+t) - F(x) = c0 + sum 2 a_i x_i + sum 4 b_ij x_i x_j (mod 8)."""
        P, lat, c = self.P, self.lat, self.c
        F = {}

        def add(mono, coef):
            m = frozenset(mono)
            if any(q not in c for q in m):
                return                                   # frozen qubits are |0>
            m = frozenset(c[q] for q in m)
            F[m] = (F.get(m, 0) + coef) % 8
        for coef, mono in W_operator(lat).terms:
            add(mono, coef)
        # c'_v(a): path sums along a BFS tree from the L_rb vertices (linear in a)
        from collections import deque
        root = lrb_vertices(lat)
        path = {v: [] for v in root}
        dq = deque(root)
        while dq:
            u = dq.popleft()
            for e in lat.incident(u):
                p, q = lat.endpoints(e)
                w = q if p == u else p
                if w not in path:
                    path[w] = path[u] + [e]
                    dq.append(w)
        for v in P.gverts:
            lin = [lat.q(e, cc) for e in path[v] for cc in (R, B)]
            for (i, j) in lat.cz_pairs(v, G):
                for k in lin:
                    add({k, i, j}, 4)
        self.Fpoly = {m: v for m, v in F.items() if v}
        t = sorted(c[int(q)] for q in np.nonzero(P.fsc.t_log)[0])
        self.t_log = t
        tset = set(t)
        D = {}
        for m, coef in self.Fpoly.items():           # F(x+t): substitute x_i -> 1 - x_i for i in t
            fixed = [q for q in m if q not in tset]
            flip = [q for q in m if q in tset]
            for sub in itertools.product((0, 1), repeat=len(flip)):
                mono = frozenset(fixed + [q for q, s in zip(flip, sub) if s])
                sign = (-1) ** sum(sub)
                D[mono] = (D.get(mono, 0) + coef * sign) % 8
        for m, coef in self.Fpoly.items():
            D[m] = (D.get(m, 0) - coef) % 8
        D = {m: v for m, v in D.items() if v}
        self.Delta_c0 = D.pop(frozenset(), 0)
        self.Delta_a, self.Delta_b = {}, {}
        for m, v in D.items():
            if len(m) == 1:
                assert v % 2 == 0, 'logical stabilizer not Clifford (odd linear term)'
                self.Delta_a[next(iter(m))] = v // 2 % 4
            elif len(m) == 2:
                assert v % 4 == 0, 'logical stabilizer not Clifford (quadratic term)'
                self.Delta_b[tuple(sorted(m))] = v // 4 % 2
            else:
                raise AssertionError('logical stabilizer not Clifford (cubic term)')
        self.Delta_a = {q: a for q, a in self.Delta_a.items() if a}
        self.Delta_b = {ij: b for ij, b in self.Delta_b.items() if b}

    def logical_proxy(self, fx, sign=1):
        """stim.PauliString of the Pauli proxy of the dressed logical stabilizer X^e H X^e
        (e = X frame fx on the data): proxy(X^t D_Delta R_e) = X^t * prod_{a_i=2} Z_i * R_e, where
        R_e = D_Delta^dag X^e D_Delta X^e = phase * Z-string (Takada Secs. VI D, VII B)."""
        n = self.nq
        ps = stim.PauliString(n)
        for q in self.t_log:
            ps[q] = 'X'
        zs = stim.PauliString(n)
        for q, a in self.Delta_a.items():
            if a == 2:
                zs[q] = 'Z'
        R = stim.PauliString(n)
        ph = 1
        for q, a in self.Delta_a.items():
            if fx[q] and a % 2 == 1:
                ph *= 1j ** a
                R *= _single(n, q, 'Z')
            elif fx[q] and a == 2:
                ph *= -1
        for (i, j), b in self.Delta_b.items():
            if fx[i] and fx[j]:
                ph *= -1
                R *= _single(n, i, 'Z') * _single(n, j, 'Z')
            elif fx[i]:
                R *= _single(n, j, 'Z')
            elif fx[j]:
                R *= _single(n, i, 'Z')
        out = ps * zs * R
        return out * complex(ph * sign)

    # ------------------------------------------------------------ blocks
    def _blocks(self):
        B_ = []
        rb, g, alld = self.data_rb, self.data_g, self.data_all
        for r in range(1, self.R0 + 1):
            pre = [[('RX', q) for q in rb]] if r == 1 else []
            B_.append(dict(name=('S0', r), code='FSC', moms=pre + self.round_moms['FSC'], data=rb))
        for r in range(1, self.Rin + 1 + self.n_pre_u):
            pre = [[('RZ', q) for q in g]] if r == 1 else []
            B_.append(dict(name=('S1', r, 'z'), code='D4', moms=pre + self.round_moms['D4z'], data=alld))
            if r <= self.Rin:            # rounds Rin+1.. have only the Z checks (flux checks before U)
                B_.append(dict(name=('S1', r, 'x'), code='D4', moms=self.round_moms['D4x'], data=alld))
        B_.append(dict(name=('U',), code=None, moms=self.U_moms, data=alld))
        B_.append(dict(name=('SO',), code=None,
                       moms=[[('FLIPX', q) for q in g], list(self.network), [('MZ0', q) for q in g]], data=alld))
        for r in range(1, self.Rout + 1):
            B_.append(dict(name=('S4', r), code='FSC', moms=self.round_moms['FSC'], data=rb))
        B_.append(dict(name=('FIN',), code='FSC', moms=self.round_moms['FSC'], data=rb, noiseless=True))
        return B_


def _single(n, q, p):
    s = stim.PauliString(n)
    s[q] = p
    return s


def left_inverse_T(S):
    """Gamma (m x n) over F2 with Gamma @ S^T = I_m, for S (m x n) of full row rank."""
    m, n = S.shape
    A = (S.T % 2).astype(np.uint8)
    M = np.concatenate([A, np.eye(n, dtype=np.uint8)], axis=1)
    r = 0
    for col in range(m):
        nz = np.nonzero(M[r:, col])[0]
        assert len(nz), 'S^T is not of full column rank'
        p = r + nz[0]
        M[[r, p]] = M[[p, r]]
        for rr in range(n):
            if rr != r and M[rr, col]:
                M[rr] ^= M[r]
        r += 1
    Gm = M[:m, m:]
    assert not ((Gm.astype(int) @ A.astype(int)) % 2 - np.eye(m, dtype=int)).any()
    return Gm

In [ ]:
# ------------------------------------------------------------------ exact reference (validation only)
class TrueSim:
    """Exact sparse state-vector simulation of the TRUE (non-Clifford) circuit, used to validate the
    Pauli-proxy simulation.  Basis states are uint64 bit masks: data qubits 0..nd-1, ancillas get a
    slot nd+k while they are alive (reset ... measurement)."""
    OMEGA = np.exp(1j * np.pi / 4)

    def __init__(self, nd, keys, amps):
        self.nd = nd
        self.keys = np.asarray(keys, dtype=np.uint64)
        self.amps = np.asarray(amps, dtype=complex)
        self.slot = {}
        self.free = []
        self.nslots = 0

    def bit(self, q):
        if q < self.nd:
            return q
        if q not in self.slot:
            raise KeyError(q)
        return self.slot[q]

    def _alloc(self, q):
        if q < self.nd:
            return q
        if q in self.slot:
            return self.slot[q]
        b = self.free.pop() if self.free else self.nd + self.nslots
        if b == self.nd + self.nslots:
            self.nslots += 1
        self.slot[q] = b
        assert b < 64
        return b

    def _bits(self, b):
        return ((self.keys >> np.uint64(b)) & np.uint64(1)).astype(np.int64)

    def _merge(self):
        u, inv = np.unique(self.keys, return_inverse=True)
        re = np.bincount(inv, weights=self.amps.real, minlength=len(u))
        im = np.bincount(inv, weights=self.amps.imag, minlength=len(u))
        a = re + 1j * im
        keep = np.abs(a) > 1e-12
        self.keys, self.amps = u[keep], a[keep]

    def X(self, q):
        self.keys = self.keys ^ np.uint64(1 << self.bit(q))

    def Z(self, q):
        self.amps = self.amps * (1 - 2 * self._bits(self.bit(q)))

    def Y(self, q):
        self.Z(q)
        self.X(q)
        self.amps = self.amps * 1j

    def H(self, q):
        b = self.bit(q)
        m = np.uint64(1 << b)
        v = self._bits(b)
        k0 = self.keys & ~m
        a = self.amps / np.sqrt(2)
        self.keys = np.concatenate([k0, k0 | m])
        self.amps = np.concatenate([a, a * (1 - 2 * v)])
        self._merge()

    def diag(self, qs, theta):
        bits = [self._bits(self.bit(q)) for q in qs]
        ph = np.array([theta(t) for t in zip(*bits)], dtype=float) if len(self.keys) else np.zeros(0)
        self.amps = self.amps * self.OMEGA ** ph

    def CX(self, c, t):
        vc = self._bits(self.bit(c)).astype(np.uint64)
        self.keys = self.keys ^ (vc << np.uint64(self.bit(t)))

    def measure_z(self, q, forced):
        b = self.bit(q)
        v = self._bits(b)
        p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
        prob = p1 if forced else 1 - p1
        keep = v == forced
        self.keys, self.amps = self.keys[keep], self.amps[keep]
        if prob > 1e-12:
            self.amps = self.amps / np.sqrt(prob)
        return prob

    def release(self, q, outcome):
        if q >= self.nd:
            if outcome:
                self.X(q)
            self.free.append(self.slot.pop(q))

    def reset(self, q, basis, rng):
        b = self._alloc(q)
        if q < self.nd:                      # data reset: measure and flip to |0>
            v = self._bits(b)
            p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
            o = int(rng.random() < p1)
            self.measure_z(q, o)
            if o:
                self.X(q)
        if basis == 'X':
            self.H(q)

    def run_moment(self, mom, errs, outcomes, rng):
        """Execute a time step with the given sampled errors; measurements forced to `outcomes`
        (list in the order of the moment's measurements).  Returns list of probabilities."""
        for when, ps, qs in errs:
            if when == 'pre':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        probs = []
        k = 0
        for op in mom:
            nm, qs = op[0], op[1:]
            if nm in DIAG_THETA:
                self.diag(qs, DIAG_THETA[nm])
            elif nm == 'CX':
                self.CX(*qs)
            elif nm == 'CZ':
                self.diag(qs, lambda x: 4 * x[0] * x[1])
            elif nm in ('S', 'SDG'):
                self.diag(qs, (lambda x: 2 * x[0]) if nm == 'S' else (lambda x: 6 * x[0]))
            elif nm in ('X', 'Z', 'Y'):
                getattr(self, nm)(qs[0])
            elif nm in ('RX', 'RZ'):
                self.reset(qs[0], nm[1], rng)
            elif nm in ('MX', 'MZ', 'MZ0'):
                q = qs[0]
                if nm == 'MX':
                    self.H(q)
                o = outcomes[k]
                k += 1
                probs.append(self.measure_z(q, o))
                if q >= self.nd:
                    self.release(q, o)
            elif nm == 'FLIPX':
                pass
            else:
                raise ValueError(nm)
        for when, ps, qs in errs:
            if when == 'post':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        return probs

    def expect_logical(self, circ):
        """<Psi| X^t D_Delta |Psi> on the data qubits (all ancillas released)."""
        assert not self.slot
        x = self.keys
        ph = np.full(len(x), circ.Delta_c0, dtype=float)
        bits = {}
        get = lambda q: bits.setdefault(q, ((x >> np.uint64(q)) & np.uint64(1)).astype(np.int64))
        for q, a in circ.Delta_a.items():
            ph += 2 * a * get(q)
        for (i, j), b in circ.Delta_b.items():
            ph += 4 * b * get(i) * get(j)
        Da = self.amps * self.OMEGA ** ph
        tm = np.uint64(sum(1 << q for q in circ.t_log))
        k2 = x ^ tm
        order = np.argsort(x)
        xs = x[order]
        pos = np.searchsorted(xs, k2)
        pos[pos >= len(xs)] = 0
        hit = xs[pos] == k2
        amp_t = np.zeros(len(x), dtype=complex)
        amp_t[hit] = self.amps[order][pos[hit]]
        return complex(np.sum(np.conj(amp_t) * Da))

In [ ]:
# ------------------------------------------------------------------ compiled protocol (fast Monte Carlo)
_KIND = {'X': 0, 'Z': 1, 'D1': 2, 'D2': 3, 'D3': 4}
_STIM1 = {'H': 'H', 'S': 'S', 'SDG': 'S_DAG', 'X': 'X', 'Y': 'Y', 'Z': 'Z'}     # single-qubit Cliffords -> stim


class CompiledMoment:
    __slots__ = ('ops', 'base', 'meas', 'cx_c', 'cx_t', 'resets', 'clears', 'nonc', 'loc_slice')


class CompiledProtocol:
    """Pre-compiled time steps of a D4SwitchingCircuit + flattened noise-location table."""

    def __init__(self, circ, noise):
        self.circ, self.noise = circ, noise
        locs = []
        self.blocks = []
        live = set()
        for blk in circ.blocks:
            moms = []
            for mom in blk['moms']:
                cm = CompiledMoment()
                cm.ops = mom
                lines, meas, cxc, cxt, rs, cl, nc = [], [], [], [], [], [], []
                for op in mom:
                    nm, qs = op[0], op[1:]
                    if nm in NONCLIFF:
                        nc.append((nm, qs))
                    elif nm == 'FLIPX':
                        pass
                    elif nm == 'CX':
                        lines.append('CX %d %d' % qs)
                        cxc.append(qs[0]); cxt.append(qs[1])
                    elif nm == 'CZ':
                        lines.append('CZ %d %d' % qs)
                    elif nm in ('RX', 'RZ'):
                        lines.append(('RX %d' if nm == 'RX' else 'R %d') % qs[0])
                        rs.append(qs[0])
                    elif nm in ('MX', 'MZ', 'MZ0'):
                        lines.append(('MX %d' if nm == 'MX' else 'M %d') % qs[0])
                        meas.append((qs[0], 'X' if nm == 'MX' else 'Z'))
                        if nm == 'MZ0':
                            cl.append(qs[0])
                    else:
                        lines.append('%s %d' % (_STIM1[nm], qs[0]))
                cm.base = stim.Circuit('\n'.join(lines)) if lines else None
                cm.meas = meas
                cm.cx_c, cm.cx_t = np.array(cxc, dtype=np.int64), np.array(cxt, dtype=np.int64)
                cm.resets, cm.clears = np.array(rs, dtype=np.int64), np.array(cl, dtype=np.int64)
                cm.nonc = nc
                touched = {q for op in mom for q in op[1:]}
                idle = [q for q in blk['data'] if q not in touched] + [q for q in live if q not in touched]
                if all(op[0] in ('NCCZ', 'FLIPX') for op in mom):
                    idle = []          # virtual moment of the deferred switch-out (no physical time step)
                ml = [] if blk.get('noiseless') else noise.locations(mom, idle)
                s0 = len(locs)
                locs += ml
                cm.loc_slice = (s0, len(locs))
                for op in mom:
                    if op[0] in ('RX', 'RZ') and op[1] >= circ.nd:
                        live.add(op[1])
                    if op[0] in ('MX', 'MZ'):
                        live.discard(op[1])
                moms.append(cm)
            self.blocks.append(dict(name=blk['name'], code=blk['code'], moms=moms))
        self.loc_when = np.array([0 if l[0] == 'pre' else 1 for l in locs], dtype=np.int8)
        self.loc_kind = np.array([_KIND[l[1]] for l in locs], dtype=np.int8)
        self.loc_p = np.array([l[3] for l in locs])
        self.loc_q = [l[2] for l in locs]
        self.n_loc = len(locs)


class FastShot:
    """One Monte Carlo shot of the Pauli-proxy circuit (same semantics as ProxyRun, compiled)."""
    _G1 = {'S': 's', 'SDG': 's_dag', 'Z': 'z'}

    def __init__(self, comp, rng, forced=None):
        self.comp, self.rng, self.circ = comp, rng, comp.circ
        self.sim = stim.TableauSimulator(seed=int(rng.integers(2 ** 62)))
        self.sim.set_num_qubits(comp.circ.nq)
        self.fx = np.zeros(comp.circ.nq, dtype=np.uint8)
        if forced is not None:                  # {location index: pauli string}
            self.errs = {i: (comp.loc_when[i], ps, comp.loc_q[i]) for i, ps in forced.items()}
            self.n_err = len(forced)
            self.n_dress = 0
            return
        hit = np.nonzero(rng.random(comp.n_loc) < comp.loc_p)[0]
        self.errs = {}
        for i in hit:
            k = comp.loc_kind[i]
            qs = comp.loc_q[i]
            if k == 0:
                ps = 'X'
            elif k == 1:
                ps = 'Z'
            else:
                ch = PAULI_CHOICES[len(qs)]
                ps = ch[rng.integers(len(ch))]
            self.errs[i] = (comp.loc_when[i], ps, qs)
        self.n_err = len(hit)
        self.n_dress = 0

    def _pauli(self, ps, qs):
        sim, fx = self.sim, self.fx
        for pc, q in zip(ps, qs):
            if pc == 'X':
                sim.x(q); fx[q] ^= 1
            elif pc == 'Y':
                sim.y(q); fx[q] ^= 1
            elif pc == 'Z':
                sim.z(q)

    def run_block(self, bi):
        """Run block bi; returns list of (qubit, basis, outcome) of its measurements."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        nmeas = 0
        meas = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs] if b > a and self.errs else []
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        if gt[0] == 'CZ':
                            sim.cz(qs[gt[1]], qs[gt[2]])
                        else:
                            getattr(sim, self._G1[gt[0]])(qs[gt[1]])
                        self.n_dress += 1
            if cm.base is not None:
                sim.do_circuit(cm.base)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            nmeas += len(cm.meas)
            meas += cm.meas
        if not nmeas:
            return []
        rec = sim.current_measurement_record()[-nmeas:]
        return [(q, bb, int(o)) for (q, bb), o in zip(meas, rec)]

    def apply_x(self, qubits):
        for q in qubits:
            self.sim.x(q)
            self.fx[q] ^= 1

    def init_frame(self, zrows, qubits):
        """X frame at the start of the first non-Clifford block (S1).  S0 is a Clifford circuit, so stim's
        state is the true state E|ideal>, |ideal> = FSC|+_L> (all checks +1); the X part of E is any
        solution x of H_Z x = s (s = current Z-check signs): solutions differ by X stabilizers / logical X
        of |ideal>, which act trivially on it, and Algorithm 2 is exact for any such representation."""
        s = []
        for lab, qs in zrows:
            ps = stim.PauliString(self.circ.nq)
            for q in qs:
                ps[q] = 'Z'
            v = self.sim.peek_observable_expectation(ps)
            assert v in (1, -1), 'Z check not deterministic at the end of S0'
            s.append(0 if v == 1 else 1)
        H = np.zeros((len(zrows), len(qubits)), dtype=np.uint8)
        pos = {q: i for i, q in enumerate(qubits)}
        for r, (lab, qs) in enumerate(zrows):
            for q in qs:
                H[r, pos[q]] = 1
        x = f2_solve(H.T, np.array(s, dtype=np.uint8))      # x H^T = s  <=>  H x = s
        assert x is not None
        self.fx[:] = 0
        for i in np.nonzero(x)[0]:
            self.fx[qubits[i]] = 1
        return np.array(s)

    _G = {'CX': 'cx', 'CZ': 'cz', 'S': 's', 'SDG': 's_dag', 'Z': 'z', 'X': 'x', 'Y': 'y'}

    def run_block_lockstep(self, bi, ts, rng):
        """Validation: same as run_block, but op by op, with the exact simulation `ts` (TrueSim) of the
        TRUE non-Clifford circuit run in lockstep with the same errors.  Before every measurement stim's
        peek value (+1/-1 deterministic, 0 random) is compared with the exact probability of the outcome.
        Returns list of (qubit, basis, outcome, peek, exact probability)."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        out = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs]
            terrs = [('pre' if w == 0 else 'post', ps, qs) for w, ps, qs in errs]
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        getattr(sim, self._G[gt[0]])(*[qs[i] for i in gt[1:]])
                        self.n_dress += 1
            outs, peeks = [], []
            for op in cm.ops:
                nm, qs = op[0], op[1:]
                if nm in NONCLIFF or nm == 'FLIPX':
                    continue
                if nm in ('RX', 'RZ'):
                    (sim.reset_x if nm == 'RX' else sim.reset_z)(qs[0])
                elif nm in ('MX', 'MZ', 'MZ0'):
                    q = qs[0]
                    if nm == 'MX':
                        peeks.append(sim.peek_x(q)); sim.h(q); o = sim.measure(q); sim.h(q)
                    else:
                        peeks.append(sim.peek_z(q)); o = sim.measure(q)
                    outs.append(int(o))
                else:
                    getattr(sim, self._G[nm])(*qs)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            probs = ts.run_moment(cm.ops, terrs, outs, rng)
            out += [(q, bb, o, pk, pr) for (q, bb), o, pk, pr in zip(cm.meas, outs, peeks, probs)]
        return out

In [ ]:
# ------------------------------------------------------------------ decoders
def _wt(p):
    p = min(max(p, 1e-12), 0.5 - 1e-9)
    return float(np.log((1 - p) / p))


_XFRAC = {0: 1.0, 1: 0.0, 2: 2 / 3, 3: 8 / 15, 4: 32 / 63}       # prob. that a location puts X/Y on a qubit


class FluxDecoder:
    """Real-time ("just-in-time") decoding of the Pauli Z checks (non-Abelian fluxes = X errors).

    [KZH] step 2 pairs the fluxes "in real time ... whenever the observation time of the syndromes is
    comparable to the separation of the syndromes" with the decoder of [Dav] (not accessible here).
    Implementation used in this study (own construction, documented as such): after every round t the
    space-time matching graph of all rounds <= t (with an open time boundary after round t) is decoded
    by minimum-weight perfect matching (PyMatching); the space-like edges whose round is <= t - delay
    are committed and the corresponding X corrections are applied physically at once (feed-forward).
    Before the first non-Clifford block (end of S0) and before U (end of S1) everything is committed.
    The g readout of the switch-out is decoded together with the g fluxes of S1 and gives the
    corrected configuration used for kappa."""

    def __init__(self, comp, delay=0, gauge_w=0.05):
        circ = comp.circ
        self.comp, self.circ, self.delay, self.gauge_w = comp, circ, delay, gauge_w
        zr = {code: circ.zchecks[code] for code in ('D4', 'FSC')}
        g_rows = [(lab, qs) for lab, qs in zr['D4'] if lab[0] == 'plaq' and lab[2] == 'g']
        # flux rounds
        self.fr = []
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                self.fr.append(dict(block=bi, name=nm, rows=zr[blk['code']]))
            elif nm[0] == 'SO':
                self.fr.append(dict(block=bi, name=nm, rows=g_rows))
        off = 0
        for k, f in enumerate(self.fr):
            f['off'] = off
            f['idx'] = {lab: off + i for i, (lab, qs) in enumerate(f['rows'])}
            off += len(f['rows'])
        self.n_nodes = off
        # relation for the SO detectors: prod_{q in g-plaq} Z^g Z^r Z^b (pre-V) = sum of D4 rows
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        d4rows = zr['D4']
        H = np.zeros((len(d4rows), circ.nq), dtype=np.uint8)
        for i, (lab, qs) in enumerate(d4rows):
            H[i, qs] = 1
        self.so_rel = {}
        for lab, qs in g_rows:
            v = np.zeros(circ.nq, dtype=np.uint8)
            for q in qs:
                e = inv[q] // 3
                for cc in (R, G, B):
                    raw = lat.q(e, cc)
                    if raw in c:
                        v[c[raw]] ^= 1
            sol = f2_solve(H, v)
            assert sol is not None, 'SO relation not in the D4 Z-check span'
            self.so_rel[lab] = [d4rows[i][0] for i in np.nonzero(sol)[0]]
        # rows containing each qubit, per code
        self.rows_of = {code: {} for code in ('D4', 'FSC')}
        for code in ('D4', 'FSC'):
            for lab, qs in zr[code]:
                for q in qs:
                    self.rows_of[code].setdefault(q, []).append(lab)
        self._error_rates()
        self._build()

    def _error_rates(self):
        """Phenomenological edge probabilities estimated from the circuit-level noise locations."""
        comp = self.comp
        self.p_data = []           # per flux round: dict qubit -> prob of X/Y since the previous flux round
        self.p_meas = []           # per flux round: dict label -> prob that the check outcome is flipped
        prev_block = -1
        for f in self.fr:
            blk = comp.blocks[f['block']]
            pd, pa = {}, {}
            for bj in range(prev_block + 1, f['block'] + 1):
                for cm in comp.blocks[bj]['moms']:
                    a, b = cm.loc_slice
                    for i in range(a, b):
                        fr_ = _XFRAC[comp.loc_kind[i]] * comp.loc_p[i]
                        for q in comp.loc_q[i]:
                            if q < self.circ.nd:
                                pd[q] = pd.get(q, 0) + fr_
                            elif bj == f['block']:
                                pa[q] = pa.get(q, 0) + fr_
            prev_block = f['block']
            pm = {}
            if blk['code']:
                for lab, qs in f['rows']:
                    a = self.circ.P.anc[lab]
                    pm[lab] = pa.get(self.circ.c[a], 0)
            self.p_data.append(pd)
            self.p_meas.append(pm)

    def _build(self):
        """Matching graphs M_t for t = 0..len(fr)-1 (rounds <= t, open boundary after t)."""
        self.mt = []
        nfr = len(self.fr)
        for t in range(nfr):
            m = pymatching.Matching()
            look = {}

            def add(u, v, w, info):
                key = (u, v) if v == -1 else (min(u, v), max(u, v))
                if key in look and look[key][0] <= w:
                    return
                look[key] = (w, info)
            for k in range(t + 1):
                f = self.fr[k]
                code = self.comp.blocks[f['block']]['code'] or 'D4'
                rows = f['rows']
                rowset = {lab for lab, _ in rows}
                # space edges: X on a data qubit before round k
                qrows = {}
                for lab, qs in rows:
                    for q in qs:
                        qrows.setdefault(q, []).append(f['idx'][lab])
                for q, nodes in qrows.items():
                    if k == 0:
                        w = self.gauge_w           # random first-round outcomes of S0 (gauge choice)
                    else:
                        w = _wt(self.p_data[k].get(q, 1e-6))
                    if len(nodes) == 2:
                        add(nodes[0], nodes[1], w, ('s', q, k))
                    elif len(nodes) == 1:
                        add(nodes[0], -1, w, ('s', q, k))
                # time edges to the previous occurrence of the same label
                if k > 0:
                    for lab, qs in rows:
                        prev = self._prev(k, lab)
                        if prev is not None:
                            kp, node_p = prev
                            w = _wt(self.p_meas[kp].get(lab, 1e-6))
                            add(node_p, f['idx'][lab], w, ('t', lab, kp))
            # switch-out round: a measurement error of a check of the last D4 Z round enters the reference
            # of the g-readout detectors (hyper-edge with its next detector); represent it by an edge between
            # the two nodes known at this time and fix the stored reference when it is committed ('m')
            if self.fr[t]['name'][0] == 'SO':
                f = self.fr[t]
                kp = t - 1
                for lab, qs in f['rows']:
                    for l2 in self.so_rel[lab]:
                        if l2 != lab and l2 in self.fr[kp]['idx']:
                            add(self.fr[kp]['idx'][l2], f['idx'][lab], _wt(self.p_meas[kp].get(l2, 1e-6)),
                                ('m', l2, kp))
            # open time boundary: nodes whose label is measured again after t
            for k in range(t + 1):
                f = self.fr[k]
                for lab, qs in f['rows']:
                    if self._next(k, lab) is not None and self._next(k, lab) > t and self._latest(k, lab, t):
                        add(f['idx'][lab], -1, _wt(self.p_meas[k].get(lab, 1e-6) + 1e-9), ('f', lab, k))
            for (u, v), (w, info) in look.items():
                if v == -1:
                    m.add_boundary_edge(u, weight=w)
                else:
                    m.add_edge(u, v, weight=w)
            nn = self.fr[t]['off'] + len(self.fr[t]['rows'])
            self.mt.append((m, {k_: v[1] for k_, v in look.items()}, nn))

    def _prev(self, k, lab):
        for kp in range(k - 1, -1, -1):
            if lab in self.fr[kp]['idx']:
                return kp, self.fr[kp]['idx'][lab]
        return None

    def _next(self, k, lab):
        for kn in range(k + 1, len(self.fr)):
            if lab in self.fr[kn]['idx']:
                return kn
        return None

    def _latest(self, k, lab, t):
        """True if round k holds the latest occurrence of lab among rounds <= t."""
        for kk in range(k + 1, t + 1):
            if lab in self.fr[kk]['idx']:
                return False
        return True

    # ---------------------------------------------------------------- per shot
    def start(self):
        self.last = {}
        self.active = set()
        self.k = -1

    def push_round(self, k, outcomes_by_label=None, readout=None):
        """Add the detection events of flux round k.  outcomes_by_label: dict label -> bit (Z checks);
        readout: dict g-qubit -> bit (switch-out)."""
        f = self.fr[k]
        self.k = k
        if f['name'][0] == 'SO':
            par = {}
            for lab, qs in f['rows']:
                v = 0
                for q in qs:
                    v ^= readout[q]
                ref = 0
                for l2 in self.so_rel[lab]:
                    ref ^= self.last.get(l2, 0)
                par[lab] = v
                if v ^ ref:
                    self.active ^= {f['idx'][lab]}
            self.readout = dict(readout)
            return
        for lab, qs in f['rows']:
            m = outcomes_by_label[lab]
            if m ^ self.last.get(lab, 0):
                self.active ^= {f['idx'][lab]}
            self.last[lab] = m

    def decode(self, flush=False, min_age=None):
        """Decode with all rounds <= current; returns list of committed space corrections (qubit, k).
        Commits space-like edges with round k <= t - delay (flush: k <= t - min_age, default all)."""
        t = self.k
        m, look, nn = self.mt[t]
        if not self.active:
            return []
        syn = np.zeros(m.num_detectors, dtype=np.uint8)
        for u in self.active:
            syn[u] = 1
        edges = m.decode_to_edges_array(syn)
        out = []
        for u, v in edges:
            u, v = int(u), int(v)
            key = (u, -1) if v == -1 else ((v, -1) if u == -1 else (min(u, v), max(u, v)))
            info = look[key]
            if info[0] == 'm' and flush:
                self.active ^= {key[0], key[1]}
                self.last[info[1]] = self.last.get(info[1], 0) ^ 1     # it was a measurement error
                continue
            if info[0] != 's':
                continue
            q, k = info[1], info[2]
            lim = (t - (min_age or 0)) if flush else (t - self.delay)
            if k > lim:
                continue
            out.append((q, k))
            if key[1] == -1:
                self.active ^= {key[0]}
            else:
                self.active ^= {key[0], key[1]}
        return out

    def register_correction(self, q, code):
        """A physical X correction on data qubit q: flip the stored reference of every Z check of the
        current code containing q (as if it had been applied before that check was measured)."""
        for lab in self.rows_of[code].get(q, []):
            if lab in self.last:
                self.last[lab] ^= 1


class ChargeDecoder:
    """Decoding of the X-type checks (Abelian charges = Z errors) after the protocol: minimum-weight
    matching on the detector error model of the Pauli-proxy circuit (stim + PyMatching).  The DEM
    describes the Pauli errors of the circuit; the extra randomness produced by non-Abelian dressings
    is not in the model (it is present in the simulated data).  Output: predicted flip of the logical
    observable (proxy of the logical Clifford stabilizer, including the Pauli-frame strings)."""

    def __init__(self, comp, dem_noise=None, aligned=True):
        circ = comp.circ
        self.comp, self.circ = comp, circ
        nz = dem_noise or comp.noise
        # measurement bookkeeping
        meas_lab = []          # per absolute measurement: (block_name, kind, label or qubit)
        blk_meas = []
        for bi, blk in enumerate(comp.blocks):
            ids = []
            for cm in blk['moms']:
                for q, b in cm.meas:
                    if blk['code']:
                        lab = circ.lab_of[(blk['code'], q)]
                        meas_lab.append((blk['name'], b, lab))
                    else:
                        meas_lab.append((blk['name'], 'R', q))
                    ids.append(len(meas_lab) - 1)
            blk_meas.append(ids)
        self.n_meas = len(meas_lab)
        self.blk_meas = blk_meas
        idx = {}
        for i, (bn, b, lab) in enumerate(meas_lab):
            idx[(bn, lab)] = i
        self.meas_idx = idx
        sup = {code: {lab: set(xs) for lab, xs, quad in circ.xgens[code]} for code in ('D4', 'FSC')}
        names = [blk['name'] for blk in comp.blocks]
        S0 = [n for n in names if n[0] == 'S0']
        S1 = [n for n in names if n[0] == 'S1' and n[2] == 'x']
        S4 = [n for n in names if n[0] in ('S4', 'FIN')]
        dets = []
        fscl = [lab for lab, _, _ in circ.xgens['FSC']]
        d4l = [lab for lab, _, _ in circ.xgens['D4']]
        for j, n in enumerate(S0):
            for lab in fscl:
                dets.append([idx[(n, lab)]] + ([idx[(S0[j - 1], lab)]] if j else []))

        def span_solve(target, basis_labels, code):
            M = np.zeros((len(basis_labels), circ.nq), dtype=np.uint8)
            for i, l in enumerate(basis_labels):
                M[i, list(sup[code][l])] = 1
            v = np.zeros(circ.nq, dtype=np.uint8)
            v[list(target)] = 1
            sol = f2_solve(M, v)
            return None if sol is None else [basis_labels[i] for i in np.nonzero(sol)[0]]
        for j, n in enumerate(S1):
            for lab in d4l:
                if j == 0:
                    if not S0:
                        continue
                    combo = span_solve(sup['D4'][lab], fscl, 'FSC')
                    if combo is None:
                        continue                       # S^g: random first outcome (gauging)
                    dets.append([idx[(n, lab)]] + [idx[(S0[-1], l)] for l in combo])
                else:
                    dets.append([idx[(n, lab)], idx[(S1[j - 1], lab)]])
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        for j, n in enumerate(S4):
            first = {}
            for lab in fscl:
                if j == 0:
                    tgt = set(sup['FSC'][lab])
                    for q in list(sup['FSC'][lab]):
                        gq = lat.q(inv[q] // 3, G)
                        if gq in c:
                            tgt ^= {c[gq]}
                    combo = span_solve(tgt, d4l, 'D4')
                    assert combo is not None, 'S4 X check not determined by the last D4 round'
                    first[lab] = [idx[(n, lab)]] + [idx[(S1[-1], l)] for l in combo]
                else:
                    dets.append([idx[(n, lab)], idx[(S4[j - 1], lab)]])
            if j == 0:
                # detector basis across the switch-out: aligned=True uses, at a vertex with separate r and
                # b checks, D_r and D_r + D_b (S^g of the last D4 round cancels in D_r + D_b), so that a g
                # charge created during the last D4 round is a graph-like error, but an r charge then has
                # 4 symptoms; aligned=False (D_r, D_b) has the opposite property.  D4Experiment decodes
                # with both and keeps the lower-weight solution (no single basis makes all graph-like)
                for lab, d in first.items():
                    v, col = lab[1], lab[2]
                    if aligned and col == 'b' and ('X', v, 'r') in first:
                        dets.append(sorted(set(d) ^ set(first[('X', v, 'r')])))
                    else:
                        dets.append(d)
        self.dets = dets
        # logical observable: proxy logical (measured by MPP at the end of the DEM circuit) + frame terms
        tset = set(circ.t_log)
        self.cross = [len(set(fs) & tset) % 2 for fs in circ.frame_strings]
        self.obs_meas = [idx[(S1[-1], lab)] for (lab, op), cr in zip(circ.gens_g, self.cross) if cr]
        self._dem_circuit(nz)
        # sparse detector evaluation
        self.det_ptr = np.cumsum([0] + [len(d) for d in dets])
        self.det_idx = np.concatenate([np.array(d, dtype=np.int64) for d in dets])

    def _dem_circuit(self, nz):
        circ, comp = self.circ, self.comp
        lines = []
        for bi, blk in enumerate(comp.blocks):
            noiseless = blk['name'][0] == 'FIN'
            for cm in blk['moms']:
                pre, post = [], []
                touched = {q for op in cm.ops for q in op[1:]}
                a, b = cm.loc_slice
                for i in range(a, b):
                    k, p, qs = comp.loc_kind[i], comp.loc_p[i], comp.loc_q[i]
                    if k == 0:
                        ln = 'X_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 1:
                        ln = 'Z_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 2:
                        ln = 'DEPOLARIZE1(%.12g) %d' % (p, qs[0])
                    elif k == 3:
                        ln = 'DEPOLARIZE2(%.12g) %d %d' % (p, qs[0], qs[1])
                    else:
                        ln = 'DEPOLARIZE1(%.12g) %d %d %d' % (p * 48 / 63, *qs)
                    (pre if comp.loc_when[i] == 0 else post).append(ln)
                lines += pre
                if cm.base is not None:
                    lines.append(str(cm.base))
                lines += post
            lines.append('TICK')
        n = circ.nq
        lp = ['X%d' % q for q in circ.t_log if circ.Delta_a.get(q) != 2]
        lp += ['Y%d' % q for q in circ.t_log if circ.Delta_a.get(q) == 2]
        lp += ['Z%d' % q for q, a in circ.Delta_a.items() if a == 2 and q not in set(circ.t_log)]
        lines.append('MPP ' + '*'.join(lp))
        tot = self.n_meas + 1
        for d in self.dets:
            lines.append('DETECTOR ' + ' '.join('rec[%d]' % (i - tot) for i in d))
        lines.append('OBSERVABLE_INCLUDE(0) rec[-1] ' + ' '.join('rec[%d]' % (i - tot) for i in self.obs_meas))
        self.stim_circuit = stim.Circuit('\n'.join(lines))
        dem = self.stim_circuit.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True,
                                                     approximate_disjoint_errors=True)
        self.matching = pymatching.Matching.from_detector_error_model(dem)
        # graph-like components of the DEM (for the flux-informed re-weighting)
        comp = {}
        for ins in dem.flattened():
            if ins.type != 'error':
                continue
            p = ins.args_copy()[0]
            cur_d, cur_o = [], 0
            parts = []
            for t in ins.targets_copy():
                if t.is_separator():
                    parts.append((tuple(sorted(cur_d)), cur_o)); cur_d, cur_o = [], 0
                elif t.is_relative_detector_id():
                    cur_d.append(t.val)
                elif t.is_logical_observable_id():
                    cur_o ^= 1
            parts.append((tuple(sorted(cur_d)), cur_o))
            for key in parts:
                if not key[0] or len(key[0]) > 2:
                    continue
                q0 = comp.get(key, 0.0)
                comp[key] = q0 * (1 - p) + p * (1 - q0)
        keys = list(comp)
        self.comp_keys = keys
        self.comp_index = {k[0]: i for i, k in enumerate(keys)}
        from scipy.sparse import csc_matrix
        rows, cols = [], []
        for j, (dd, o) in enumerate(keys):
            for dtt in dd:
                rows.append(dtt); cols.append(j)
        nd = self.stim_circuit.num_detectors
        self.H = csc_matrix((np.ones(len(rows), dtype=np.uint8), (rows, cols)), shape=(nd, len(keys)))
        self.O = csc_matrix((np.ones(sum(o for _, o in keys), dtype=np.uint8),
                             ([0] * sum(o for _, o in keys), [j for j, (dd, o) in enumerate(keys) if o])),
                            shape=(1, len(keys)))
        self.w0 = np.array([_wt(comp[k]) for k in keys])
        # detectors containing each measurement
        self.dets_of_meas = {}
        for j, d_ in enumerate(self.dets):
            for i in d_:
                self.dets_of_meas.setdefault(i, []).append(j)

    def detectors(self, outcomes):
        o = np.asarray(outcomes, dtype=np.uint8)[self.det_idx]
        return np.bitwise_xor.reduceat(o, self.det_ptr[:-1]) if len(o) else np.zeros(0, np.uint8)

    def flux_erasures(self, flux_events, w_erase=0.1):
        """Flux-informed re-weighting (our construction, following [KZH]'s description of cleaning up the
        Abelian charges after the fluxes have been paired): a flux on qubit q present during the X-type
        round ('S1', r, 'x') randomizes the outcomes of the D4 stabilizers whose CZ dressing contains q and
        leaves Z errors on the CZ partners with probability 1/2.  The corresponding time-like components
        and the space-like components of the partner Z errors get the weight w_erase."""
        circ = self.circ
        if not hasattr(self, '_cz_of'):
            self._cz_of = {}
            for lab, xs, quad in circ.xgens['D4']:
                for (i, j) in quad:
                    self._cz_of.setdefault(i, []).append((lab, j))
                    self._cz_of.setdefault(j, []).append((lab, i))
            self._stars = {}
            for lab, xs, quad in circ.xgens['D4']:
                for q in xs:
                    self._stars.setdefault(q, []).append(lab)
        er = set()
        for q, r in flux_events:
            blk = ('S1', r, 'x')
            nxt = ('S1', r + 1, 'x')
            for lab, partner in self._cz_of.get(q, []):
                m = self.meas_idx.get((blk, lab))
                if m is None:
                    continue
                dd = tuple(sorted(self.dets_of_meas.get(m, [])))
                if 1 <= len(dd) <= 2 and dd in self.comp_index:
                    er.add(self.comp_index[dd])
                # space-like component: Z on the partner flips the two stars containing it
                st = self._stars.get(partner, [])
                for rr in (blk, nxt):
                    ds = []
                    for l2 in st:
                        m2 = self.meas_idx.get((rr, l2))
                        if m2 is not None:
                            ds += [j for j in self.dets_of_meas.get(m2, []) if self.dets[j][0] == m2]
                    dd = tuple(sorted(ds))
                    if 1 <= len(dd) <= 2 and dd in self.comp_index:
                        er.add(self.comp_index[dd])
        return er

    def predict(self, outcomes, erasures=None, w_erase=0.1):
        """Returns (predicted flip of the logical observable, weight of the matching)."""
        det = self.detectors(outcomes)
        if not erasures:
            pred, w = self.matching.decode(det, return_weight=True)
            return int(pred[0]), float(w)
        w = self.w0.copy()
        w[list(erasures)] = w_erase
        m = pymatching.Matching.from_check_matrix(self.H, weights=w, faults_matrix=self.O)
        pred, wt = m.decode(det, return_weight=True)
        return int(pred[0]), float(wt)


class D4Experiment:
    """Noisy KZH protocol + decoders.  run(shots) returns per-shot logical infidelity samples
    (0, 1/2 or 1; Takada Sec. VI D with the infidelity (1 - <H>)/2 of the output) and diagnostics."""

    def __init__(self, L, R0, Rin, Rout, noise, delay=0, pre_u_age=0, n_pre_u=2, flux_informed=True,
                 quiet_window=False, fsc_schedule='classes'):
        self.pre_u_age = pre_u_age
        self.flux_informed = flux_informed
        self.circ = D4SwitchingCircuit(L, R0, Rin, Rout, n_pre_u=n_pre_u, fsc_schedule=fsc_schedule)
        self.comp = CompiledProtocol(self.circ, noise)
        self.noise = noise
        self.flux = FluxDecoder(self.comp, delay=delay)
        self.charges = [ChargeDecoder(self.comp, aligned=a) for a in (True, False)]
        # diagnostic (not a physical model): no noise in the switching window = the last D4 round, the Z
        # rounds before U, U, the switch-out and the first FSC round, where the single faults that our
        # decoders fail to correct are located (validation N5).  Gives an optimistic bound.  The decoders keep
        # the priors of the full noise model (they are built before the window is silenced).
        self.quiet_window = quiet_window
        win = {('S1', Rin, 'z'), ('S1', Rin, 'x'), ('U',), ('SO',), ('S4', 1)}
        win |= {('S1', r, 'z') for r in range(Rin + 1, Rin + 1 + n_pre_u)}
        self.window = win
        if quiet_window:
            for blk in self.comp.blocks:
                if blk['name'] in win:
                    a, b = blk['moms'][0].loc_slice[0], blk['moms'][-1].loc_slice[1]
                    self.comp.loc_p[a:b] = 0.0
        self.gset = set(self.circ.data_g)

    def shot(self, rng, decode=True, forced=None, ts=None, init=None):
        """One Monte Carlo shot.  forced: {location index or dynamic fault id: Pauli string} = single-fault
        injection (all other noise off); dynamic fault ids are ('SOCZ', i, j) (after the physical switch-out
        CZ on i, j, if it is applied in this shot) and ('SOIDLE', layer, q).  ts: TrueSim run in lockstep
        (validation only; then the returned stats contain the lockstep record)."""
        circ, comp, fl = self.circ, self.comp, self.flux
        fstatic = {k: v for k, v in forced.items() if not isinstance(k, tuple)} if forced is not None else None
        fdyn = {k: v for k, v in forced.items() if isinstance(k, tuple)} if forced is not None else None
        sh = FastShot(comp, rng, fstatic)
        if init is not None:
            init(sh)               # validation: prepared initial state (see lockstep_full)
        lock = []
        fl.start()
        outs = []
        kfr = 0
        m_g_last = None
        flux_events = []
        stats = dict(n_err=sh.n_err, n_corr=0)
        names = [b['name'] for b in comp.blocks]
        last_S0 = max([i for i, n in enumerate(names) if n[0] == 'S0'], default=-1)
        last_S1 = max(i for i, n in enumerate(names) if n[0] == 'S1')          # Z-only round before U
        last_S1x = max(i for i, n in enumerate(names) if n[0] == 'S1' and n[2] == 'x')
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm == ('S1', 1, 'z'):
                sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
            if init is not None and nm[0] == 'S0':
                # validation start (lockstep_full): S0 replaced by the prepared state; its (noiseless,
                # ideal) outcomes are all 0
                res = [(q, bb, 0) for cm in comp.blocks[bi]['moms'] for (q, bb) in cm.meas]
            elif ts is None:
                res = sh.run_block(bi)
            else:
                r5 = sh.run_block_lockstep(bi, ts, rng)
                lock += r5
                res = [(q, bb, o) for q, bb, o, pk, pr in r5]
            outs += [o for (_, _, o) in res]
            if bi == last_S1x:
                gl = {circ.lab_of[('D4', q)]: o for (q, b, o) in res if b == 'X'}
                m_g_last = [gl[lab] for lab, _ in circ.gens_g]
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                code = blk['code']
                zl = {circ.lab_of[(code, q)]: o for (q, b, o) in res if b == 'Z'}
                fl.push_round(kfr, zl)
                kfr += 1
                if decode:
                    flush = bi in (last_S0, last_S1) or nm[0] == 'FIN'
                    # before U: corrections supported only by the newest round stay undecided (decoded
                    # after the switch-out with the S4 / g-readout data)
                    cc = fl.decode(flush=flush, min_age=self.pre_u_age if bi == last_S1 else 0)
                    corr = [q for q, k in cc]
                    if nm[0] == 'S1':
                        # the flux was present during the X-type round preceding this Z round
                        flux_events += [(q, nm[1] - 1) for q in corr if nm[1] >= 2]
                    sh.apply_x(corr)
                    if ts is not None:
                        for q in corr:
                            ts.X(q)
                    for q in corr:
                        fl.register_correction(q, code)
                    stats['n_corr'] += len(corr)
            elif nm[0] == 'SO':
                readout = {q: o for (q, b, o) in res}
                fl.push_round(kfr, readout=readout)
                kfr += 1
                a_dec = dict(readout)
                phys = []
                if decode:
                    for q, k in fl.decode(flush=True):
                        if q in self.gset:
                            a_dec[q] ^= 1
                        else:
                            phys.append(q)
                    for q in phys:
                        fl.register_correction(q, 'FSC')
                    stats['n_corr'] += len(phys)
                stats['so_cz'] = self._switch_out(sh, readout, a_dec, phys, rng, fdyn,
                                                  noisy=forced is None and not self.quiet_window, ts=ts)
        # Pauli frame of the S^g eigenvalues (last S1 round), then the logical Clifford stabilizer
        for v, m in enumerate(m_g_last):
            if m:
                for q in circ.frame_strings[v]:
                    sh.sim.z(q)
                    if ts is not None:
                        ts.Z(q)
        val = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
        if ts is not None:
            stats['lockstep'] = lock
            stats['H_true'] = ts.expect_logical(circ)
            stats['H_proxy'] = val
        if decode:
            sols = [ch.predict(outs, ch.flux_erasures(flux_events) if self.flux_informed else None)
                    for ch in self.charges]
            flip = min(sols, key=lambda s: s[1])[0]
        else:
            flip = 0
        val = val * (-1) ** flip
        stats['n_dress'] = sh.n_dress
        return (1 - val) / 2, stats

    def _switch_out(self, sh, readout, a_dec, phys, rng, fdyn=None, noisy=True, ts=None):
        """Physical switch-out after the Z^g readout ([KZH] step 4), in the order of the real protocol:
        (1) remove the virtual correction CZ(Gamma(a_raw)) of the deferred-measurement network (noiseless
            Clifford bookkeeping of the simulation, no physical operation);
        (2) the X corrections decided by the flux decoder (applied as a noiseless Pauli layer, like all
            feed-forward X corrections of this simulation);
        (3) the physical CZ(Gamma(a_dec)) in the fixed schedule circ.so_colour: in every layer each CZ is
            followed by DEPOLARIZE2(p) and every other rb data qubit by DEPOLARIZE1(p_idle) (exact Pauli
            propagation through the later layers, because they are applied in order).
        fdyn: forced dynamic faults (single-fault injection).  Returns the number of physical CZs."""
        circ, sim = self.circ, sh.sim
        ar = np.array([readout[q] for q in circ.data_g], dtype=np.uint8)
        ad = np.array([a_dec[q] for q in circ.data_g], dtype=np.uint8)
        kr = (circ.Gamma.astype(int) @ ar) % 2
        kd = (circ.Gamma.astype(int) @ ad) % 2

        def pairs_of(k):
            cnt = {}
            for v in np.nonzero(k)[0]:
                for pr in circ.gpairs[v]:
                    pr = tuple(sorted(pr))
                    cnt[pr] = cnt.get(pr, 0) ^ 1
            return sorted(pr for pr, c in cnt.items() if c)

        def pauli(ps, qs):
            sh._pauli(ps, qs)
            if ts is not None:
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(ts, pc)(q)

        def cz(i, j):
            sim.cz(i, j)
            if ts is not None:
                ts.diag((i, j), lambda x: 4 * x[0] * x[1])
        for (i, j) in pairs_of(kr):                       # (1)
            cz(i, j)
        sh.apply_x(phys)                                  # (2)
        if ts is not None:
            for q in phys:
                ts.X(q)
        gates = pairs_of(kd)                              # (3)
        nl = circ.so_steps - 1
        p, pi = self.noise.p, self.noise.p_idle
        for layer in range(nl):
            here = [pr for pr in gates if circ.so_colour[pr] == layer]
            busy = {q for pr in here for q in pr}
            for (i, j) in here:
                cz(i, j)
            for (i, j) in here:
                if noisy and p > 0 and rng.random() < p:
                    pauli(PAULI_CHOICES[2][rng.integers(15)], (i, j))
                if fdyn and ('SOCZ', i, j) in fdyn:
                    pauli(fdyn[('SOCZ', i, j)], (i, j))
            for q in circ.data_rb:
                if q in busy:
                    continue
                if noisy and pi > 0 and rng.random() < pi:
                    pauli('XYZ'[rng.integers(3)], (q,))
                if fdyn and ('SOIDLE', layer, q) in fdyn:
                    pauli(fdyn[('SOIDLE', layer, q)], (q,))
        return len(gates)

    def run(self, shots, rng, decode=True):
        inf = np.zeros(shots)
        st = []
        for s in range(shots):
            inf[s], d = self.shot(rng, decode)
            st.append(d)
        return inf, st

In [ ]:
# ------------------------------------------------------------------ applicability of Takada et al.
def equal_up_to_phase(A, B, tol=1e-12):
    """True iff A = e^{i phi} B for one global phase (exact operator comparison, NOT elementwise |.|)."""
    A, B = np.asarray(A, complex), np.asarray(B, complex)
    if A.shape != B.shape:
        return False
    k = np.unravel_index(np.argmax(np.abs(B)), B.shape)
    if abs(B[k]) < tol:
        return bool(np.allclose(A, 0, atol=tol))
    ph = A[k] / B[k]
    return bool(abs(abs(ph) - 1) < 1e-9 and np.allclose(A, ph * B, atol=1e-9))


def takada_conditions(L):
    """Algebraic checks of the sufficient conditions of Takada et al. (Definition 2 and Sec. VIII) for
    the D4 code switching protocol on the L x L lattice.  Returns dict name -> (passed, detail)."""
    P = D4Protocol(L)
    lat, d4, fsc = P.lat, P.d4, P.fsc
    res = {}
    # (a) admissible factorisation: every X-type Clifford stabilizer = X factors x CZ factors with disjoint
    #     supports (transversal, so every effectively measured operator stays Hermitian, Eq. (131))
    bad = 0
    for lab, op in d4.xgens:
        qs = list(op.s) + [q for pr in op.quad for q in pr]
        bad += len(qs) != len(set(qs))
    res['factors transversal (X and CZ factors on disjoint qubits)'] = (bad == 0, '%d generators' % len(d4.xgens))
    # (b) Pauli dressings: X/Y errors on the Clifford factors (CZ; S, CZ in the logical stabilizer; S,
    #     CZ dressings) give weight-1 Z dressings; all dressings commute (Eq. (91))
    I2 = np.eye(2); X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Zm = np.diag([1, -1])
    CZm = np.diag([1, 1, 1, -1]); Sm = np.diag([1, 1j])
    ok = True
    for E in (X, Y):
        for k in (0, 1):                                                   # error on either qubit of the CZ
            Ek = np.kron(E, I2) if k == 0 else np.kron(I2, E)
            D = Ek @ CZm @ Ek.conj().T @ CZm.conj().T                          # group commutator [E, CZ]
            ok &= equal_up_to_phase(D, np.kron(I2, Zm) if k == 0 else np.kron(Zm, I2))
        D1 = E @ Sm @ E.conj().T @ Sm.conj().T                                 # [E, S]
        ok &= equal_up_to_phase(D1, Zm)
    for E in (Zm,):                                                        # Z errors commute: no dressing
        ok &= equal_up_to_phase(np.kron(E, I2) @ CZm @ np.kron(E, I2).conj().T @ CZm.conj().T, np.eye(4))
    res['dressings of CZ / S by X, Y are single-qubit Z on the partner / same qubit (exact up to a global '
        'phase); Z errors give no dressing'] = (bool(ok), '')
    # (c) effective commutation: group commutators of measured generators are Z-type Paulis in the
    #     persistent Z-stabilizer span (D4 Z checks), Eq. (58) / Lemma 1
    HZr, HZp = f2_rref(d4.HZ)
    nbad, npair = 0, 0
    for (l1, a), (l2, b) in itertools.combinations(d4.xgens, 2):
        lin, const = commutator_affine(a, b)
        npair += 1
        if lin and not f2_in_rowspace(HZr, HZp, lat.vec(lin)):
            nbad += 1
    res['D4 generators: commutators in the Z-stabilizer span'] = (nbad == 0, '%d pairs' % npair)
    # switch-in: S^g measured on FSC|+_L> (x) |0>_g: commutators with the FSC generators and the logical X
    # must lie in the span of the FSC Z checks and the single-qubit Z^g (current Z stabilizers)
    # The state before switch-in, FSC|+_L> (x) |0>_g, is stabilized by the FSC Z checks, the single-qubit
    # Z^g, and by the D4-dressed X-type operators S^r_v, S^b_v (their CZ^{g,.} dressing acts trivially
    # on |0>_g) and a dressed logical X = X^t D (D diagonal Clifford, also trivial on |0>_g).  Takada
    # Sec. VIII B: the generators may be chosen such that the new measurements (S^g) effectively commute
    # or anticommute with them.  Check this choice:
    zin = [rows for lab, rows in lat.z_checks((R, B))] + [{lat.q(e, G)} for e in range(lat.E)]
    Hin = np.array([lat.vec(r) for r in zin], dtype=np.uint8)
    Hir, Hip = f2_rref(Hin)
    gens_in = [op for lab, op in d4.xgens if lab[2] != 'g']
    nbad = 0
    for lab, op in d4.xgens:
        for g in gens_in:
            lin, const = commutator_affine(op, g)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
    # dressed logical X: find D = prod CZ^{c_k} over the CZ pairs of the D4 generators with
    # [X^t D, S] in the D4 Z-stabilizer span for every X-type generator S (linear system over F2)
    t = set(int(q) for q in np.nonzero(fsc.t_log)[0])
    pairs = sorted({pr for _, op in d4.xgens for pr in op.quad})
    rows_eq = []                          # for each generator: commutator linear part as function of c
    HZr2, HZp2 = f2_rref(d4.HZ)
    # basis of the complement of the Z span to express "lin in span" as linear equations
    Npre = f2_nullspace(d4.HZ)            # vectors orthogonal to the span: lin in span <=> N.lin = 0
    A_cols, b_vec = [], None
    base_list = []
    for lab, op in d4.xgens:
        lin0, _ = commutator_affine(PSOp(s=t), op)
        base_list.append(lat.vec(lin0) if lin0 else np.zeros(lat.n, dtype=np.uint8))
    for pr in pairs:
        col = []
        for lab, op in d4.xgens:
            lin, _ = commutator_affine(PSOp(quad=[pr]), op)
            col.append(lat.vec(lin) if lin else np.zeros(lat.n, dtype=np.uint8))
        A_cols.append(col)
    # equations: N @ (base_v + sum_k c_k col_k,v) = 0 for all v
    eqA = np.array([[int((Npre.astype(int) @ A_cols[k][v].astype(int) % 2)[i]) for k in range(len(pairs))]
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    eqb = np.array([int((Npre.astype(int) @ base_list[v].astype(int) % 2)[i])
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    sol = f2_solve(eqA.T, eqb) if eqA.size else np.zeros(0, dtype=np.uint8)
    logical_ok = sol is not None
    if logical_ok:
        D = [pr for pr, cbit in zip(pairs, sol) if cbit]
        Xbar = PSOp(s=t, quad=D)
        # dressing trivial on |0>_g: every CZ of D touches a g qubit
        triv = all(any(q % 3 == G for q in pr) for pr in D)
        for lab, op in d4.xgens:
            lin, const = commutator_affine(Xbar, op)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
        logical_ok = triv
    res['switch-in: S^g vs dressed generators of FSC|+_L>|0>_g (current Z span), dressed logical X exists'] = (
        nbad == 0 and logical_ok, 'logical dressing with %s CZ pairs' % (len(D) if logical_ok else '-'))
    # (d) primary-support exactness (Definition 1): X supports of the X-type generators (and the logical
    #     X) are linearly independent, so no product of them is a non-Pauli operator without X part
    Ssup = np.vstack([d4.S, d4.t_log[None, :]])
    rk = f2_rank(Ssup[:, d4.cols])
    res['primary-support exactness (independent X supports)'] = (rk == Ssup.shape[0], 'rank %d / %d' % (rk, Ssup.shape[0]))
    # (e) the transversal U and the deferred switch-out network consist of CNOTs and diagonal third-level
    #     gates only (Sec. VIII A, Eq. (178)): Pauli / diagonal-Clifford errors stay of that form
    names = {g[0] for g in P.U_gates()}
    res['U = CNOT + diagonal 3rd-level gates (Sec. VIII A)'] = (names <= {'CNOT', 'CS', 'CSDG', 'TDG'}, str(sorted(names)))
    # (f) logical Clifford stabilizer of the output: Clifford (no cubic / odd terms) and effectively
    #     commuting with every FSC X check (dressing = Z string in the FSC Z span with sign +-1)
    circ = D4SwitchingCircuit(L, 1, 1, 1)
    Hf = np.array([[1 if q in qs else 0 for q in range(circ.nq)] for lab, qs in circ.zchecks['FSC']], dtype=np.uint8)
    Hfr, Hfp = f2_rref(Hf)
    nbad = 0
    for lab, xs, quad in circ.xgens['FSC']:
        fx = np.zeros(circ.nq, dtype=np.uint8)
        fx[xs] = 1
        R_ = circ.logical_proxy(fx) * circ.logical_proxy(np.zeros(circ.nq, dtype=np.uint8))
        zpart = np.array([1 if R_[q] in (2, 3) else 0 for q in range(circ.nq)], dtype=np.uint8)
        xpart = np.array([1 if R_[q] in (1, 2) else 0 for q in range(circ.nq)], dtype=np.uint8)
        if xpart.any() or (zpart.any() and not f2_in_rowspace(Hfr, Hfp, zpart)) or R_.sign.imag != 0:
            nbad += 1
    res['output logical stabilizer: Clifford, effectively commutes with FSC X checks'] = (nbad == 0, '')
    return res


def lockstep_validation(L, Rin, Rout, noise, rng, sequential=False, corr_prob=0.05):
    """Run the Pauli-proxy shot (FastShot) and the exact simulation of the TRUE circuit (TrueSim) in
    lockstep from the start of S1 (initial state E|+_L>_FSC|0>_g with a random Pauli E on the FSC data),
    with the same sampled circuit-level errors and random feed-forward X corrections.  Returns dict with
    the number of measurements, the number of mismatches (deterministic vs random, probability of the
    outcome), and the final <H> from both simulations."""
    circ = D4SwitchingCircuit(L, 0, Rin, Rout, sequential=sequential)
    comp = CompiledProtocol(circ, noise)
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st in (zero, one):
        for k, v in st.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    keys = [sum(1 << c[q] for q in k) for k in amps]
    ts = TrueSim(circ.nd, keys, list(amps.values()))
    sh = FastShot(comp, rng)
    n = circ.nq
    stabs = []
    for lab, xs, quad in circ.xgens['FSC']:
        ps = stim.PauliString(n)
        for q in xs:
            ps[q] = 'X'
        stabs.append(ps)
    for lab, qs in circ.zchecks['FSC']:
        ps = stim.PauliString(n)
        for q in qs:
            ps[q] = 'Z'
        stabs.append(ps)
    ps = stim.PauliString(n)
    for q in circ.t_log:
        ps[q] = 'X'
    stabs.append(ps)
    for q in range(n):
        if q not in circ.data_rb:
            ps = stim.PauliString(n)
            ps[q] = 'Z'
            stabs.append(ps)
    sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
    for q in rng.choice(circ.data_rb, 2, replace=False):
        pc = 'XYZ'[rng.integers(3)]
        getattr(ts, pc)(int(q)); getattr(sh.sim, pc.lower())(int(q))
    sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
    nmeas = nbad = nrand = 0
    m_g = None
    for bi, blk in enumerate(comp.blocks):
        res = sh.run_block_lockstep(bi, ts, rng)
        for q, bb, o, pk, pr in res:
            nmeas += 1
            if pk == 0:
                nrand += 1
                nbad += abs(pr - 0.5) > 1e-9
            else:
                nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
        nm = blk['name']
        if nm[0] == 'S1' and nm[2] == 'x':
            gl = {circ.lab_of[('D4', q)]: o for q, bb, o, pk, pr in res if bb == 'X'}
            m_g = [gl[lab] for lab, _ in circ.gens_g]
        if corr_prob and nm[0] in ('S1', 'S4') and rng.random() < corr_prob * len(blk['moms']):
            q = int(rng.choice(circ.data_rb if nm[0] == 'S4' else circ.data_all))
            sh.apply_x([q]); ts.X(q)
    for v, m in enumerate(m_g):
        if m:
            for q in circ.frame_strings[v]:
                sh.sim.z(q); ts.Z(q)
    true = ts.expect_logical(circ)
    proxy = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
    return dict(n_meas=nmeas, n_random=nrand, n_mismatch=int(nbad), H_true=true, H_proxy=proxy,
                n_dress=sh.n_dress, agree=bool(nbad == 0 and abs(true - proxy) < 1e-9))


def lockstep_full(L, Rin, Rout, noise, rng, ex=None):
    """Validation of the adaptive protocol AS USED IN PRODUCTION (D4Experiment.shot, from S1 on): real-time
    flux decoding with X corrections, the switch-out with the decoded readout and the physical CZ layers
    with their noise, the Pauli frame and the charge decoder, run in lockstep on the Pauli-proxy circuit
    (stim) and on the exact sparse state vector of the TRUE non-Clifford circuit (TrueSim), with the same
    sampled errors and outcomes.  S0 (a Clifford circuit, simulated exactly by stim alone) is replaced by
    its ideal output E|+_L>_FSC|0>_g with a random 2-qubit Pauli E on the FSC data (noiseless S0 outcomes
    for the decoders; the exact state vector of S0's many parallel ancillas does not fit in memory).  Checks every measurement (proxy
    deterministic <-> exact probability 1, proxy random <-> exact 1/2) and the final <H> before the
    charge-decoder correction."""
    ex = ex or D4Experiment(L, 1, Rin, Rout, noise)
    circ = ex.circ
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st_ in (zero, one):
        for k, v in st_.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    ts = TrueSim(circ.nd, [sum(1 << c[q] for q in k) for k in amps], list(amps.values()))
    n = circ.nq
    errs = [('XYZ'[rng.integers(3)], int(q)) for q in rng.choice(circ.data_rb, 2, replace=False)]

    def init(sh):
        stabs = []
        for lab, xs, quad in circ.xgens['FSC']:
            ps = stim.PauliString(n)
            for q in xs:
                ps[q] = 'X'
            stabs.append(ps)
        for lab, qs in circ.zchecks['FSC']:
            ps = stim.PauliString(n)
            for q in qs:
                ps[q] = 'Z'
            stabs.append(ps)
        ps = stim.PauliString(n)
        for q in circ.t_log:
            ps[q] = 'X'
        stabs.append(ps)
        for q in range(n):
            if q not in circ.data_rb:
                ps = stim.PauliString(n)
                ps[q] = 'Z'
                stabs.append(ps)
        sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
        for pc, q in errs:
            getattr(ts, pc)(q)
            getattr(sh.sim, pc.lower())(q)
    inf, st = ex.shot(rng, ts=ts, init=init)
    nbad = nrand = 0
    for q, bb, o, pk, pr in st['lockstep']:
        if pk == 0:
            nrand += 1
            nbad += abs(pr - 0.5) > 1e-9
        else:
            nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
    ht, hp = complex(st['H_true']), float(st['H_proxy'])
    return dict(n_meas=len(st['lockstep']), n_random=nrand, n_mismatch=int(nbad), H_true=ht.real, H_proxy=hp,
                n_err=st['n_err'], n_corr=st['n_corr'], so_cz=st['so_cz'], infidelity=float(inf),
                agree=bool(nbad == 0 and abs(ht - hp) < 1e-9))


def single_fault_scan(ex, n_sample=None, seeds=2, rng_seed=1, keep_failures=True, only_window=False):
    """Single-fault injection over ALL noise locations of the protocol as simulated: every static
    location of the compiled circuit (gates incl. CCZ / CS / T^dag, resets, measurements, idling) with
    every Pauli of its channel, plus the dynamic switch-out faults (each Pauli after each possible physical
    CZ, idle faults of the CZ layers).  All other noise is off.  Every fault is run with `seeds` different
    measurement-randomness seeds (the outcomes of random measurements, hence the branches, differ); a
    fault 'fails' if any seed gives a nonzero output infidelity.  Not covered: other branches than the
    sampled ones.  only_window: restrict to the faults in the switching window (ex.window).
    Returns dict with the counts, the probability-weighted first-order coefficient
        c1 = sum_f (P(f) / p) * mean_seeds(infidelity)      (eps ~ c1 p for small p, this decoder),
    its split into the switching window (ex.window) and the rest, and (optionally) the failing faults."""
    comp, circ, nz = ex.comp, ex.circ, ex.noise
    p = nz.p
    names = [b['name'] for b in comp.blocks]
    blk_of = np.zeros(comp.n_loc, dtype=int)
    for bi, b in enumerate(comp.blocks):
        for cm in b['moms']:
            a_, b_ = cm.loc_slice
            blk_of[a_:b_] = bi
    faults = []                                   # (fault id, pauli, weight P(f)/p, block name)
    for i in range(comp.n_loc):
        k, qs, pl = comp.loc_kind[i], comp.loc_q[i], comp.loc_p[i]
        if pl == 0:
            continue
        chs = ['X'] if k == 0 else ['Z'] if k == 1 else PAULI_CHOICES[len(qs)]
        for ps in chs:
            faults.append((int(i), ps, pl / p / len(chs), names[blk_of[i]]))
    for pr in circ.so_colour:
        for ps in PAULI_CHOICES[2]:
            faults.append((('SOCZ',) + pr, ps, 1 / 15, ('SO',)))
    if nz.p_idle > 0:
        for layer in range(circ.so_steps - 1):
            for q in circ.data_rb:
                for ps in 'XYZ':
                    faults.append((('SOIDLE', layer, q), ps, nz.p_idle / p / 3, ('SO',)))
    if only_window:                               # exhaustive over the switching window only
        faults = [f for f in faults if f[3] in ex.window]
    r_ = np.random.default_rng(rng_seed)
    n_all = len(faults)
    if n_sample and n_sample < n_all:
        idx = r_.choice(n_all, n_sample, replace=False)
        faults = [faults[j] for j in idx]
        scale = n_all / n_sample                  # unbiased estimate of the full sum
    else:
        scale = 1.0
    c1 = c1_win = 0.0
    nfail = nfail_win = 0
    by_block, fails = {}, []
    for fid, ps, w, bn in faults:
        infs = [ex.shot(np.random.default_rng((rng_seed, s_, zlib.crc32(str(fid).encode()))), forced={fid: ps})[0]
                for s_ in range(seeds)]
        m = float(np.mean(infs))
        c1 += w * m * scale
        inwin = bn in ex.window
        if inwin:
            c1_win += w * m * scale
        if max(infs) > 0:
            nfail += 1
            nfail_win += inwin
            by_block[str(bn)] = by_block.get(str(bn), 0) + 1
            if keep_failures:
                fails.append(dict(fault=str(fid), pauli=ps, block=str(bn), in_window=inwin, infidelities=infs))
    return dict(n_faults_total=n_all, n_tested=len(faults), seeds=seeds, n_fail=nfail, n_fail_window=nfail_win,
                n_fail_outside=nfail - nfail_win, by_block=by_block, c1=c1, c1_window=c1_win,
                c1_outside=c1 - c1_win, failures=fails)


def block_steps(circ, blk):
    """Physical time steps of a block: the switch-out is the g readout plus the fixed CZ-layer schedule
    (its simulation moments FLIPX / NCCZ / MZ0 are a deferred-measurement representation, not time)."""
    return circ.so_steps if blk['name'][0] == 'SO' else len(blk['moms'])

### 雑音付き 15-to-1 ライブラリ（Litinski の誤り予算・密度行列計算・GoSC 配置・表面符号の p_L 較正）

In [ ]:
"""msd_noisy_lib: output error and cost of 15-to-1 magic-state distillation under circuit-level noise.

Error model (D. Litinski, "Magic State Distillation: Not as Costly as You Think", Quantum 3, 205 (2019),
and its published resource-estimation code github.com/litinski/magicstates, Apache-2.0; re-implemented
here, the error budgets are transcribed from that code):
  * the distillation circuit acts on 5 logical qubits (surface-code patches) and consists of Z-type
    pi/8 Pauli-product rotations; a faulty rotation is followed by a P_{pi/2}, P_{-pi/4} or P_{pi/4}
    error with probabilities p1, p2, p3 (faulty magic state: p/3 each; lattice-surgery measurement
    errors and errors in the ancilla region add to p1, p2);
  * patches suffer X / Z storage errors with probability 0.5 * p_L(d) per code cycle each;
  * p_L(p, d) = 0.1 (100 p)^((d+1)/2) per patch and code cycle (GoSC Eq. (10)), or a calibration of the
    same form obtained by stim simulations of the rotated surface code under the SAME circuit-level
    noise model as the D4 simulation (`calibrate_surface_code`);
  * the output error is the infidelity of the post-selected output state (exact density-matrix
    calculation, 128-bit arithmetic), the success probability is the post-selection probability.
Protocols:
  * `litinski2019_one_level(p, dx, dz, dm)` and `litinski2019_two_level(...)`: Litinski's optimised
    factories (transcribed; validated against the outputs stored in his notebook);
  * `gosc_15to1(p, d)` and `gosc_225to1(p, d1, d2)`: the A Game of Surface Codes Sec. 3 layouts
    (Fig. 17c: 11 tiles x 11 tau; Fig. 19: 176 tiles x 15 tau) evaluated with the same error model.
    The per-rotation error budget of these layouts is OUR adaptation of Litinski (2019) to the GoSC
    geometry (single distance d, two ancilla tiles per multi-patch measurement) - see `gosc_steps`.
"""
import numpy as np
import itertools
import mpmath
from mpmath import mp

mp.prec = 128


def plog_gosc(p, d):
    """GoSC Eq. (10) / Litinski (2019): logical error per patch per code cycle."""
    return mpmath.mpf('0.1') * (100 * mpmath.mpf(p)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ 5-qubit density-matrix engine
class DM5:
    """Density matrix of n (=5) qubits.  backend='ld': numpy extended precision (np.clongdouble, ~19 digits,
    fast; used for the searches); backend='mp': numpy object array of mpmath.mpc (128 bits; used to check
    the transcription against Litinski's published outputs down to 1e-25).  Qubit 1 = most significant bit
    (Litinski's kron ordering)."""

    def __init__(self, n=5, singles=None, backend='ld'):
        self.n, self.backend = n, backend
        N = 2 ** n
        idx = np.arange(N)
        self.bits = [(idx >> (n - 1 - q)) & 1 for q in range(n)]
        if singles is None:
            h = 1 / mpmath.sqrt(2) if backend == 'mp' else np.longdouble(1) / np.sqrt(np.longdouble(2))
            singles = [[h, h]] * n
        if backend == 'mp':
            psi = np.array([mpmath.mpc(1)] * N, dtype=object)
            for q, s in enumerate(singles):
                psi = psi * np.array([mpmath.mpc(s[b]) for b in self.bits[q]], dtype=object)
        else:
            psi = np.ones(N, dtype=np.clongdouble)
            for q, s in enumerate(singles):
                psi = psi * np.array([s[b] for b in self.bits[q]], dtype=np.clongdouble)
        self.rho = np.outer(psi, np.conj(psi))

    def _f(self, x):
        return mpmath.mpf(x) if self.backend == 'mp' else np.longdouble(float(x)) if not isinstance(x, np.longdouble) else x

    def _eig(self, axis):
        s = np.ones(2 ** self.n, dtype=int)
        for q, a in enumerate(axis):
            if a:
                s = s * (1 - 2 * self.bits[q])
        return s

    def rot(self, axis, p1=0, p2=0, p3=0):
        """Z-type Pauli-product rotation exp(i pi/8 P) followed by the faulty-rotation channel of
        Litinski's `apply_rot`: with prob p1 / p2 / p3 the angle is 5pi/8 / -pi/8 / 3pi/8."""
        s = self._eig(axis)
        diff = s[:, None] - s[None, :]            # in {-2, 0, 2}
        if self.backend == 'mp':
            p1, p2, p3 = mpmath.mpf(p1), mpmath.mpf(p2), mpmath.mpf(p3)
            w = [(1 - p1 - p2 - p3, mp.pi / 8), (p1, 5 * mp.pi / 8), (p2, -mp.pi / 8), (p3, 3 * mp.pi / 8)]
            m2 = sum(wk * mpmath.expj(2 * ph) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, mpmath.conj(m2), mpmath.mpf(1)))
        else:
            p1, p2, p3 = (np.longdouble(float(x)) for x in (p1, p2, p3))
            pi = np.longdouble('3.14159265358979323846264338327950288')
            w = [(1 - p1 - p2 - p3, pi / 8), (p1, 5 * pi / 8), (p2, -pi / 8), (p3, 3 * pi / 8)]
            m2 = sum(wk * (np.cos(2 * ph) + 1j * np.sin(2 * ph)) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, np.conj(m2), 1)).astype(np.clongdouble)
        self.rho = self.rho * M

    def z_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        sg = (1 - 2 * self.bits[q])
        self.rho = self.rho * np.where(sg[:, None] * sg[None, :] == 1, 1, 1 - 2 * p)

    def x_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        f = np.arange(2 ** self.n) ^ (1 << (self.n - 1 - q))
        self.rho = (1 - p) * self.rho + p * self.rho[np.ix_(f, f)]

    def store(self, px, pz):
        for q in range(self.n):
            self.x_err(q, px[q])
            self.z_err(q, pz[q])

    def postselect_x(self, measured):
        """Project the qubits in `measured` onto |+> (X = +1).  Returns (success probability, normalised
        2x2 density matrix of the remaining qubit)."""
        keep = [q for q in range(self.n) if q not in measured]
        assert len(keep) == 1
        k = keep[0]
        out = np.zeros((2, 2), dtype=self.rho.dtype)
        for a in range(2):
            for b in range(2):
                ia = self.bits[k] == a
                ib = self.bits[k] == b
                out[a, b] = self.rho[np.ix_(ia, ib)].sum() / 2 ** len(measured)
        ps = (out[0, 0] + out[1, 1]).real
        return ps, out / ps


def run_steps(steps, n=5, out_qubit=0, init=None, backend='ld'):
    """Evaluate a protocol given as a list of steps
       ('rot', axis (0/1 tuple), p1, p2, p3)   faulty Z-type pi/8 rotation
       ('store', px (list), pz (list))        X / Z Pauli channels
    on |+>^n; the qubits != out_qubit are measured in X and post-selected on +1.  Returns
    (success probability, output infidelity).  The ideal output is taken from the same circuit without
    errors; the infidelity is evaluated as the population <psi_perp|rho|psi_perp> of the orthogonal
    state (no cancellation in 1 - F)."""
    def go(noisy):
        dm = DM5(n, init, backend)
        for st in steps:
            if st[0] == 'rot':
                dm.rot(st[1], *(st[2:5] if noisy else (0, 0, 0)))
            elif st[0] == 'store' and noisy:
                dm.store(st[1], st[2])
        return dm.postselect_x([q for q in range(n) if q != out_qubit])
    ps0, r0 = go(False)
    # ideal output |psi> from the noiseless (pure) reduced state: rho0 = |psi><psi|
    if backend == 'mp':
        a = mpmath.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a] if a != 0 else [mpmath.mpc(0), mpmath.mpc(1)]
        perp = [-mpmath.conj(psi[1]), mpmath.conj(psi[0])]
    else:
        a = np.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a]
        perp = [-np.conj(psi[1]), np.conj(psi[0])]
    ps, r = go(True)
    eps = sum(np.conj(perp[i]) * r[i, j] * perp[j] for i in range(2) for j in range(2)).real
    return ps, eps


def infidelity(rho1, ideal1):
    """1 - <ideal|rho|ideal> for a single-qubit rho (2x2) and an ideal ket."""
    f = sum(np.conj(ideal1[a]) * rho1[a, b] * ideal1[b] for a in range(2) for b in range(2))
    return 1 - f.real

In [ ]:
# ------------------------------------------------------------------ Litinski (2019), transcribed
_Z = 1


def _ax(*qs):
    return tuple(int(q in qs) for q in range(5))


def litinski2019_one_level_steps(p, dx, dz, dm, plog=plog_gosc):
    """Error budget of Litinski's (15-to-1)_{dx,dz,dm} (onelevel15to1.py), qubit 1 = output (index 0)."""
    p = mpmath.mpf(p)
    px, pz, pm = plog(p, dx), plog(p, dz), plog(p, dm)
    t = p / 3
    S = []
    single = lambda q: ('rot', _ax(q), t + 0.5 * (dm / dz) * pz * dm, t + 0.5 * dz * pm, t)
    multi = lambda qs, w: ('rot', _ax(*qs), t + 0.5 * pm * dm, t + 0.5 * pm * dm + 0.5 * w * dx / dm * pm, t)
    sx = lambda c0, c: ('store', [c0] + [c] * 4, [0] * 5)
    S += [single(1), single(2), single(3), multi((1, 2, 3), 3 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 3 + [0], [0] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 1, 2), dx + 2 * dz), multi((0, 1, 3), dx + 3 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 2 * dz) + (dx + 3 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [('store', [0.5 * px * dm] + [0.5 * (dz / dx) * px * dm] * 3 + [0],
           [0.5 * px * dm] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 2, 3), dx + 3 * dz), multi((0, 3, 4), dx + 4 * dz), single(4)]
    S += [('store', [0] * 5, [0.5 * ((dx + 3 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    full = lambda a: ('store', [a * px] + [0.5 * (dz / dx) * px * dm] * 4, [a * px] + [0.5 * (dx / dz) * pz * dm] * 4)
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 4), dx + 4 * dz), multi((0, 2, 4), dx + 4 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 4 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 2, 3, 4), dx + 4 * dz), multi((2, 3, 4), 3 * dz)]
    S += [('store', [0] * 5, [0.5 * (dx + 4 * dz) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * (dm + 2 * dx))]
    S += [multi((1, 3, 4), 4 * dz), multi((1, 2, 4), 4 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 4, [0] + [0.5 * (dx / dz) * pz * dm] * 4)]
    return S


def litinski2019_one_level(p, dx, dz, dm, plog=plog_gosc, backend='ld'):
    ps, eps = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    qubits = 2 * ((dx + 4 * dz) * 3 * dx + 2 * dm)
    cycles = 6 * dm / ps
    return dict(protocol='Litinski2019 (15-to-1)_{%d,%d,%d}' % (dx, dz, dm), eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1)


def litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    p = mpmath.mpf(p)
    px2, pz2, pm2 = plog(p, dx2), plog(p, dz2), plog(p, dm2)
    ps1, pl1 = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    ps1, pl1 = mpmath.mpf(float(ps1)) if backend != 'mp' else ps1, mpmath.mpf(float(pl1)) if backend != 'mp' else pl1
    pfail = 1 - ps1
    l1time = max(6 * dm / (mpmath.mpf(nl1) / 2) / (1 - pfail), dm2)
    lmove = 10 * dm2 + mpmath.mpf(nl1) / 4 * (dx + 4 * dz)
    R = lambda qs, w: ('rot', _ax(*qs), pl1 + 0.5 * lmove * pm2,
                       0.5 * lmove * pm2 + 0.5 * w * dx2 / dm2 * pm2, 0)
    st = lambda mask, c0=0: ('store', [c0] + [0.5 * (dz2 / dx2) * px2 * l1time if m else 0 for m in mask],
                             [c0] + [0.5 * (dx2 / dz2) * pz2 * l1time if m else 0 for m in mask])
    zs = lambda v: ('store', [0] * 5, [v, 0, 0, 0, 0])
    a = 0.5 * px2 * l1time
    S = [R((1,), dx2 + dz2 + dm2), R((2,), 3 * dz2 + dm2), st([1, 1, 0, 0]),
         R((3,), dx2 + 3 * dz2 + dm2), R((4,), dz2 + dm2), st([1, 1, 1, 1]),
         R((0, 1, 2), dx2 + 2 * dz2 + dm2), R((1, 2, 3), 4 * dz2 + dm2),
         zs(0.5 * (dx2 + 2 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 3), dx2 + 3 * dz2 + dm2), R((0, 1, 3), dx2 + 4 * dz2 + dm2),
         zs(0.5 * ((dx2 + 3 * dz2 + dm2) + (dx2 + 4 * dz2 + dm2)) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 1, 4), dx2 + 4 * dz2 + dm2), R((0, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 4), dx2 + 4 * dz2 + dm2), R((0, 1, 2, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((1, 3, 4), dx2 + 4 * dz2 + dm2), R((2, 3, 4), 3 * dz2 + dm2),
         zs(0.5 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], 0.5 * px2 * (dm2 + 2 * dx2)),
         R((1, 2, 4), 4 * dz2 + dm2), st([1, 1, 0, 1])]
    # note: Litinski's code passes the X/Z storage of qubit 1 only where stated (copied as above)
    return S, ps1, pl1, l1time


def litinski2019_two_level(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    S, ps1, pl1, l1time = litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog, backend)
    ps2, eps = run_steps(S, out_qubit=0, backend=backend)
    qubits = 2 * int((dx2 + 4 * dz2) * 3 * dx2 + nl1 * ((dx + 4 * dz) * (3 * dx + dm2 / 2) + 2 * dm)
                     + 20 * dm2 * dm2 + 2 * dx2 * dm2)
    cycles = 7.5 * l1time / ps2
    return dict(protocol='Litinski2019 (15-to-1)x(15-to-1)_{%d,%d,%d;%d,%d,%d;%d}' % (dx, dz, dm, dx2, dz2, dm2, nl1),
                eps=float(eps), p_success=float(ps2), qubits=int(qubits), cycles=float(cycles),
                volume=float(qubits * cycles), levels=2, eps_level1=float(pl1))

In [ ]:
# ------------------------------------------------------------------ surface-code p_L calibration (stim)
def noisy_surface_code(d, rounds, basis, p, p_idle=None):
    """Rotated surface-code memory (stim's generated circuit) with the SAME circuit-level noise model and
    time-step convention as the D4 simulation (noisy_d4_lib.NoiseModel / D4SwitchingCircuit rounds):
    X-type ancillas are prepared / measured with RX / MX (no H layers); stim's combined MR is split into a
    measurement step and a reset step, so that one round = reset, 4 CNOT layers, measurement = 6 time
    steps; 2-qubit gates are followed by DEPOLARIZE2(p), resets by a flip, measurements are preceded by a
    flip, and the idle qubits of every time step are depolarized with p_idle."""
    p_idle = p if p_idle is None else p_idle
    base = stim.Circuit.generated('surface_code:rotated_memory_' + basis, distance=d, rounds=rounds).flattened()
    allq, hq = set(), set()
    for ins in base:
        if ins.name in ('R', 'CX', 'M', 'MR', 'H'):
            allq |= {t.value for t in ins.targets_copy()}
        if ins.name == 'H':
            hq |= {t.value for t in ins.targets_copy()}
    out = stim.Circuit()

    def emit(pre, body, post, touched):
        for e in pre:
            out.append(*e)
        for b in body:
            out.append(*b) if isinstance(b, tuple) else out.append(b)
        for e in post:
            out.append(*e)
        idle = sorted(allq - touched)
        if idle and p_idle > 0:
            out.append('DEPOLARIZE1', idle, p_idle)
        out.append('TICK')

    def flush(layer):
        touched, pre, body, post, resets = set(), [], [], [], []
        for ins in layer:
            nm = ins.name
            ts = [t.value for t in ins.targets_copy() if t.is_qubit_target]
            if nm == 'H':
                continue
            if nm in ('R', 'RX'):
                for q in ts:
                    x = q in hq or nm == 'RX'
                    body.append(('RX' if x else 'R', [q]))
                    post.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
            elif nm in ('MR', 'M', 'MX'):
                # one target per instruction keeps stim's measurement-record order
                for q in ts:
                    x = q in hq or nm == 'MX'
                    pre.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
                    body.append(('MX' if x else 'M', [q]))
                    if nm == 'MR':
                        resets.append(q)
            elif nm == 'CX':
                body.append(('CX', ts))
                post.append(('DEPOLARIZE2', ts, p))
            else:
                body.append(ins)
                continue
            touched |= set(ts)
        if not body:
            return                                   # H-only layer: no time step
        emit(pre, body, post, touched)
        if resets:                                   # the reset half of MR: its own time step
            emit([], [('RX' if q in hq else 'R', [q]) for q in resets],
                 [('Z_ERROR' if q in hq else 'X_ERROR', [q], p) for q in resets], set(resets))
    layer = []
    for ins in base:
        if ins.name == 'TICK':
            flush(layer)
            layer = []
        elif ins.name in ('DETECTOR', 'OBSERVABLE_INCLUDE', 'QUBIT_COORDS', 'SHIFT_COORDS'):
            if layer:
                flush(layer)
                layer = []
            out.append(ins)
        else:
            layer.append(ins)
    if layer:
        flush(layer)
    return out


def surface_code_logical_rate(d, p, shots, rounds=None, seed=0):
    """Logical error per round (X + Z memories summed) of the rotated surface code (MWPM)."""
    rounds = rounds or 3 * d
    tot = 0.0
    out = {}
    for basis in ('x', 'z'):
        c = noisy_surface_code(d, rounds, basis, p)
        m = pymatching.Matching.from_detector_error_model(c.detector_error_model(decompose_errors=True))
        det, obs = c.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
        pred = m.decode_batch(det)
        PL = float(np.mean(pred[:, 0] != obs[:, 0]))
        eps = (1 - (1 - 2 * PL) ** (1 / rounds)) / 2 if PL < 0.5 else 0.5
        out[basis] = (PL, eps)
        tot += eps
    return tot, out


def fit_plog(rows):
    """Fit p_L = A (p/p_th)^((d+1)/2) to rows (d, p, p_L) by least squares in log p_L."""
    ds = np.array([r[0] for r in rows], float)
    ps = np.array([r[1] for r in rows], float)
    pl = np.array([r[2] for r in rows], float)
    k = (ds + 1) / 2
    # log pl = log A + k log p - k log pth  -> linear in (log A, log pth)
    A_ = np.stack([np.ones_like(k), -k], axis=1)
    y = np.log(pl) - k * np.log(ps)
    (lA, lpth), *_ = np.linalg.lstsq(A_, y, rcond=None)
    return float(np.exp(lA)), float(np.exp(lpth))


def make_plog(A, pth):
    return lambda p, d: mpmath.mpf(A) * (mpmath.mpf(p) / mpmath.mpf(pth)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ GoSC Sec. 3 layouts with the same model
# the 11 rotations of GoSC Fig. 15 (= msd_lib.fig15_rotations(): columns 5..15 of the 15-to-1 matrix,
# qubit indices 0..4 = qubits 1..5; validated in MSD_15to1_validation M1)
GOSC_ROTATIONS = [(2, 3, 4), (1, 3, 4), (1, 2, 4), (1, 2, 3), (0, 3, 4), (0, 2, 4), (0, 2, 3),
                  (0, 1, 4), (0, 1, 3), (0, 1, 2), (0, 1, 2, 3, 4)]


def gosc_15to1_steps(p, d, plog=plog_gosc, p_in=None, anc_tiles=2, in_channel='pauli'):
    """GoSC Fig. 15 / 17c: qubits 1-4 start in |m> (faulty, error p_in), qubit 5 in |+>; the 11 Z-type
    pi/8 rotations of Fig. 15 (msd_lib.fig15_rotations) are auto-corrected pi/8 rotations of 1 tau = d
    code cycles each; qubits 1-4 are measured in X, qubit 5 is the output.
    Error budget per rotation (our adaptation of Litinski 2019 to the single-distance GoSC layout):
      p1 (P_{pi/2})  = e1 + 0.5 p_L d                        (time-like error of the P x Z_m measurement)
      p2 (P_{-pi/4}) = e2 + 0.5 p_L d + 0.5 anc_tiles p_L d   (space-like errors in the ancilla tiles)
      p3 (P_{pi/4})  = e3
    with (e1, e2, e3) = (p_in/3,)*3 for a raw state with a random Pauli error (in_channel='pauli', GoSC
    Sec. 4 assumption) or (p_in, 0, 0) for a distilled input (in_channel='z', Litinski two-level code);
    every data tile suffers X and Z storage errors 0.5 p_L d per tau."""
    p = mpmath.mpf(p)
    p_in = p if p_in is None else mpmath.mpf(p_in)
    pl = plog(p, d)
    e = (p_in / 3,) * 3 if in_channel == 'pauli' else (p_in, 0, 0)
    S = [('rot', _ax(q), *e) for q in range(4)]               # the four |m> states (0 tau)
    st = ('store', [0.5 * pl * d] * 5, [0.5 * pl * d] * 5)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e[0] + 0.5 * pl * d, e[1] + 0.5 * pl * d + 0.5 * anc_tiles * pl * d, e[2]))
        S.append(st)
    return S


def gosc_15to1(p, d, plog=plog_gosc, backend='ld', **kw):
    """Output error and cost of the GoSC 11-tile 15-to-1 block (Fig. 17c): 11 tiles x 11 tau."""
    ps, eps = run_steps(gosc_15to1_steps(p, d, plog, **kw), out_qubit=4, backend=backend)
    qubits = 11 * 2 * d * d                       # 11 tiles, d^2 data + d^2 measurement qubits each
    cycles = 11 * d / ps
    return dict(protocol='GoSC 15-to-1 (11 tiles x 11 tau), d=%d' % d, eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1, d=d)


def gosc_225to1(p, d1, d2, plog=plog_gosc, move_tiles=5, backend='ld', level1=None):
    """GoSC Fig. 19 (176 tiles, 15 tau): 11 level-1 15-to-1 blocks (distance d1) feed a level-2 15-to-1
    whose 15 rotations (incl. the 4 single-qubit ones) consume distilled states.  Level-1 output error
    enters as a Z-type (P_{pi/2}) error (Litinski two-level code); moving a level-1 state through the
    blue ancilla of Fig. 19 adds 0.5 * move_tiles * p_L(d2) * d2 (our assumption, move_tiles = 5)."""
    ps1, e1 = level1 if level1 is not None else run_steps(gosc_15to1_steps(p, d1, plog), out_qubit=4, backend=backend)
    ps1, e1 = mpmath.mpf(float(ps1)), mpmath.mpf(float(e1))
    pl2 = plog(mpmath.mpf(p), d2)
    mv = 0.5 * move_tiles * pl2 * d2
    S = []
    st = ('store', [0.5 * pl2 * d2] * 5, [0.5 * pl2 * d2] * 5)
    for q in range(4):
        S.append(('rot', _ax(q), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    ps2, eps = run_steps(S, out_qubit=4, backend=backend)
    qubits = 2 * (121 * d1 * d1 + 55 * d2 * d2)   # 11 level-1 blocks of 11 tiles + 55 level-2 tiles
    # GoSC Sec. 3.5: each of the 15 level-2 steps needs one level-1 output; when the level-1 block fails
    # (probability 1 - ps1) that 225-to-1 step is skipped, so 15 steps take 15 / ps1 steps on average;
    # 1 step = d2 cycles; the whole protocol is repeated 1 / ps2 times on average
    cycles = (15 / ps1) * d2 / ps2
    return dict(protocol='GoSC 225-to-1 (176 tiles x 15 tau), d1=%d, d2=%d' % (d1, d2), eps=float(eps),
                p_success=float(ps2), qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles),
                levels=2, d=d2, d1=d1, eps_level1=float(e1))

In [ ]:
print('library loaded')

In [ ]:
import time, itertools
VAL = {}
def record(name, passed, detail=''):
    detail = str(detail).replace('\n', ' ')
    VAL[name] = dict(status='PASS' if passed else 'FAIL', detail=detail)
    print(('PASS ' if passed else 'FAIL ') + name + (f'  | {detail}' if detail else ''))
def info(name, detail):
    VAL[name] = dict(status='INFO', detail=str(detail))
    print('INFO ' + name + '  | ' + str(detail))
QUICK = False
L_MAX_INVARIANCE = 5 if QUICK else 8
rng = np.random.default_rng(SEED)

# 第 1 部 D4 符号と code switching（雑音なし、v4 から引き継ぎ）

## V1. 安定化子の定義と [KZH] Fig. 4 の照合

Fig. 4（$L=9$）から読み取れる事項を、コードから得た構造と照合する。
* 左境界 $\mathcal L_r$ 上：緑・青の $X$ 安定化子（3辺）。赤は無し。
* 上境界 $\mathcal L_b$ 上：赤・緑。左上角：緑（2辺）。
* 下・右境界 $\mathcal L_{rb}$：赤青の積 $XX$。右下角も $XX$（2辺）。
* 境界 $Z$：$\mathcal L_r$ に $Z^r$、$\mathcal L_b$ に $Z^b$、$\mathcal L_{rb}$ に $Z^rZ^b$ と $Z^g$。
* バルクの CZ：$S^r$ は右上 plaquette に CZ$^{g,b}$ 2本、$S^g$ は左上・右下 plaquette に CZ$^{r,b}$ 各1本、$S^b$ は左下 plaquette に CZ$^{r,g}$ 2本。

In [ ]:
lat = D4Lattice(9); L = 9
exp = {'bulk': [('r',), ('g',), ('b',)], 'Lr': [('g',), ('b',)], 'Lb': [('r',), ('g',)], 'TL': [('g',)],
       'Lrb': [('r', 'b')], 'BL': [], 'TR': []}
def vclass(v):
    x, y = v
    if v == (0, 0): return 'BL'
    if v == (0, L): return 'TL'
    if v == (L, L): return 'TR'
    if 0 < x < L and 0 < y < L: return 'bulk'
    if x == 0: return 'Lr'
    if y == L: return 'Lb'
    return 'Lrb'
ok = True
for v in lat.vertices:
    got = [tuple(CNAME[c] for c in combo) for combo in lat.vertex_x_types(v)]
    if sorted(got) != sorted(exp[vclass(v)]):
        ok = False; print('mismatch at', v, got)
record('V1a boundary X-generator types match Fig. 4', ok)
zl = lat.z_checks()
bd = Counter((lab[2], lat.boundary_of_edge(lab[1])) for lab, _ in zl if lab[0] == 'bdZ')
exp_bd = {('r', 'Lr'): L, ('b', 'Lb'): L, ('rb', 'Lrb'): 2 * L, ('g', 'Lrb'): 2 * L}
record('V1b boundary Z checks (Z^r on L_r, Z^b on L_b, Z^rZ^b and Z^g on L_rb)', dict(bd) == exp_bd, dict(bd))
def rel_plaq(v, c):
    """plaquette offsets (relative to v) that carry the CZ dressing of S^c_v"""
    out = []
    for (i, j) in lat.SX(v, c).quad:
        es = {i // 3, j // 3}
        for p in lat.plaquettes:
            if es <= set(lat.plaq_edges(p)):
                out.append((p[0] - v[0], p[1] - v[1]))
    return Counter(out)
v0 = (4, 4)
ok = (rel_plaq(v0, R) == Counter({(0, 0): 2}) and rel_plaq(v0, G) == Counter({(-1, 0): 1, (0, -1): 1})
      and rel_plaq(v0, B) == Counter({(-1, -1): 2}))
record('V1c bulk CZ positions (S^r: upper-right x2, S^g: upper-left + lower-right, S^b: lower-left x2)', ok,
       {c: dict(rel_plaq(v0, c)) for c in (R, G, B)})
colpairs = {c: {tuple(sorted((i % 3, j % 3))) for (i, j) in lat.SX(v0, c).quad} for c in (R, G, B)}
record('V1d CZ colour pairs (S^r: g-b, S^g: r-b, S^b: r-g)', colpairs == {R: {(G, B)}, G: {(R, B)}, B: {(R, G)}})

## V2. 符号空間の整合性（$\mathbb F_2$ 上の厳密判定）

$X$ 型 Clifford 安定化子 $A,B$ について $AB=(-1)^{g(x)}BA$ の $g$ はアフィン関数になる（2次項は相殺）。$C_Z$ 上で $g\equiv 0$ ⇔ 定数項0かつ線形部分が $H_Z$ の行空間に属する。
**負の対照**：FSC（$g$ を除いた2層）に CZ 修飾付きの生成元を使うと、$g$ 量子ビットの $Z$ 安定化子が無いため非可換になるはず → 検出できることを確認。

In [ ]:
for L in range(1, 7):
    cd = CodeData(D4Lattice(L))
    r = code_consistency(cd)
    record(f'V2 D4 L={L}: shifts in C_Z, commuting on C_Z, S^2=1, k=1',
           r['shift_in_CZ'] and not r['noncommuting_pairs'] and not r['bad_squares'] and r['k'] == 1 and r['shifts_independent'],
           {k: (len(v) if isinstance(v, list) else v) for k, v in r.items()})
    rf = code_consistency(CodeData(D4Lattice(L), (R, B), dressed=False))
    record(f'V2 FSC L={L}: CSS code, k=1', rf['shift_in_CZ'] and not rf['noncommuting_pairs'] and rf['k'] == 1)
neg = code_consistency(CodeData(D4Lattice(3), (R, B), dressed=True))
record('V2 negative control: dressed generators without g-qubit Z checks are detected as non-commuting',
       len(neg['noncommuting_pairs']) > 0, f"{len(neg['noncommuting_pairs'])} non-commuting pairs")

## V3. 独立実装（$L=1$、12量子ビット密ベクトル）

埋め込みの D4 ライブラリの関数を一切使わず、式(2)(3)(7)(8) を直接 4096 次元のベクトル演算として実装する。$Z$ 部分空間の計算基底を列挙し、その上で $S|\psi\rangle=|\psi\rangle$ の解空間を SVD で求め、論理 $Z$ 演算子を重み≤2 の $Z$ 型 Pauli から探索し、$U=WV$ の論理行列を求める。

In [ ]:
n1 = 12; N1 = 2 ** n1; idx1 = np.arange(N1)
qq = lambda e, c: 3 * e + c        # edges: 0 bottom h(0,0), 1 top h(0,1), 2 left v(0,0), 3 right v(1,0)
bit1 = lambda i, k: (i >> (n1 - 1 - k)) & 1
def Zop(qs):
    par = np.zeros(N1, dtype=int)
    for k in qs: par ^= bit1(idx1, k)
    return lambda v: (1 - 2 * par) * v
def Xop(qs):
    m = 0
    for k in qs: m |= 1 << (n1 - 1 - k)
    return lambda v: v[idx1 ^ m]
def CZop(pairs):
    par = np.zeros(N1, dtype=int)
    for a, b in pairs: par ^= bit1(idx1, a) & bit1(idx1, b)
    return lambda v: (1 - 2 * par) * v
def compose(*fs):
    def h(v):
        for f in reversed(fs): v = f(v)
        return v
    return h
Zchecks = [[qq(e, c) for e in range(4)] for c in range(3)]
Zchecks += [[qq(2, 0)], [qq(1, 2)], [qq(0, 0), qq(0, 2)], [qq(3, 0), qq(3, 2)], [qq(0, 1)], [qq(3, 1)]]
S_rb = compose(Xop([qq(0, 0), qq(3, 0)]), Xop([qq(0, 2), qq(3, 2)]))           # vertex (1,0) on L_rb
S_g = compose(Xop([qq(2, 1), qq(1, 1)]), CZop([(qq(2, 0), qq(1, 2))]))          # vertex (0,1), v = vertex 2 of p
okZ = np.ones(N1, bool)
for zc in Zchecks:
    par = np.zeros(N1, dtype=int)
    for k in zc: par ^= bit1(idx1, k)
    okZ &= par == 0
basis = np.nonzero(okZ)[0]
Bm = np.zeros((N1, len(basis)), complex); Bm[basis, np.arange(len(basis))] = 1
rows_ = []
for S in (S_rb, S_g):
    M = np.stack([S(Bm[:, j]) for j in range(Bm.shape[1])], 1)
    assert np.allclose(M[~okZ], 0)
    rows_.append(Bm.conj().T @ M - np.eye(len(basis)))
u_, s_, vh_ = np.linalg.svd(np.concatenate(rows_))
code1 = Bm @ vh_[np.sum(s_ > 1e-9):].conj().T
record('V3a independent L=1: code dimension = 2', code1.shape[1] == 2, f'dim C_Z={len(basis)}')
P1 = code1 @ code1.conj().T
found = None
for w in (1, 2):
    for qs in itertools.combinations(range(n1), w):
        M = np.stack([Zop(qs)(code1[:, j]) for j in range(2)], 1)
        if np.allclose(P1 @ M, M):
            Mc = code1.conj().T @ M
            if not np.allclose(Mc, Mc[0, 0] * np.eye(2)):
                found = (qs, Mc); break
    if found: break
ev, evec = np.linalg.eigh(found[1]); logical1 = code1 @ evec[:, ::-1]
def CNOT1(c, t):
    return lambda v: np.where(bit1(idx1, c) == 1, v[idx1 ^ (1 << (n1 - 1 - t))], v)
V1 = compose(*[CNOT1(qq(e, c), qq(e, 1)) for e in range(4) for c in (0, 2)])
def phase1(terms):
    ph = np.zeros(N1)
    for c, qs in terms:
        m = np.ones(N1, dtype=int)
        for k in qs: m &= bit1(idx1, k)
        ph = ph + c * m
    return lambda v: np.exp(1j * np.pi / 4 * ph) * v
W1 = phase1([(2, (qq(0, 0), qq(3, 2))), (-2, (qq(2, 0), qq(1, 2)))] + [(-1, (qq(e, 0),)) for e in (0, 3)])
U1 = compose(W1, V1)
UL = np.stack([U1(logical1[:, j]) for j in range(2)], 1)
Um = logical1.conj().T @ UL; Um = Um / Um[0, 0]
record('V3b independent L=1: U preserves the code space', np.allclose(P1 @ UL, UL))
record('V3c independent L=1: logical action of U = T^dag (Z-basis defined by the trivial gauge field)',
       np.allclose(Um, np.diag([1, np.exp(-1j * np.pi / 4)])), np.round(Um, 4))

## V4. 論理演算子と距離（MILP を全探索で照合）
$d_Z$：$Su=0$, $t_{\log}\cdot u=1$ を満たす最小重み $u$。$d_X^{\rm shift}$：$H_Z t=0$, $z_{\log}\cdot t=1$ を満たす最小重み $t$。frozen 量子ビットを含む最小解は無い（frozen 上の $Z$ は安定化子、$X$ は検出される）ので frozen を除いて全探索する。

In [ ]:
def brute_min(Hrows, lvec, cols, wmax):
    Hc = Hrows[:, cols].astype(np.int64); lc = lvec[cols].astype(np.int64)
    for w in range(1, wmax + 1):
        for sup in itertools.combinations(range(len(cols)), w):
            sup = list(sup)
            if (lc[sup].sum() % 2 == 1) and not (Hc[:, sup].sum(axis=1) % 2).any():
                return w
    return None
for L, kind in [(2, 'Z'), (3, 'Z'), (2, 'X')]:
    lat = D4Lattice(L); fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1: fz |= rows
    for name, cd in [('FSC', CodeData(lat, (R, B), dressed=False)), ('D4', CodeData(lat))]:
        cols = [c for c in cd.cols if c not in fz]
        dz, dx, _, _ = distances(cd)
        if kind == 'Z':
            bf = brute_min(cd.S, cd.t_log, cols, dz)
            record(f'V4 {name} L={L}: brute-force d_Z = MILP d_Z = L', bf == dz == L, (bf, dz))
        else:
            if len(cols) > 50:
                continue
            bf = brute_min(cd.HZ, cd.z_log, cols, dx)
            record(f'V4 {name} L={L}: brute-force X-shift distance = MILP = L+3', bf == dx == L + 3, (bf, dx))

## V5. Knill–Laflamme 条件（$L=2$）
$\langle i|E|j\rangle = c_E\delta_{ij}$ を、論理基底 $|0_L\rangle,|1_L\rangle$（各 $2^{11}$ 項の疎ベクトル）と全ての weight≤2 Pauli $E$ について直接検査する（約1分）。D4 相の真の距離が $L=2$ で 2 であることを確認する（$L\ge3$ は**未検証**）。

In [ ]:
t0 = time.time()
lat2 = D4Lattice(2); d42 = CodeData(lat2)
kb = [coset_state(d42, set()), coset_state(d42, set(np.nonzero(d42.t_log)[0]))]
def apply_pauli(st, xs, zs):
    xs = frozenset(xs); out = {}
    for x, v in st.a.items():
        out[x ^ xs] = v * (-1) ** len(x & zs)
    return SparseState(out)
viol = {}
for w in (1, 2):
    viol[w] = 0
    for qs in itertools.combinations(range(lat2.n), w):
        for ps in itertools.product('XYZ', repeat=w):
            xs = [a for a, p in zip(qs, ps) if p in 'XY']; zs = {a for a, p in zip(qs, ps) if p in 'YZ'}
            M = np.array([[kb[i].overlap(apply_pauli(kb[j], xs, zs)) for j in range(2)] for i in range(2)])
            if abs(M[0, 1]) > 1e-9 or abs(M[1, 0]) > 1e-9 or abs(M[0, 0] - M[1, 1]) > 1e-9:
                viol[w] += 1
record('V5 D4 L=2 Knill-Laflamme: all weight-1 Paulis correctable-detectable, some weight-2 logical (distance = 2)',
       viol[1] == 0 and viol[2] > 0, f'violations {viol}, {time.time() - t0:.0f}s')

## V5b. D4 相の Knill–Laflamme 距離 $=L$ の厳密証明（$L\le 5$、`FULL_KL=True` で $L=6$）

$E=X^tZ^u$ について：(i) $t\notin C_Z$ なら $E$ は符号空間を直交空間へ写す（検出可能）。(ii) $t$ が論理類なら $|t|\ge L+3$（V4）。(iii) $t\in\mathrm{span}(S)$ なら $X^t=S_cF_c$（$S_c$ は安定化子の積、$F_c=(-1)^{f_c}$ は対角）なので $PEP=PF_cZ^uP$。$F_c$ を $Z$ 型 Pauli $Z^v$（$v\subseteq Q_f=\mathrm{supp}f_c$）に展開すると、KL 条件が破れるには $v+u$ が論理 $Z$ でなければならない。よって
$$|E|\ \ge\ |t| + \min_{\ell:\ \text{logical }Z}|\ell\setminus(Q_f\cup t)|=:LB_t .$$
重み $\le L-1$ の全ての $t\in\mathrm{span}(S)$ を MILP（no-good cut 付き）で**厳密に列挙**し、$\min_t LB_t\ge L$ を確認する。$t=0$ は $d_Z=L$ で扱える。重み $L$ の論理 $Z$ があるので距離はちょうど $L$。

In [ ]:
FULL_KL = False
for L in range(3, 7 if FULL_KL else 6):
    t0 = time.time()
    P_ = D4Protocol(L)
    lb, ts, det, stop = kl_distance_lower_bound(P_.d4, L - 1, P_.frozen)
    dz, dx, _, _ = distances(P_.d4)
    # stop is 'infeasible' or 'min weight > wmax': the enumeration ended because the solver
    # proved there is no further t (any other solver status raises inside _milp_binary)
    ok = stop in ('infeasible', 'min weight > wmax') and (lb is None or lb >= L) and dz == L and dx >= L
    record(f'V5b L={L}: every Pauli error of weight <= L-1 satisfies Knill-Laflamme -> D4-phase distance = L',
           ok, f'{len(ts)} stabilizer shifts of weight <= {L-1} enumerated (stop: {stop}); min LB_t = {lb}; d_Z = {dz}; X-shift = {dx}; {time.time()-t0:.0f}s')

## V6. transversal $U=WV$（式(5)(8)）

各生成元 $S$ について、[KZH] SM の式（permuting stabilizers）が予言する像 $S'$（$S^r\to S^rS^g$, $S^g\to S^g$, $S^b\to S^gS^b$, $\mathcal L_{rb}$ 上の $S^rS^b\to S^rS^b$）と $USU^\dagger$ が $C_Z$ 上で一致することを、**位相差が依存する量子ビットへの $C_Z$ の射影を厳密に全列挙**して確認する。
負の対照：(i) 境界の $T^\dagger$ を除く、(ii) CS と CS$^\dagger$ を入れ替える、(iii) 境界を $T$ にする、(iv) 両方 CS にする → 失敗するか論理作用が変わるはず。

In [ ]:
def check_U(L, cs=(1, -1), bt=-1):
    lat = D4Lattice(L); cd = CodeData(lat); Wop = W_operator(lat, cs, bt)
    fails, mx = [], 0
    for lab, op in cd.xgens:
        ok, n = check_conjugation(cd, Wop, op, U_target(lat, lab))
        mx = max(mx, n)
        if not ok: fails.append(lab)
    return fails, mx, logical_phases(cd, Wop), cd
for L in range(1, 7):
    fails, mx, ph, cd = check_U(L)
    # V maps C_Z into C_Z
    Bz = CZ_basis(cd)
    VB = np.array([cd.lat.vec(V_map(cd.lat, frozenset(np.nonzero(b)[0]))) for b in Bz])
    vz = not ((cd.HZ.astype(int) @ VB.T.astype(int)) % 2).any()
    record(f'V6 L={L}: U S U^dag = S\' on C_Z for all {len(cd.xgens)} generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)',
           not fails and vz and ph == [0, 7], f'max local configs {mx}, phases {ph}')
for name, cs, bt in [('no boundary T^dag', (1, -1), 0), ('CS<->CS^dag swapped', (-1, 1), -1),
                     ('boundary T instead of T^dag', (1, -1), 1), ('CS,CS (no dagger)', (1, 1), -1)]:
    fails, mx, ph, _ = check_U(4, cs, bt)
    record(f'V6 negative control L=4 [{name}] is rejected or changes the logical action', bool(fails) or ph != [0, 7],
           f'{len(fails)} failing generators, phases {ph}')
fails, mx, ph, _ = check_U(4, (-1, 1), 1)
record('V6 conjugate convention (CS^dag,CS, boundary T) gives a valid logical T (consistency)', not fails and ph == [0, 1], ph)

## V7. プロトコル全体の厳密シミュレーション（$L=1,2$）

[KZH] の手順を疎状態ベクトル（計算基底の辞書）でそのまま実行する。
1. FSC の論理状態 $\alpha|0_L\rangle+\beta|1_L\rangle$（ランダム）、$g$ は $|0\rangle$。全 $Z$ 安定化子が $+1$ であることを確認。
2. D4 の全 $X$ 型生成元を射影測定（ボルン則で分岐をサンプル）。2回目の測定で結果が再現すること（ノイズなしで1ラウンドで十分）を確認。
3. $U=WV$。
4. 全 $Z^g_e$ を測定し、$\delta\kappa=a_g$ を解いて $\prod_{v\in\kappa}S^g_{X;v}$ を適用。
5. $m_v=1$ の各頂点に $\mathcal L_{rb}$ から $Z^rZ^b$ 文字列（Pauli フレーム補正）。
6. 目標状態 $\alpha|0_L\rangle+e^{-i\pi/4}\beta|1_L\rangle$ との忠実度。

負の対照：補正5を外す／補正4の CZ 部分を外す（$X^g$ のみ）／境界 $T^\dagger$ を外す。

In [ ]:
def run_protocol(L, alpha, beta, rng, skip_pauli=False, sg_x_only=False, W_bt=-1):
    lat = D4Lattice(L); fsc = CodeData(lat, (R, B), dressed=False); d4 = CodeData(lat)
    zero = coset_state(fsc, set()); one = coset_state(fsc, set(np.nonzero(fsc.t_log)[0]))
    psi = SparseState({k: alpha * v for k, v in zero.a.items()})
    for k, v in one.a.items(): psi.a[k] = psi.a.get(k, 0) + beta * v
    for lab, rows in lat.z_checks():
        assert abs(psi.expect(PSOp(lin=rows)) - 1) < 1e-9
    m = {}
    for lab, op in d4.xgens:
        bit, psi, p = psi.measure_op(op, rng); m[lab] = bit
    repeat_ok = all(abs(psi.expect(op) - (-1) ** m[lab]) < 1e-9 for lab, op in d4.xgens)
    zero_ok = all(abs(psi.expect(PSOp(lin=rows)) - 1) < 1e-9 for lab, rows in lat.z_checks())
    psi = psi.apply_perm(lambda x: V_map(lat, x)).apply_diag(W_operator(lat, (1, -1), W_bt))
    ag = set()
    for e in range(lat.E):
        bit, psi, p = psi.measure_z(lat.q(e, G), rng)
        if bit: ag.add(e)
    ggens = [(lab, op) for lab, op in d4.xgens if lab[2] == 'g']
    Sg = np.array([lat.vec(op.s) for _, op in ggens])
    kappa = f2_solve(Sg, lat.vec({lat.q(e, G) for e in ag}))
    for i in np.nonzero(kappa)[0]:
        op = ggens[i][1]
        psi = psi.apply_psop(PSOp(s=op.s) if sg_x_only else op)
    gclean = all(all(lat.q(e, G) not in x for e in range(lat.E)) for x in psi.a)
    if not skip_pauli:
        for lab, op in ggens:
            if m[lab]:
                x0, y0 = lab[1]
                qs = {lat.q(lat.eidx[('v', x0, y)], c) for y in range(y0) for c in (R, B)}
                psi = psi.apply_diag(DiagPhase([(4, (q,)) for q in qs]))
    target = SparseState({k: alpha * v for k, v in zero.a.items()})
    for k, v in one.a.items(): target.a[k] = target.a.get(k, 0) + beta * OMEGA ** 7 * v
    F = abs(target.overlap(psi)) ** 2 / (target.norm2() * psi.norm2())
    mg = tuple(m[lab] for lab, _ in ggens)
    return dict(F=F, repeat_ok=repeat_ok, zero_ok=zero_ok, g_cleared=gclean, m_g=mg,
                m_other_nontrivial=any(m[lab] for lab in m if lab[2] != 'g'), kappa_weight=int(kappa.sum()),
                n_g=len(ggens))
def rand_state(rng):
    a, b = rng.normal(size=2) + 1j * rng.normal(size=2); nrm = np.hypot(abs(a), abs(b)); return a / nrm, b / nrm
stats = {}
for L, ntr in [(1, 60), (2, 200 if not QUICK else 40)]:
    t0 = time.time(); res = [run_protocol(L, *rand_state(rng), rng) for _ in range(ntr)]
    Fmin = min(r['F'] for r in res)
    record(f'V7 L={L}: exact protocol, {ntr} random inputs/branches -> fidelity 1 with T^dag action',
           Fmin > 1 - 1e-9 and all(r['repeat_ok'] and r['zero_ok'] and r['g_cleared'] for r in res)
           and not any(r['m_other_nontrivial'] for r in res), f'min F = {Fmin:.12f}, {time.time() - t0:.0f}s')
    stats[L] = res
# basis inputs (logical information preserved for |0>, |1>, |+>, |+i>)
for name, (a, b) in {'|0>': (1, 0), '|1>': (0, 1), '|+>': (2 ** -.5, 2 ** -.5), '|+i>': (2 ** -.5, 1j * 2 ** -.5)}.items():
    Fs = [run_protocol(2, a, b, rng)['F'] for _ in range(10)]
    record(f'V7 L=2 input {name}: output = T^dag {name}', min(Fs) > 1 - 1e-9, f'min F={min(Fs):.12f}')
res2 = stats[2]; ng = res2[0]['n_g']
cm = Counter(r['m_g'] for r in res2); ck = Counter(r['kappa_weight'] for r in res2)
from scipy.stats import chisquare
p_m = chisquare([cm.get(t, 0) for t in itertools.product((0, 1), repeat=ng)]).pvalue
record(f'V7 L=2: S^g outcomes m are random (all {2**ng} patterns occur; chi^2 test vs uniform p={p_m:.2f})',
       len(cm) == 2 ** ng and p_m > 1e-3, dict(sorted(cm.items())))
mean_k = np.mean([r['kappa_weight'] for r in res2])
record('V7 L=2: switch-out correction |kappa| has mean ~ n_g/2 (supports expected CZ count = worst/2)',
       abs(mean_k - ng / 2) < 0.35, f'mean |kappa| = {mean_k:.2f}, n_g = {ng}, hist {dict(sorted(ck.items()))}')
negs = {'without Z^rZ^b Pauli-frame correction': dict(skip_pauli=True),
        'S^g correction without its CZ part': dict(sg_x_only=True),
        'W without boundary T^dag': dict(W_bt=0)}
for name, kw in negs.items():
    Fs = [run_protocol(2, *rand_state(rng), rng, **kw)['F'] for _ in range(60)]
    record(f'V7 negative control [{name}] fails for some branches', min(Fs) < 1 - 1e-6, f'min F = {min(Fs):.4f}')

## V7b. プロトコル（回路）と位相公式 $F(a)$ の照合（$L=3,4$）

V8 の局所証明は配置ごとの位相の式 $F(a)$ に基づく。その式自体を、$L=1,2$ の符号状態シミュレーション（V7）とは別に、**実際の測定・$U$・補正の回路**を単一の配置入力 $|a\rangle|0\rangle_g$ に作用させて $L=3,4$ で照合する（ノイズなし最小版：$S^g_X$ のみ測定）。同じ測定分岐（$m$ と $Z^g$ の結果を固定）で 2 つの配置 $a_1,a_2$（異なる論理剰余類を含む）の出力振幅比が $\omega^{F(a_2)-F(a_1)}$ に一致すること＝回路が配置ごとに $F$ の位相を与えること、を確認する。プロトコルは線形なので、これは符号状態に対する作用を決める。

In [ ]:
def protocol_on_config(lat, d4, a, rng, forced_m=None, forced_z=None):
    psi = SparseState({frozenset(a): 1.0})
    ggens = [(lab, op) for lab, op in d4.xgens if lab[2] == 'g']
    m = []
    for i, (lab, op) in enumerate(ggens):
        bit, psi, p = psi.measure_op(op, rng, forced=None if forced_m is None else forced_m[i]); m.append(bit)
    psi = psi.apply_perm(lambda x: V_map(lat, x)).apply_diag(W_operator(lat))
    zb = []
    for j, e in enumerate(range(lat.E)):
        bit, psi, p = psi.measure_z(lat.q(e, G), rng, forced=None if forced_z is None else forced_z[j]); zb.append(bit)
        if p < 1e-12: return None, m, zb
    ag = {lat.q(e, G) for e, b in zip(range(lat.E), zb) if b}
    kappa = f2_solve(np.array([lat.vec(op.s) for _, op in ggens]), lat.vec(ag))
    for i in np.nonzero(kappa)[0]:
        psi = psi.apply_psop(ggens[i][1])
    for (lab, op), bit in zip(ggens, m):
        if bit:
            x0, y0 = lab[1]
            psi = psi.apply_diag(DiagPhase([(4, (lat.q(lat.eidx[('v', x0, y)], c),)) for y in range(y0) for c in (R, B)]))
    assert set(psi.a) == {frozenset(a)}, 'output not proportional to the input configuration'
    return psi.a[frozenset(a)], m, zb
for L, ntr in [(3, 6), (4, 2 if QUICK else 4)]:
    t0 = time.time()
    lat = D4Lattice(L); d4 = CodeData(lat); fsc = CodeData(lat, (R, B), dressed=False)
    gverts = [lab[1] for lab, op in d4.xgens if lab[2] == 'g']; Wop = W_operator(lat)
    def rand_cfg(coset):
        x = set()
        for i in np.nonzero(rng.integers(0, 2, len(fsc.xgens)))[0]: x ^= set(fsc.xgens[i][1].s)
        if coset: x ^= set(np.nonzero(fsc.t_log)[0])
        return frozenset(x)
    ok, nb = True, 0
    for _ in range(ntr):
        a1 = rand_cfg(0)
        A1, m, zb = protocol_on_config(lat, d4, a1, rng)
        for a2 in [rand_cfg(0), rand_cfg(1), rand_cfg(1)]:
            A2, _, _ = protocol_on_config(lat, d4, a2, rng, forced_m=m, forced_z=zb)
            if A2 is None: continue
            nb += 1
            pred = OMEGA ** ((protocol_phase(lat, Wop, a2, gverts) - protocol_phase(lat, Wop, a1, gverts)) % 8)
            ok &= abs(A2 / A1 - pred) < 1e-9
    record(f'V7b L={L}: protocol circuit on configuration inputs reproduces the phase formula F(a) (incl. T^dag between cosets)',
           ok and nb > 0, f'{nb} same-branch pairs, {time.time()-t0:.0f}s')

## V8. 大きな $L$ でのプロトコルの正しさ（局所厳密証明）

V7 の手順を解析すると、FSC の配置 $a=(a_r,a_b)$ が受け取る位相は
$$F(a) = w(a) + 4\sum_{v} c'_v(a)\, f^g_v(a)\pmod 8,\qquad \delta c' = a_r + a_b,\; c'|_{\mathcal L_{rb}}=0$$
（$w$ は $W$ の位相指数、$f^g_v$ は $S^g_{X;v}$ の CZ$^{r,b}$ 部分）となる（$m$ に依存する部分は Pauli 補正で消え、$\kappa$ に依存する部分は $S^g$ 補正で消える）。この式は V7 の厳密シミュレーションで $L\le2$ について裏付けられている。
プロトコルが論理 $T^\dagger$ を実装する ⇔ $F$ が論理剰余類上で一定で $F(1)-F(0)=7$。
FSC の各生成元シフト $t$ について $F(a+t)=F(a)$ を、$F(a+t)-F(a)$ が依存する量子ビット（と $c'_{w_0}(a)$ という $\mathbb F_2$ 線形汎関数）への $C_{\rm FSC}$ の**厳密な射影**の全要素で確認する。

In [ ]:
for L in range(1, L_MAX_INVARIANCE + 1):
    t0 = time.time()
    lat = D4Lattice(L); fsc = CodeData(lat, (R, B), dressed=False); d4 = CodeData(lat)
    gverts = [lab[1] for lab, op in d4.xgens if lab[2] == 'g']; Wop = W_operator(lat)
    fails, mx = 0, 0
    for lab, op in fsc.xgens:
        ok, nloc = check_protocol_invariance(lat, fsc, Wop, gverts, op); fails += (not ok); mx = max(mx, nloc)
    F0 = protocol_phase(lat, Wop, frozenset(), gverts)
    F1 = protocol_phase(lat, Wop, frozenset(np.nonzero(fsc.t_log)[0]), gverts)
    record(f'V8 L={L}: F invariant under all {len(fsc.xgens)} FSC generators; F(1)-F(0)=7 (T^dag)',
           fails == 0 and (F1 - F0) % 8 == 7, f'max local configs {mx}, F0={F0}, F1={F1}, {time.time() - t0:.1f}s')

## V9. 測定ガジェット（制御-$S$ ＋ $X$ 測定）の正しさ

$L=3$ の全 $X$ 型生成元（frozen を除かない版：最も多くの CCZ を含む）について、ガジェット回路をその台の上の密ベクトルで実行し、補助を $|\pm\rangle$ に射影した後のデータ状態が $(1\pm S)|\psi\rangle/2$ と一致することを確認する（$|\psi\rangle$ はランダム）。frozen 量子ビットを除いた版は、frozen を $|0\rangle$ とした状態で同じ検査を行う。

In [ ]:
def gadget_dense_check(P, lab, rng, frozen_zero=False):
    gad = P.x_gadget(lab, True)
    op = PSOp()
    for ch in lab[2]:
        op = op @ P.lat.SX(lab[1], CNAME.index(ch))
    anc = gad[0][1]
    qs = sorted((gadget_support(gad) | op.support()) - {anc}) + [anc]
    k = len(qs); pos = {q: i for i, q in enumerate(qs)}; idx = np.arange(2 ** k)
    b = lambda q: (idx >> (k - 1 - pos[q])) & 1
    psi = rng.normal(size=2 ** k) + 1j * rng.normal(size=2 ** k)
    psi = np.where(b(anc) == 0, psi, 0)
    if frozen_zero:
        for q in qs:
            if q in P.frozen: psi = np.where(b(q) == 0, psi, 0)
    psi /= np.linalg.norm(psi)
    st = (psi + psi[idx ^ (1 << (k - 1 - pos[anc]))]) / np.sqrt(2)          # ancilla |0> -> |+>
    for g in gad[1:-1]:
        if g[0] == 'CNOT':
            st = np.where(b(g[1]) == 1, st[idx ^ (1 << (k - 1 - pos[g[2]]))], st)
        elif g[0] == 'CCZ':
            st = np.where(b(g[1]) & b(g[2]) & b(g[3]), -st, st)
        else:
            raise ValueError(g)
    Spsi = np.zeros_like(psi)
    for i in np.nonzero(np.abs(psi) > 0)[0]:
        x = frozenset(q for q in qs if q != anc and (i >> (k - 1 - pos[q])) & 1)
        y, sb = op.apply(x)
        j = sum(1 << (k - 1 - pos[q]) for q in y)
        Spsi[j] += (-1) ** sb * psi[i]
    ok = True
    for sgn in (1, -1):
        a0 = np.where(b(anc) == 0, st, 0); a1 = np.where(b(anc) == 1, st, 0)
        post = (a0 + sgn * a1[idx ^ (1 << (k - 1 - pos[anc]))]) / np.sqrt(2)
        ok &= np.allclose(post, (psi + sgn * Spsi) / 2)
    return ok
Pu = D4Protocol(3, prune_frozen=False); Pp = D4Protocol(3, prune_frozen=True)
okU = all(gadget_dense_check(Pu, lab, rng) for lab, _ in Pu.d4.xgens)
okP = all(gadget_dense_check(Pp, lab, rng, frozen_zero=True) for lab, _ in Pp.d4.xgens)
record('V9 L=3: every X-type gadget (unpruned) implements the projective measurement of its Clifford stabilizer', okU)
record('V9 L=3: pruned gadgets (frozen qubits removed) agree when frozen qubits are |0>', okP)

## V10. スケジュールの妥当性（独立な行列計算）

可換クラス・スケジュールの上界は「同じクラスの異なるガジェットのゲートは全て可換」という条件に依存する（その場合、同時実行は逐次測定と等価）。役割による判定とは独立に、共有量子ビットを持つゲート対の行列（≤6量子ビット）を作って交換子を計算する。$U$ のゲート同士の可換性も同様に確認する（$U$ を1層に詰めてよい根拠）。

In [ ]:
GMAT = {'CNOT': np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], complex),
        'CZ': np.diag([1, 1, 1, -1]).astype(complex), 'CS': np.diag([1, 1, 1, 1j]), 'CSDG': np.diag([1, 1, 1, -1j]),
        'CCZ': np.diag([1] * 7 + [-1]).astype(complex), 'T': np.diag([1, np.exp(1j * np.pi / 4)]),
        'TDG': np.diag([1, np.exp(-1j * np.pi / 4)])}
def embed(g, qs):
    k = len(qs); pos = {q: i for i, q in enumerate(qs)}; G_ = GMAT[g[0]]; a = len(g) - 1
    M = np.zeros((2 ** k, 2 ** k), complex)
    for col in range(2 ** k):
        bits = [(col >> (k - 1 - i)) & 1 for i in range(k)]
        sub = sum(bits[pos[q]] << (a - 1 - t) for t, q in enumerate(g[1:]))
        for out in range(2 ** a):
            amp = G_[out, sub]
            if amp == 0: continue
            nb = bits[:]
            for t, q in enumerate(g[1:]): nb[pos[q]] = (out >> (a - 1 - t)) & 1
            M[sum(bb << (k - 1 - i) for i, bb in enumerate(nb)), col] += amp
    return M
_cache = {}
def commute(g1, g2):
    key = (g1, g2)
    if key not in _cache:
        qs = sorted(set(g1[1:]) | set(g2[1:])); A, Bm_ = embed(g1, qs), embed(g2, qs)
        _cache[key] = np.allclose(A @ Bm_, Bm_ @ A)
    return _cache[key]
P3 = D4Protocol(3)
allok, npairs = True, 0
for code_ in ['D4', 'FSC']:
    for cls_ in commuting_classes(P3.round_gadgets(code_)):
        ents = [[g for g in gad if g[0] in GMAT] for gad in cls_]
        byq = defaultdict(list)
        for gi, gl in enumerate(ents):
            for g in gl:
                for q in g[1:]: byq[q].append((gi, g))
        for q, lst in byq.items():
            for (i1, g1), (i2, g2) in itertools.combinations(lst, 2):
                if i1 != i2:
                    npairs += 1
                    if not commute(g1, g2): allok = False
record('V10 L=3: all gates of different gadgets within each commuting class commute (matrix check)', allok, f'{npairs} gate pairs')
Ug = P3.U_gates(); okUc = True
for g1, g2 in itertools.combinations(Ug, 2):
    if set(g1[1:]) & set(g2[1:]) and not commute(g1, g2): okUc = False
record('V10 L=3: all gates of U = W V mutually commute (U can be layered freely)', okUc, f'{len(Ug)} gates')
# Clifford+T decompositions used in the 'clifford_t' gate-set model
def circuit_matrix(gates, k):
    M = np.eye(2 ** k, dtype=complex)
    for g in gates: M = embed(g, list(range(k))) @ M
    return M
record('V10 CCZ = 7-T decomposition (matrix)', np.allclose(circuit_matrix(ccz_to_clifford_t(0, 1, 2), 3), GMAT['CCZ']))
record('V10 CS = 3-T decomposition (matrix)', np.allclose(circuit_matrix(cs_to_clifford_t(0, 1), 2), GMAT['CS']))
record('V10 CS^dag decomposition (matrix)', np.allclose(circuit_matrix(csdg_to_clifford_t(0, 1), 2), GMAT['CSDG']))

## V11. 資源計数の独立照合（幾何からの直接公式）
格子の幾何から手で導いた式と、ガジェット生成コードの計数を比較する。
* データ（frozen 除く）：辺数 $2L(L+1)$ × 3色 − frozen（$\mathcal L_r$ の $r$: $L$、$\mathcal L_b$ の $b$: $L$、$\mathcal L_{rb}$ の $g$: $2L$）$=6L^2+2L$。
* $X$ 型生成元：バルク $3(L-1)^2$、$\mathcal L_r$・$\mathcal L_b$ 各 $2(L-1)$、左上角1、$\mathcal L_{rb}$ $2L-1$ → 合計 $3L^2-2L+1$… などを直接列挙と比較。
* CCZ／ラウンド（frozen 除去後）：CZ 対のうち両端が非 frozen のもの。

In [ ]:
ok = True; rows = []
for L in range(2, 10):
    P = D4Protocol(L); qs = P.qubit_sets()
    E = 2 * L * (L + 1)
    data_formula = 3 * E - (L + L + 2 * L)
    xg_formula = 3 * (L - 1) ** 2 + 2 * (L - 1) + 2 * (L - 1) + 1 + (2 * L - 1)
    zc_formula = 3 * L * L + 2 * L        # plaquettes (3 colours) + Z^rZ^b on the 2L L_rb edges
    ccz_direct = sum(1 for lab, _ in P.d4.xgens for ch in lab[2]
                     for p in P.lat.cz_pairs(lab[1], CNAME.index(ch)) if p[0] not in P.frozen and p[1] not in P.frozen)
    ccz_code = sum(1 for gd in P.round_gadgets('D4') for g in gd if g[0] == 'CCZ')
    r = dict(L=L, data=len(qs['d4_data']), data_formula=data_formula, xgens=len(P.d4.xgens), xg_formula=xg_formula,
             anc=len(qs['d4_anc']), anc_formula=xg_formula + zc_formula, ccz=ccz_code, ccz_direct=ccz_direct,
             ccz_closed_form=6 * (L - 1) ** 2)
    rows.append(r)
    ok &= r['data'] == data_formula and r['xgens'] == xg_formula and r['anc'] == r['anc_formula'] and ccz_code == ccz_direct == r['ccz_closed_form']
record('V11 qubit/generator/CCZ counts agree with direct geometric formulas (L=2..9)', ok)
pd.DataFrame(rows)

# 第 2 部 15-to-1 と GoSC の配置（雑音なし、v4 から引き継ぎ）

## M1. 行列と回路の整合
Fig. 15 の各列（qubit 1..5 の作用）を図から手で転記した表 `FIG15_TRANSCRIBED` と、式(1) から機械的に生成した回転を比較する。

In [ ]:
FIG15_TRANSCRIBED = ['11ZZZ', '1Z1ZZ', '1ZZ1Z', '1ZZZ1', 'Z11ZZ', 'Z1Z1Z', 'Z1ZZ1', 'ZZ11Z', 'ZZ1Z1', 'ZZZ11', 'ZZZZZ']
derived = [''.join('Z' if i in r else '1' for i in range(5)) for r in fig15_rotations()]
record('M1a triorthogonality of G (eq. 2)', all(triorthogonal(G_TRI)), triorthogonal(G_TRI))
# G~ is not literally row-equivalent to G (the row spaces differ): the elimination in [Lit] also reorders
# columns.  Equivalence up to row operations + column permutation: the 16 columns of G are exactly the affine
# hyperplane {c : c_0 = 1}; we check that the 16 columns of G~ are exactly an affine hyperplane {c : f.c = 1}.
def affine_hyperplane_functional(Mx):
    for f in itertools.product((0, 1), repeat=Mx.shape[0]):
        if len({tuple(c) for c in Mx.T}) == 2 ** (Mx.shape[0] - 1) and all(np.dot(f, c) % 2 == 1 for c in Mx.T):
            return f
    return None
fG, fGt = affine_hyperplane_functional(G_TRI), affine_hyperplane_functional(G_TILDE)
record('M1b G~ (eq. 4) is triorthogonal and equivalent to G (eq. 2) up to row operations and a column permutation',
       all(triorthogonal(G_TILDE)) and fG is not None and fGt is not None,
       f'columns of G: hyperplane f={fG}; columns of G~: hyperplane f={fGt}; literal row-equivalence: '
       f'{f2_rank(np.vstack([G_TRI, G_TILDE])) == 5} (column order differs)')
record('M1c M_15-to-1 (eq. 1) = G~ with the first column removed (puncturing)', (G_TILDE[:, 1:] == M15).all())
record('M1d 11 rotations derived from eq. (1) = Fig. 15 (manual transcription)', derived == FIG15_TRANSCRIBED, derived)

## M2. 理想入力での 5 量子ビット回路

In [ ]:
res = run_15to1_5qubit()
nz = {k: v for k, v in res.items() if v[0] > 1e-12}
Tp, Tdp = T_GATE @ KETP, T_GATE.conj() @ KETP
out = nz.get((1, 1, 1, 1), (0, None))[1]
record('M2a ideal inputs: X outcomes of qubits 1-4 are deterministically +1', list(nz) == [(1, 1, 1, 1)] and abs(nz[(1, 1, 1, 1)][0] - 1) < 1e-12)
record('M2b output of qubit 5 = T^dag|+> (not T|+>)', equal_up_to_phase(out, Tdp) and not equal_up_to_phase(out, Tp))
record('M2c X T^dag|+> = e^{-i pi/4} T|+>  (Pauli-frame unification used for both protocols)',
       np.allclose(X @ Tdp, np.exp(-1j * np.pi / 4) * Tp))

## M3. 15 量子ビット符号（Fig. 14）

In [ ]:
rT = rm15_logical_check('T'); rTd = rm15_logical_check('Tdag')
record('M3a transversal T on |+_L> gives <X_L>=1/sqrt2, <Y_L>=-1/sqrt2 (logical T^dag|+>), Z stabilisers kept',
       abs(rT['X_L'] - 2 ** -.5) < 1e-9 and abs(rT['Y_L'] + 2 ** -.5) < 1e-9 and rT['min_Zstab'] > 1 - 1e-9, rT)
record('M3b transversal T^dag gives logical T|+>', abs(rTd['Y_L'] - 2 ** -.5) < 1e-9, rTd)
bloch_5q = [np.real(np.vdot(out, P @ out)) for P in (X, Y, Z)]
record('M3c 5-qubit circuit output has the same Bloch vector as the 15-qubit logical state',
       np.allclose(bloch_5q, [rT['X_L'], rT['Y_L'], rT['Z_L']], atol=1e-9), np.round(bloch_5q, 6))

## M4. 誤り検出（15 個の π/8 回転のどれかに $Z$ 誤り）

In [ ]:
cols15 = [tuple(int(i) for i in np.nonzero(M15[:, j])[0]) for j in range(15)]
def accept(err):
    psi = kron_all([KETP.reshape(2, 1)] * 5).reshape(-1)
    for j, qs in enumerate(cols15):
        psi = z_rotation_diag(5, qs, np.pi / 8 + (np.pi / 2 if j in err else 0)) * psi
    p = 1.0; st = psi
    for q in range(4):
        _, st, pp = measure_pauli(st, pauli_on(5, {q: 'X'}), forced=1); p *= pp
        if pp < 1e-14: return 0.0, None
    o = kron_all([np.array([1, 1]) / np.sqrt(2)] * 4).reshape(-1).conj() @ st.reshape(16, 2)
    return p, o / np.linalg.norm(o)
und = {}
ideal = accept(())[1]
wrong3 = 0
for w in (1, 2, 3):
    und[w] = 0
    for e in itertools.combinations(range(15), w):
        p, o = accept(e)
        if p > 1e-9:
            und[w] += 1
            wrong3 += (w == 3 and not equal_up_to_phase(o, ideal))
record('M4 all 1- and 2-error patterns detected; 35 undetected 3-error patterns, all with a wrong output (output error 35 p^3)',
       und[1] == 0 and und[2] == 0 and und[3] == 35 and wrong3 == 35, f'undetected {und}; wrong outputs among undetected weight-3: {wrong3}')

## M5. selective π/4（Fig. 17a）と auto-corrected π/8（Fig. 17b）
選んだ 4 種の Pauli 積 $P$（1〜3 量子ビット）とランダムな入力状態について、**全ての測定結果の分岐**で、補正後のデータ状態が目的の回転と一致するか（必要な補正が Pauli $P$ だけか）を確認する。任意の $P$・任意の入力についての証明ではない（線形性により入力は 1 つのランダム状態で十分だが、$P$ は代表例のみ）。

In [ ]:
def selective_pi4(P_data, psi, basis):
    k = int(np.log2(len(psi))); n = k + 1; a = k
    st0 = np.kron(psi, KET0)
    Pd = pauli_on(k, P_data)
    target = (np.cos(np.pi / 4) * psi - 1j * np.sin(np.pi / 4) * (Pd @ psi)) if basis == 'X' else psi
    outs = []
    for o1 in (1, -1):
        _, st, p1 = measure_pauli(st0, pauli_on(n, {**P_data, a: 'Y'}), forced=o1)
        if p1 < 1e-14: continue
        for o2 in (1, -1):
            _, st2, p2 = measure_pauli(st, pauli_on(n, {a: basis}), forced=o2)
            if p2 < 1e-14: continue
            anc = (np.array([1, o2]) / np.sqrt(2)) if basis == 'X' else (KET0 if o2 == 1 else np.array([0, 1]))
            dat = st2.reshape(2 ** k, 2) @ anc.conj(); dat /= np.linalg.norm(dat)
            outs.append('I' if equal_up_to_phase(dat, target) else 'P' if equal_up_to_phase(Pd @ dat, target) else 'FAIL')
    return outs
ok = True; table = []
for P in [{0: 'Z'}, {0: 'Z', 1: 'Z'}, {0: 'X', 1: 'Y'}, {0: 'Y', 1: 'Z', 2: 'X'}]:
    k = max(P) + 1
    psi = rng.normal(size=2 ** k) + 1j * rng.normal(size=2 ** k); psi /= np.linalg.norm(psi)
    for basis in 'XZ':
        o = selective_pi4(P, psi, basis); ok &= 'FAIL' not in o
    br = auto_corrected_pi8(P, psi)
    ok &= all(b['correction'] != 'FAIL' for b in br) and abs(sum(b['prob'] for b in br) - 1) < 1e-9
    table += [dict(P=str(P), **{kk: b[kk] for kk in ['PZ', 'ZY', 'anc_basis', 'anc', 'Xm', 'correction']}, prob=round(b['prob'], 4)) for b in br]
record('M5 selective pi/4 and auto-corrected pi/8 (4 sample Pauli products, random input): every branch correct up to a Pauli P correction', ok)
pd.DataFrame(table).head(16)

## M6. tile 数・time step の再計数
(a) 格子転記 `FIG17C_*` から数える。(b) [Lit] 本文の記述（「4 qubits + 1 output」「左の 2×2 ブロックは $|m\rangle$ と $|0\rangle$ 用」「2 tiles は多体測定のアンシラ」「11 回転 × 1τ」）から独立に数える。

In [ ]:
sched = tile_schedule(); n_a = len(tiles_used(sched)); tau_a = sum(t for _, t, _ in sched)
n_b = 4 + 1 + 2 * 2 + 2; tau_b = 11 * 1
record('M6 tiles = 11 and time steps = 11 (two independent countings), space-time = 121 d^3',
       n_a == n_b == 11 and tau_a == tau_b == 11, dict(grid=(n_a, tau_a), text=(n_b, tau_b)))
for dd in [3, 5, 7, 9]:
    s = summarize(msd_stages(MSDCostData(dd), 'FT_ref', DEFAULT_PARAMS, 'A', level='tile', data_only=True))
    s2 = summarize(msd_stages(MSDCostData(dd), 'FT_ref', DEFAULT_PARAMS, 'A', level='tile', data_only=False))
    record(f'M6 d={dd}: tile model reproduces 121 d^3 (data) and 242 d^3 (incl. measurement qubits)',
           s['V_qubit_round'] == 121 * dd ** 3 and s2['V_qubit_round'] == 242 * dd ** 3)

## M7. 物理レベルの格子手術（stim、ノイズなし）
2 つの $d\times d$ 回転表面符号パッチを $|+_L\rangle$ に準備し、間の 1 tile（幅 $d$）を $|+\rangle$ で初期化して合併矩形の安定化子を 1 ラウンド測定する。$\bar Z_1\bar Z_2$ の期待値が合併前 0 → 合併後 ±1 となり、その符号が合併で新たに測った $Z$ 型チェックの積と一致することを確認する（[Lit] App. A の手順）。

In [ ]:
for dd in [3, 5, 7]:
    for seed in range(3):
        r = lattice_surgery_zz_check(dd, seed=seed)
        record(f'M7 d={dd} seed={seed}: Z1Z2 random before, deterministic after merge, equals product of merged Z checks',
               r['exp_before'] == 0 and abs(r['exp_after']) == 1 and r['consistent'] and r['O_in_span'],
               f"after={r['exp_after']}, qubits={r['total_qubits']}")
    md_ = MSDCostData(dd)
    record(f'M7 d={dd}: one gap tile adds 2d^2+1 qubits to the merged region (used for the ancilla strip)', md_.gap_tile == 2 * dd * dd + 1, md_.gap_tile)

## M8. パッチの量子ビット数・CNOT 数と換算の一貫性

In [ ]:
ok = True
for dd in range(3, 16, 2):
    m = MSDCostData(dd)
    ok &= m.sq_patch == 2 * dd * dd - 1 and m.tall_patch == 4 * dd * dd - 1 and m.cnot_sq == 4 * dd * (dd - 1)
    phys = summarize(msd_stages(m, 'FT_ref', DEFAULT_PARAMS, 'A', level='physical'))
    ok &= phys['peak_qubits'] == 22 * dd * dd - 6 and phys['V_qubit_round'] == 11 * dd * (22 * dd * dd - 6)
record('M8 rotated patch = 2d^2-1 qubits, 4d(d-1) CNOT/round; explicit physical model: 22d^2-6 qubits per rotation (tile model 22d^2), V = 11 d (22d^2-6)', ok)
m = MSDCostData(5)
record('M8 surface-code round time bounds from the shared scheduler (lb=6: init+4 CNOT+meas)', m.round['time_lb'] == 6, m.round)

## M9. Fig. 17c の 11 個の多パッチ測定の物理レベル構成と検証

各回転 $k$ で測る演算子は $O_k=\bigotimes_{i\in P_k}\bar Z_i\otimes\bar Z_m$（$P_k$ は式(1)の列）。補助帯を $|+\rangle$ に初期化し、新しい検査 $N$ を測ったときに：
* V1 $N$ が互いに可換、V2 各パッチが単独で $k=1$・距離 $d$、
* V3 $O_k\in\mathrm{span}(N_Z)+\mathrm{span}(G_{0,Z})$ で、測られる $Z$ 論理の組合せはちょうど 1 個、V4 $X$ 論理は 1 個も測られない、
* V5 $O_k$ と可換な全ての論理演算子（全 $\bar Z_i$、非参加パッチの $\bar X_j$、参加パッチ対の $\bar X_i\bar X_{i'}$）が $N$ と可換な代表元をもつ（保存される）、
* 合併符号 $G_1=\langle N,\ G_0\text{ の }N\text{ 可換部分群}\rangle$ の論理量子ビット数が 5、距離 $\ge d$（MILP）、

を厳密に確認する。さらに stim で、合併前は $O_k$ がランダム、合併 1 ラウンド後は決定的で、その値が測定結果の積から予言どおりであることを確認する。

In [ ]:
import time
rot_sets = [tuple(sorted({'q%d' % (i + 1) for i in r})) + ('m',) for r in fig15_rotations()]
for dd in [3, 5, 7]:
    t0 = time.time(); allv = True; dists = set()
    for P in rot_sets:
        r = verify_multipatch(dd, P, distance=(dd <= 5))
        allv &= r['valid'] and r['V2_patches_k1']
        if dd <= 5:
            allv &= r['V2_patch_distance_ok']; dists.add(r['V6_merged_distance'])
    record(f'M9 d={dd}: all 11 multi-patch measurements P(x)Z_m are valid lattice surgeries (exact F2 criteria'
           + (', merged-code distance >= d' if dd <= 5 else '') + ')', allv,
           (f'merged (d_X, d_Z) = {sorted(dists)}; ' if dists else 'distance not computed at this d; ') + f'{time.time() - t0:.0f}s')
for dd in [3, 5]:
    ok = all(stim_multipatch_check(dd, P, seed=s)['consistent'] for P in rot_sets for s in range(2))
    record(f'M9 d={dd}: stim - O_k random before, deterministic after one merged round, equal to predicted product (11 rotations x 2 seeds)', ok)
ok = True
for dd in [3, 5, 7]:
    for P in rot_sets:
        a, res = rotation_measured_checks(dd, P)
        b, pats, own = rotation_measured_checks_fast(dd, P)
        qs = sorted(own); idx = {q: i for i, q in enumerate(qs)}; n = len(qs)
        for t in 'XZ':
            A = np.array([_vec(c[1], idx, n) for c in a if c[0] == t]); Bm = np.array([_vec(c[1], idx, n) for c in b if c[0] == t])
            ok &= _rank(A) == _rank(Bm) == _rank(np.vstack([A, Bm]))
        ok &= len(a) == len(b) and max(len(c[1]) for c in b) <= 4
record('M9 fast measured-set construction (used for d up to 15) = verified construction (d=3,5,7); all checks weight <= 4', ok)

# 第 3 部 雑音付きシミュレーションの検証（v5）
## N1. Takada らの方法の適用条件
Takada–Bartlett–Williamson の Clifford-stabilizer simulation（Pauli proxy 法）は、非 Clifford な「Clifford 安定化子」を測る回路に一般に使えるわけではない。十分条件（Definition 2 と Sec. VIII）を D4 code switching の各段について確認する（`takada_conditions`）：
* (a) X 型 Clifford 安定化子の X 因子と CZ 因子の台が互いに素（transversal、式 (131)）。
* (b) CZ, S への X/Y 誤りのドレッシングが単一量子ビット $Z$（ドレッシングの軸が全量子ビットで $Z$）。行列は**大域位相だけを許す厳密比較**（`equal_up_to_phase`）で比べる（以前の版は要素の絶対値を比べており、$I$ と $Z$ を区別できなかった。下の負の対照で、この判定が $I$・別の軸・相対位相の違いを FAIL にすることを確認する）。
* (c) 測定する生成元どうしの群交換子が、その時点の $Z$ 安定化子の張る空間に入る（実効的な可換性、式 (58) / Lemma 1）。switch-in では FSC $|+_L\rangle\otimes|0\rangle_g$ を、D4 側の CZ でドレスした生成元とドレスした論理 $\bar X$ で表して確認する（Sec. VIII B の生成元の選び方）。
* (d) primary-support exactness（Definition 1）：X 台が線形独立。
* (e) $U$ と switch-out の補正網が CNOT と対角第 3 階層ゲートのみ（Sec. VIII A, 式 (178)）。
* (f) 出力の論理安定化子 $\bar H$ が Clifford で、FSC の X 検査と実効的に可換。

**範囲**：これらは各段で測る生成元・論理演算子についての有限サイズ（$L=2..7$, 9；本番は $d=3,5,7,9$）の代数的確認である。プロトコル全体（切り替え・$U$・switch-out の順序を含む）で proxy が真の回路と一致することは、別に N3 で数値的に確認する（$L\le2$）。一般の $L$ についての証明ではない。

In [ ]:
I2_, Z_, S_ = np.eye(2), np.diag([1, -1]), np.diag([1, 1j])
neg = [equal_up_to_phase(np.eye(4), np.kron(I2_, Z_)), equal_up_to_phase(S_, Z_),
       equal_up_to_phase(np.kron(Z_, I2_), np.kron(I2_, Z_)), equal_up_to_phase(np.diag([1, 1j]), np.diag([1, -1j]))]
pos = [equal_up_to_phase(1j * Z_, Z_), equal_up_to_phase(-np.kron(I2_, Z_), np.kron(I2_, Z_))]
record('N1 operator comparison is exact up to a global phase (I vs Z, S vs Z, wrong qubit, relative phase rejected)',
       not any(neg) and all(pos), f'negative controls {neg}, positive controls {pos}')
for L in list(range(2, 8)) + ([] if QUICK else [9]):
    t0 = time.time()
    for name, (ok, det) in takada_conditions(L).items():
        record(f'N1 L={L}: {name}', ok, det)
    print('   (%.0f s)' % (time.time() - t0))

## N2. ドレッシングの代数
誤りの X 成分 $X^e$ が対角ゲート $D$（CCZ, CS, CS$^\dagger$, $T$, $T^\dagger$）を通過するとき、Pauli proxy 回路は $D$ を取り除き、$C_e=D^\dagger X^e D X^e$ を挿入する。$C_e$ が S, Z, CZ の積（Clifford）であり、`dressing()` が返すゲート列と大域位相を除いて一致することを全ての $e$ について行列で確かめる。

In [ ]:
def _kron(ms):
    out = np.array([[1.0 + 0j]])
    for m in ms:
        out = np.kron(out, m)
    return out
_I2 = np.eye(2, dtype=complex); _X = np.array([[0, 1], [1, 0]], dtype=complex)
_G1 = {'S': np.diag([1, 1j]), 'SDG': np.diag([1, -1j]), 'Z': np.diag([1, -1]).astype(complex)}
ok, n = True, 0
for name, k in [('CCZ', 3), ('NCCZ', 3), ('CS', 2), ('CSDG', 2), ('T', 1), ('TDG', 1)]:
    D = np.diag([np.exp(1j * np.pi / 4 * DIAG_THETA[name](x)) for x in itertools.product((0, 1), repeat=k)])
    for e in itertools.product((0, 1), repeat=k):
        Xe = _kron([_X if b else _I2 for b in e])
        Ce = D.conj().T @ Xe @ D @ Xe
        M = np.eye(2 ** k, dtype=complex)
        for g in dressing(name, e):
            if g[0] == 'CZ':
                M = np.diag([(-1) ** (x[g[1]] * x[g[2]]) for x in itertools.product((0, 1), repeat=k)]).astype(complex) @ M
            else:
                M = _kron([_G1[g[0]] if i == g[1] else _I2 for i in range(k)]) @ M
        ph = Ce[0, 0] / M[0, 0]
        ok &= abs(abs(ph) - 1) < 1e-12 and np.allclose(Ce, ph * M)
        n += 1
record('N2 dressing C_e = D^dag X^e D X^e equals the inserted S/Z/CZ gates (all diagonal gates, all e)', ok, f'{n} cases')

## N3. Pauli proxy 回路と真の非 Clifford 回路の並走（最も重要な検査）
同じ誤り（雑音モデルから標本）・同じ測定結果・同じランダムな $X$ 補正を、(i) Pauli proxy 回路（stim、本番と同じ `FastShot`）と (ii) 真の非 Clifford 回路の厳密な疎状態ベクトル（`TrueSim`、CCZ・CS・$T^\dagger$ をそのまま作用）に与え、
* proxy で決定的な測定は、真の回路でも確率 1 で同じ値になる；proxy でランダムな測定は、真の回路でも確率 1/2；
* 最後の論理 Clifford 安定化子の期待値 $\langle\bar H\rangle$ が一致する
ことを全測定について確認する。Takada らの定理（proxy が真の回路と同じ測定統計を与える）を、この回路について数値的に直接確かめるもの。$L=2$ は D4 相のラウンドが並列スケジュール（本番）と逐次スケジュールの両方。

In [ ]:
rows = []
for L, Rin, Rout, seq, p, ntr in [(1, 2, 1, False, 0.03, 30), (2, 2, 1, False, 0.01, 4 if QUICK else 8),
                                  (2, 2, 1, True, 0.01, 2 if QUICK else 4), (2, 2, 1, False, 0.0, 1)]:
    t0 = time.time(); bad = 0; nm = nr = 0; hs = []
    for tr in range(ntr):
        r = lockstep_validation(L, Rin, Rout, NoiseModel(p), rng, sequential=seq, corr_prob=0.05 if p else 0)
        bad += not r['agree']; nm += r['n_meas']; nr += r['n_random']; hs.append(round(float(np.real(r['H_true'])), 6))
        rows.append(dict(L=L, sequential=seq, p=p, trial=tr, n_meas=r['n_meas'], n_random=r['n_random'],
                         n_mismatch=r['n_mismatch'], H_true=np.real(r['H_true']), H_proxy=r['H_proxy'], n_dress=r['n_dress']))
    record(f'N3 L={L} {"sequential" if seq else "parallel"} schedule, p={p}: proxy = exact non-Clifford circuit ({ntr} runs)',
           bad == 0 and (p > 0 or hs == [1.0]), f'{nm} measurements ({nr} random), <H> values {sorted(set(hs))}, {time.time() - t0:.0f}s')
pd.DataFrame(rows).groupby(['L', 'sequential', 'p']).agg(runs=('trial', 'count'), measurements=('n_meas', 'sum'),
                                                          mismatches=('n_mismatch', 'sum'), dressings=('n_dress', 'sum'))

### N3b. 本番と同じ適応処理の全経路（$L=2$）
`lockstep_full`：本番の `D4Experiment.shot` をそのまま使い、flux 復号器の補正、switch-out の復号済み読み出し、物理的な補正 CZ の層とその雑音、Pauli frame、charge 復号器までを、proxy（stim）と真の回路の状態ベクトル（`TrueSim`）に同じ誤り・同じ測定結果で並走させる。S0（Clifford 回路で stim だけで厳密）は、その理想出力 $E|+_L\rangle|0\rangle_g$（$E$ はランダムな 2 量子ビット Pauli）で置き換える（S0 の多数の補助量子ビットを含む状態ベクトルはメモリに入らない）。真の回路側は proxy 側の状態や誤りの表現を使わず、ゲートを状態ベクトルに直接作用させる。

In [ ]:
rows = []
for Rin, p, ntr in [(2, 0.01, 2 if QUICK else 6), (2, 0.03, 2 if QUICK else 6), (1, 0.001, 2)]:
    ex = D4Experiment(2, 1, Rin, 1, NoiseModel(p))
    t0 = time.time(); bad = nm = nr = ncz = nco = 0
    for tr in range(ntr):
        r = lockstep_full(2, Rin, 1, NoiseModel(p), rng, ex=ex)
        bad += not r['agree']; nm += r['n_meas']; nr += r['n_random']; ncz += r['so_cz']; nco += r['n_corr']
        rows.append(dict(Rin=Rin, p=p, trial=tr, **r))
    record(f'N3b L=2 Rin={Rin}, p={p}: full adaptive path (flux corrections, decoded switch-out, physical CZ layers '
           f'with noise, Pauli frame) proxy = exact non-Clifford circuit ({ntr} runs)', bad == 0,
           f'{nm} measurements ({nr} random), {nco} flux corrections, {ncz} physical switch-out CZs, {time.time() - t0:.0f}s')
pd.DataFrame(rows).groupby(['Rin', 'p']).agg(runs=('trial', 'count'), measurements=('n_meas', 'sum'),
                                             mismatches=('n_mismatch', 'sum'), corrections=('n_corr', 'sum'),
                                             switch_out_cz=('so_cz', 'sum'))

## N4. 雑音なしの全体（復号器込み）
雑音を 0 にした本番と同じ実験（`D4Experiment`、復号器込み）で出力の不忠実度が 0 になること。switch-in のランダムな $S^g$ 測定結果、switch-out の $Z^g$ 読み出しとその補正、Pauli frame の扱いが正しいことを $d=3,5,7$ で確認する。

In [ ]:
for d in (3, 5, 7):
    ex = D4Experiment(d, d, d, d, NoiseModel(0.0))
    inf, st = ex.run(5, rng)
    record(f'N4 d={d}: noiseless protocol (with decoders) outputs the exact magic state', bool(np.all(inf == 0)),
           f'infidelities {inf.tolist()}')

## N5. 1 個の故障（INFO：故障の網羅範囲と失敗の記録。合否ではなく、耐故障性を保証するものでもない）
距離 $d\ge3$ で耐故障な回路と復号器なら、1 個の故障は必ず訂正される。本研究の回路・復号器（[Dav] の JIT decoder は参照できず独自に構成）で、1 個の故障を強制し（他の雑音は 0）出力が誤るかを調べる（`single_fault_scan`）。
* **網羅範囲**：コンパイルした回路の全ての雑音位置（ゲート、CCZ・CS・$T^\dagger$、初期化、測定、アイドル）× その雑音チャネルの全ての Pauli、さらに switch-out の**動的な故障**（測定結果に応じて適用される各補正 CZ の後の 15 通りの Pauli、補正 CZ 層のアイドル）。古典処理は雑音なし。
* **測定分岐**：各故障を 2 つの測定乱数の種で実行する（ランダムな測定結果、したがって分岐が変わる）。どちらかで誤れば「失敗あり」。失敗しなかった故障が**全分岐で**訂正されるとは言えない。
* **一次の係数**：$c_1=\sum_f (P(f)/p)\,\overline{\varepsilon}(f)$（$P(f)$ はその故障の確率：1 量子ビット脱分極の各 Pauli $p/3$、2 量子ビット $p/15$、3 量子ビット $p/63$、反転 $p$、アイドルは $p_{\rm idle}/3$；$\overline\varepsilon$ は 2 つの種の不忠実度の平均）。小さい $p$ で $\varepsilon_K\approx c_1p$（この復号器での単一故障の寄与）。件数の比とは異なる。
* 窓（`D4Experiment.window`：D4 相最後のラウンド〜FSC の最初のラウンド）の内外に分けて記録する。窓の外の失敗（例えば D4 相のより前のラウンド）も数える。$d=3$ は全列挙、$d=5,7$ は失敗がまれなので無作為抽出では $c_1$ を推定できず、**窓の中を全列挙**する（$d=5$：A, B、$d=7$：B）。窓の外の寄与は $d=3$ の全列挙と $d=5$ の抽出で小さいことを確認する。$c_1$ の $d$ 依存は、比較ノートの「窓の項が $d$ によらない」という仮定の確認になる。
失敗の原因は、少なくとも (i) D4 相最後のラウンドの CCZ ガジェット内の故障（CZ のドレッシングによる非 Pauli な効果を charge 復号器の誤りモデルが含まない）、(ii) switch-out の時間境界での検出器の基底（どの基底でも一部の誤りがハイパーエッジ）、(iii) $U$ 直前の $Z$ ラウンドの後に起きて $U$ を通る $X$ 誤り（その時点で検出できない）を含む。原因を復号器・測定ガジェット・順序・境界のどれに帰すべきかの完全な分類はしていない（未解決）。

In [ ]:
t0 = time.time()
N5 = {}
CONFIGS = [(3, 1.0, None, False), (3, 0.0, None, False), (5, 1.0, 1500 if QUICK else 6000, False)]
if not QUICK:                                    # exhaustive over the switching window (where c1 sits)
    CONFIGS += [(5, 1.0, None, True), (5, 0.0, None, True), (7, 0.0, None, True)]
rows = []
for d, pi, ns, win in CONFIGS:
    ex = D4Experiment(d, d, d, d, NoiseModel(1e-3, p_idle=pi * 1e-3))
    r = single_fault_scan(ex, n_sample=ns, seeds=2, only_window=win)
    N5[(d, pi, win)] = r
    scope = 'switching window, all' if win else ('all' if ns is None else 'random sample of')
    info(f'N5 d={d}, scenario {"A" if pi else "B"}: single faults ({scope} {r["n_tested"]} of {r["n_faults_total"]}, '
         f'incl. dynamic switch-out faults, 2 seeds)',
         f'failing {r["n_fail"]} (window {r["n_fail_window"]}, outside {r["n_fail_outside"]}); first-order coefficient '
         f'c1 = {r["c1"]:.2f} (window {r["c1_window"]:.2f}, outside {r["c1_outside"]:.2f}); by block {r["by_block"]}')
    rows.append(dict(d=d, scenario='A' if pi else 'B', scope=scope, faults=r['n_tested'], failing=r['n_fail'],
                     c1_window=r['c1_window'], c1_outside=r['c1_outside']))
dyn = [f for f in N5[(3, 1.0, False)]['failures'] if f['fault'].startswith("('SO")]
print('failing dynamic switch-out faults at d=3 (A):', len(dyn))
print('%.0f s' % (time.time() - t0))
display(pd.DataFrame(rows).round(3))
pd.DataFrame(N5[(3, 1.0, False)]['failures'][:30])

## N9. switch-out の物理スケジュール
switch-out の補正 $\mathrm{CZ}(\Gamma(a))$ は、全ての $S^g_v$ の CZ 対の和集合の辺彩色による**固定の層スケジュール**で適用する（層数は測定結果によらない最悪値で、時間の計数に使う）。(i) 各層で同じ量子ビットを 2 回使わない、(ii) どの $\kappa$ でも適用される CZ が $\prod_{v\in\kappa}S^g_v$ の CZ 部分（同じ対が偶数回なら相殺）と一致する、(iii) 時間の計数 `block_steps` が $g$ の読み出し 1 ＋層数であること、を確認する。

In [ ]:
for d in (3, 5, 7, 9):
    circ = D4SwitchingCircuit(d, 1, 1, 1)
    ok = True
    layers = {}
    for pr, c_ in circ.so_colour.items():
        layers.setdefault(c_, []).append(pr)
    for c_, prs in layers.items():
        qs = [q for pr in prs for q in pr]
        ok &= len(qs) == len(set(qs))
    r_ = np.random.default_rng(d)
    for _ in range(20):
        kap = r_.integers(0, 2, len(circ.gens_g))
        want = {}
        for v in np.nonzero(kap)[0]:
            for pr in circ.gpairs[v]:
                pr = tuple(sorted(pr)); want[pr] = want.get(pr, 0) ^ 1
        want = {pr for pr, c_ in want.items() if c_}
        ok &= want <= set(circ.so_colour)
    so = [b for b in circ.blocks if b['name'][0] == 'SO'][0]
    ok &= block_steps(circ, so) == 1 + len(layers)
    record(f'N9 d={d}: switch-out CZ schedule valid (no qubit used twice per layer, covers every kappa), '
           f'time = readout + {len(layers)} CZ layer(s)', ok, f'{len(circ.so_colour)} distinct CZ pairs')

## N10. FSC の並列スケジュール（費用の感度用）
本番のシミュレーションは FSC の X 検査と Z 検査を順に測る 14 ステップのラウンドを使う。全検査を同時に測る 6 CNOT 層（初期化・測定と合わせて 8 ステップ；$\mathcal L_{rb}$ の重さ 6 の $X^rX^b$ 検査があるので 6 層が最小）の順序を探索し（`fsc_parallel_schedule`）、stim のメモリ実験（X 基底・Z 基底）で、雑音なしで全ての検出器と論理観測量が決定的であること、CNOT の後の 2 量子ビット脱分極を含む回路レベルの最短の論理誤りが $L$ 以上であること（フック誤りで距離が下がらないこと）を確認する。この順序は比較ノートの時間の感度（`*_fsc8`）に使う。

In [ ]:
for L in (3, 5, 7):
    t0 = time.time()
    P_ = D4Protocol(L)
    sch = fsc_parallel_schedule(P_)
    ok = True
    for basis in 'XZ':
        s0 = fsc_memory_circuit(P_, sch, basis, 3, 0.0).compile_detector_sampler(seed=2).sample(50, append_observables=True)
        dd = len(fsc_memory_circuit(P_, sch, basis, L, 1e-3).detector_error_model(decompose_errors=True).shortest_graphlike_error())
        ok &= (not s0.any()) and dd >= L
    record(f'N10 L={L}: parallel FSC schedule ({sch["T"]} CNOT layers) deterministic, circuit distance >= L in both bases',
           ok, f'distances {sch["distance"]}, search seed {sch["seed"]}, {time.time() - t0:.0f}s')

## N6. Litinski（2019）の誤り予算の再実装
公開コード（github.com/litinski/magicstates、Apache-2.0）は実行せずに読んで再実装した（`msd_noisy_lib`）。そのノートブックに保存されている出力（15-to-1 の 1 段 4 件、(15-to-1)×(15-to-1) の 2 段 4 件）を、出力誤り（有効数字 2 桁）・量子ビット数・code cycle 数（小数 1 桁）で再現することを確認する（128 ビット演算）。

In [ ]:
LIT_PUBLISHED = [  # (args, output error, qubits, code cycles) as printed in Python/magicstates.ipynb
    ((1e-4, 7, 3, 3), '4.4e-08', 810, '18.1'), ((1e-4, 9, 3, 3), '9.3e-10', 1146, '18.1'),
    ((1e-4, 11, 5, 5), '1.9e-11', 2066, '30.0'), ((1e-3, 17, 7, 7), '4.5e-08', 4618, '42.6'),
    ((1e-4, 9, 3, 3, 25, 9, 9, 4), '6.3e-25', 18630, '67.7'), ((1e-3, 11, 5, 5, 25, 11, 11, 6), '2.7e-12', 30732, '82.5'),
    ((1e-3, 13, 5, 5, 29, 11, 13, 6), '3.3e-14', 39108, '97.5'), ((1e-3, 17, 7, 7, 41, 17, 17, 6), '4.5e-20', 73460, '127.5')]
rows = []
for args, e_pub, q_pub, c_pub in LIT_PUBLISHED:
    f = litinski2019_one_level if len(args) == 4 else litinski2019_two_level
    r = f(*args, backend='mp')
    ok = ('%.1e' % r['eps'] == e_pub) and r['qubits'] == q_pub and ('%.1f' % r['cycles'] == c_pub)
    rows.append(dict(protocol=r['protocol'], p=args[0], eps=r['eps'], eps_published=e_pub, qubits=r['qubits'],
                     qubits_published=q_pub, cycles=round(r['cycles'], 2), cycles_published=c_pub, match=ok))
    record(f'N6 Litinski 2019 {r["protocol"]} at p={args[0]:g} reproduces the published output', ok,
           f'eps {r["eps"]:.2e} (published {e_pub}), qubits {r["qubits"]} ({q_pub}), cycles {r["cycles"]:.2f} ({c_pub})')
pd.DataFrame(rows)

## N7. 表面符号の較正回路
`noisy_surface_code` が stim の回転表面符号メモリ回路を D4 と同じ雑音モデルに置き換えたものであることを確認する：雑音 0 で全ての検出器と観測量が 0（決定的）、回路レベルの最短の論理誤り（graph-like）が $d$、1 ラウンドが 6 時間ステップ。

In [ ]:
for d in (3, 5, 7):
    ok, det = True, []
    for basis in ('x', 'z'):
        c0 = noisy_surface_code(d, d, basis, 0.0)
        s = c0.compile_detector_sampler(seed=1).sample(200, append_observables=True)
        c1 = noisy_surface_code(d, d, basis, 1e-3)
        dist = len(c1.detector_error_model(decompose_errors=True).shortest_graphlike_error())
        c2 = noisy_surface_code(d, 2 * d, basis, 1e-3)
        steps = (c2.num_ticks - c1.num_ticks) / d
        ok &= (not s.any()) and dist == d and steps == 6
        det.append(f'{basis}: distance {dist}, {steps:g} steps/round')
    record(f'N7 d={d}: calibration circuit deterministic without noise, circuit distance = d, 6 time steps per round', ok, '; '.join(det))

## N8. GoSC 15-to-1 の誤り予算モデル
(i) 雑音 0 で出力誤り 0・成功確率 1、(ii) 表面符号の誤りを 0 にして入力の魔法状態に確率 $p$ の $Z$ 誤りだけを入れると、出力誤りが 15-to-1 の既知の値 $35p^3$（主要項）になる — Fig. 15 の 11 個の回転（`GOSC_ROTATIONS`、第 2 部 M1 で照合済みの行列から）が正しい符号を構成していることの確認。

In [ ]:
zero = lambda p, d: mpmath.mpf(0)
ps0, e0 = run_steps(gosc_15to1_steps(1e-3, 7, zero, p_in=0), out_qubit=4, backend='mp')
record('N8 GoSC 15-to-1 without errors: output error 0, success probability 1', abs(e0) < 1e-30 and abs(ps0 - 1) < 1e-30)
for p in (1e-3, 1e-4):
    ps1, e1 = run_steps(gosc_15to1_steps(p, 7, zero, p_in=p, in_channel='z'), out_qubit=4, backend='mp')
    ratio = float(e1 / (35 * mpmath.mpf(p) ** 3))
    record(f'N8 GoSC 15-to-1 with Z errors (prob. {p:g}) on the input states only: output error = 35 p^3 (leading order)',
           abs(ratio - 1) < 0.02, f'ratio {ratio:.4f}, success probability {float(ps1):.6f}')

## まとめ

In [ ]:
n_pass = sum(v['status'] == 'PASS' for v in VAL.values()); n_fail = sum(v['status'] == 'FAIL' for v in VAL.values())
print(f'{n_pass} PASS, {n_fail} FAIL, {len(VAL) - n_pass - n_fail} INFO')
json.dump(dict(notebook='validation', results=VAL,
               not_verified=['Knill-Laflamme distance of the D4 phase for L >= 6 (L >= 7 with FULL_KL=True)',
                             'exact (state-vector) simulation of the noisy protocol for L >= 3 (N3 is L = 1, 2; for L >= 3 the Pauli-proxy method is justified by the conditions of N1 and Takada et al.)',
                             'N3b replaces S0 by its ideal output (S0 is Clifford and exact in stim, but its exact state vector does not fit in memory)',
                             'N1 is a finite-size algebraic check (L = 2..7, 9), not a proof for general L',
                             'fault tolerance: single faults are NOT always corrected by our circuit + decoders (N5); only the sampled measurement branches are tested',
                             'causes of the single-fault failures not fully classified (decoder vs gadget vs ordering vs boundary)',
                             'feed-forward X corrections are applied as a noiseless, zero-time Pauli layer (idealisation)',
                             'constants of the O(d) round counts (kappa)',
                             'FSC -> standard rotated-patch conversion',
                             'stabilizer-level layout of the Z_m (x) Y_0 twist-based seam (O(d) measurement qubits not counted)',
                             'error budget of the GoSC layouts (our adaptation of Litinski 2019, not simulated at circuit level)',
                             'classical processing latency']),
          open(f'{RESULTS}/validation.json', 'w'), indent=1)
pd.DataFrame([(k, v['status']) for k, v in VAL.items()], columns=['check', 'status'])

### 検証できていない事項（正しさが確認されたものとして扱わない）
* $L\ge 6$ の D4 相の Knill–Laflamme 距離（$L\le5$ は V5b で厳密に $=L$）。
* $L\ge3$ での雑音付きプロトコルの状態ベクトルによる直接照合（N3 は $L\le2$。$L\ge3$ の Pauli proxy 法の正しさは N1 の有限サイズの条件確認と Takada らの定理に依拠）。N3b は S0 を理想出力で置き換えている。
* 耐故障性：本研究の回路・復号器は 1 個の故障を必ずしも訂正しない（N5）。N5 は 2 つの測定分岐しか試していない。失敗の原因（復号器・測定ガジェット・順序・境界）の完全な分類はしていない。
* フィードフォワードの $X$ 補正は雑音なし・時間 0 の Pauli 層として適用する理想化。
* $O(d)$ ラウンドの定数 $\kappa$、FSC → 通常の回転表面符号パッチへの変換。
* Fig. 17c の継ぎ目（$Z_m\otimes Y_0$ の twist）の安定化子レベルの配置、GoSC 配置の誤り予算（Litinski 2019 の方法を本研究で GoSC の形に当てはめたもの。回路レベルのシミュレーションではない）。
* 古典処理の待ち時間。